# MAE Figure 6 analysis

# 1. Load MAE analysis inputs and configure paths

## Dependencies, paths, and final parameters

In [ ]:
from pathlib import Path
import json
import gc
import math
import os
import re

import numpy as np
import pandas as pd
import scipy.sparse as sp
from collections import defaultdict
import scanpy as sc
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Documented private input and output path variables.
# Private AnnData objects, model files, and intermediate matrices are not distributed.
analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path

ROOT = resolve_analysis_path(
    os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai"))
)
HVG_ADATA_PATH = resolve_analysis_path(
    os.environ.get("MAE_HVG_H5AD", os.path.join("outputs", "ai", "adata_mono_MAE_HVG5000.h5ad"))
)
METADATA_PATH = resolve_analysis_path(
    os.environ.get("MAE_METADATA_XLSX", "Metadata.xlsx")
)
ROOT.mkdir(parents=True, exist_ok=True)

CART = "CAR-T_CRS"
COVID = "COVID19"
SLE = "SLE"
LOCALIZATION_DISEASES = [CART, COVID, SLE]
FINAL_SEEDS = [20260810, 20260811, 20260812, 20260813, 20260814]

DISEASE_COL = "disease"
PATIENT_COL = "patient_id"
SAMPLE_COL = "sample_id"
SEVERITY_COL = "severity"
STAGE_COL = "sampling_time"

BATCH_SIZE = 256
K_GRID = [5, 10, 20, 30, 50, 75, 100, 150, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
K_MAX = 1000
K90_BY_DISEASE = {CART: 229, COVID: 649, SLE: 116}
N_TITRATION_CONTROLS = 100
N_MATCHED_CONTROLS = 100
TOP_N_AVAILABLE_PROGRAM_MARKERS = 100


In [ ]:
adata_m = sc.read_h5ad(HVG_ADATA_PATH)

if PATIENT_COL not in adata_m.obs.columns:
    metadata = pd.read_excel(METADATA_PATH)
    sample_to_patient = dict(zip(metadata[SAMPLE_COL].astype(str), metadata[PATIENT_COL].astype(str)))
    adata_m.obs[PATIENT_COL] = adata_m.obs[SAMPLE_COL].astype(str).map(sample_to_patient)

if not sp.isspmatrix_csr(adata_m.X):
    adata_m.X = sp.csr_matrix(adata_m.X)

INPUT_DIM = adata_m.n_vars
STEPS_PER_EPOCH = math.ceil(adata_m.n_obs / BATCH_SIZE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

hierarchy = defaultdict(lambda: defaultdict(list))
for idx, row in enumerate(adata_m.obs[[DISEASE_COL, PATIENT_COL]].itertuples(index=False)):
    hierarchy[str(row[0])][str(row[1])].append(idx)

print("Device:", DEVICE)
print("Input dimension:", INPUT_DIM)
print("Steps per epoch:", STEPS_PER_EPOCH)


# 2. Helper functions

In [ ]:
class MaskedExpressionAutoencoder(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim_1=512,
        hidden_dim_2=128,
        latent_dim=64
    ):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim_1),
            nn.LayerNorm(hidden_dim_1),
            nn.GELU(),

            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.LayerNorm(hidden_dim_2),
            nn.GELU(),

            nn.Linear(hidden_dim_2, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim_2),
            nn.LayerNorm(hidden_dim_2),
            nn.GELU(),

            nn.Linear(hidden_dim_2, hidden_dim_1),
            nn.LayerNorm(hidden_dim_1),
            nn.GELU(),

            nn.Linear(hidden_dim_1, input_dim)
        )

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        z = self.encode(x)
        recon = self.decoder(z)
        return recon, z

In [ ]:
def load_frozen_model(seed):

    checkpoint_path = (
        MODEL_DIR /
        f"Fig6_MAE_seed_{seed}.pt"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    model = MaskedExpressionAutoencoder(
        input_dim=checkpoint["input_dim"],
        hidden_dim_1=checkpoint["hidden_dim_1"],
        hidden_dim_2=checkpoint["hidden_dim_2"],
        latent_dim=checkpoint["latent_dim"]
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model = model.to(DEVICE)
    model.eval()

    return model

In [ ]:
def get_anchor_indices(
    disease,
    group
):
    """
    Return cell indices for each disease-specific anchor group
    """

    if disease == CART:

        if group == "risk":
            mask = (
                (disease_arr == CART)
                & (severity_arr == "Severe")
                & (stage_arr == "CAR-T_CRS_pro")
            )

        elif group == "temporal_ref":
            mask = (
                (disease_arr == CART)
                & (severity_arr == "Severe")
                & np.isin(
                    stage_arr,
                    [
                        "CAR-T_CRS_before",
                        "CAR-T_CRS_con"
                    ]
                )
            )

        elif group == "stage_ref":
            mask = (
                (disease_arr == CART)
                & (severity_arr == "No_CRS")
                & (stage_arr == "CAR-T_CRS_pro")
            )

        else:
            raise ValueError(f"Unknown group: {group}")


    elif disease == COVID:

        if group == "risk":
            mask = (
                (disease_arr == COVID)
                & (severity_arr == "Severe")
                & (stage_arr == "COVID19_pro")
            )

        elif group == "stage_ref":
            mask = (
                (disease_arr == COVID)
                & (severity_arr == "Moderate")
                & (stage_arr == "COVID19_pro")
            )

        elif group == "temporal_ref":
            mask = (
                (disease_arr == COVID)
                & (severity_arr == "Severe")
                & (stage_arr == "COVID19_con")
            )

        else:
            raise ValueError(f"Unknown group: {group}")


    elif disease == SLE:

        if group == "risk":
            mask = (
                (disease_arr == SLE)
                & (severity_arr == "Severe")
            )

        elif group == "severity_ref":
            mask = (
                (disease_arr == SLE)
                & (severity_arr == "Moderate")
            )

        else:
            raise ValueError(f"Unknown group: {group}")

    else:
        raise ValueError(f"Unknown disease: {disease}")

    return np.where(mask)[0]

In [ ]:
def unit_vector(x):
    x = np.asarray(x, dtype=np.float32)
    norm = np.linalg.norm(x)

    if norm < 1e-10:
        raise ValueError("Near-zero vector.")

    return x / norm

In [ ]:
def rows_to_dense_float32(
    X,
    idx
):
    """
    Extract rows from AnnData matrix and convert to dense float32
    """

    if sp.issparse(X):
        arr = X[idx, :].toarray()
    else:
        arr = np.asarray(X[idx, :])

    return arr.astype(np.float32, copy=False)

In [ ]:
def patient_balanced_mean(
    matrix,
    idx
):
    """
    Equal-patient mean vector from either:
    - dense/memmap latent matrix, or
    - sparse/dense expression matrix
    """

    idx = np.asarray(idx, dtype=np.int64)
    sub_patients = patient_arr[idx]

    patient_vectors = []

    for patient in np.unique(sub_patients):

        idx_p = idx[sub_patients == patient]

        if sp.issparse(matrix):
            vec = np.asarray(
                matrix[idx_p, :].mean(axis=0)
            ).ravel()
        else:
            vec = np.asarray(
                matrix[idx_p, :].mean(axis=0)
            ).ravel()

        patient_vectors.append(
            vec.astype(np.float32, copy=False)
        )

    return np.vstack(patient_vectors).mean(axis=0).astype(np.float32)

In [ ]:
def select_balanced_cells(
    idx,
    max_cells_per_patient=64,
    random_seed=20260811
):
    """
    Balanced subsampling of cells for attribution
    """

    rng = np.random.default_rng(random_seed)

    idx = np.asarray(idx, dtype=np.int64)
    sub_patients = patient_arr[idx]

    selected = []

    for patient in np.unique(sub_patients):

        idx_p = idx[sub_patients == patient]

        n_take = min(
            len(idx_p),
            max_cells_per_patient
        )

        if n_take < len(idx_p):
            idx_p = rng.choice(
                idx_p,
                size=n_take,
                replace=False
            )

        selected.append(idx_p)

    return np.concatenate(selected)

In [ ]:
def build_disease_state(
    seed,
    disease
):
    """
    Build disease-specific latent direction(s) and
    expression baseline for Integrated Gradients
    """

    Z = Z_by_seed[seed]

    risk_idx = get_anchor_indices(disease, "risk")

    mu_risk = patient_balanced_mean(
        Z,
        risk_idx
    )

    if disease == CART:

        ref_idx = get_anchor_indices(
            disease,
            "temporal_ref"
        )

        mu_ref = patient_balanced_mean(
            Z,
            ref_idx
        )

        d_temporal = unit_vector(
            mu_risk - mu_ref
        )

        baseline = patient_balanced_mean(
            adata_m.X,
            ref_idx
        )

        return {
            "disease": disease,
            "baseline": baseline,
            "d_temporal": d_temporal
        }


    elif disease == COVID:

        stage_ref_idx = get_anchor_indices(
            disease,
            "stage_ref"
        )

        temporal_ref_idx = get_anchor_indices(
            disease,
            "temporal_ref"
        )

        mu_stage_ref = patient_balanced_mean(
            Z,
            stage_ref_idx
        )

        mu_temporal_ref = patient_balanced_mean(
            Z,
            temporal_ref_idx
        )

        d_stage = unit_vector(
            mu_risk - mu_stage_ref
        )

        d_temporal = unit_vector(
            mu_risk - mu_temporal_ref
        )

        baseline_stage = patient_balanced_mean(
            adata_m.X,
            stage_ref_idx
        )

        baseline_temporal = patient_balanced_mean(
            adata_m.X,
            temporal_ref_idx
        )

        baseline = (
            baseline_stage
            + baseline_temporal
        ) / 2

        return {
            "disease": disease,
            "baseline": baseline.astype(np.float32),
            "d_stage": d_stage,
            "d_temporal": d_temporal
        }

    elif disease == SLE:

        ref_idx = get_anchor_indices(
            disease,
            "severity_ref"
        )

        mu_ref = patient_balanced_mean(
            Z,
            ref_idx
        )

        d_severity = unit_vector(
            mu_risk - mu_ref
        )

        baseline = patient_balanced_mean(
            adata_m.X,
            ref_idx
        )

        return {
            "disease": disease,
            "baseline": baseline,
            "d_severity": d_severity
        }

    else:
        raise ValueError(f"Unknown disease: {disease}")

In [ ]:
def prepare_state_for_torch(
    state
):
    out = dict(state)

    out["baseline_t"] = torch.from_numpy(
        state["baseline"][None, :]
    ).to(DEVICE)

    if state["disease"] == CART:
        out["d_temporal_t"] = torch.from_numpy(
            state["d_temporal"][None, :]
        ).to(DEVICE)

    elif state["disease"] == COVID:
        out["d_stage_t"] = torch.from_numpy(
            state["d_stage"][None, :]
        ).to(DEVICE)

        out["d_temporal_t"] = torch.from_numpy(
            state["d_temporal"][None, :]
        ).to(DEVICE)

    elif state["disease"] == SLE:
        out["d_severity_t"] = torch.from_numpy(
            state["d_severity"][None, :]
        ).to(DEVICE)

    return out

In [ ]:
def risk_scalar_from_input(
    model,
    x,
    state_t
):
    """
    Scalar disease-specific risk score for attribution.
    """

    z = model.encode(x)

    if state_t["disease"] == CART:
        score = (
            z * state_t["d_temporal_t"]
        ).sum(dim=1)

    elif state_t["disease"] == COVID:
        score = 0.5 * (
            (z * state_t["d_stage_t"]).sum(dim=1)
            +
            (z * state_t["d_temporal_t"]).sum(dim=1)
        )

    elif state_t["disease"] == SLE:
        score = (
            z * state_t["d_severity_t"]
        ).sum(dim=1)

    else:
        raise ValueError("Unknown disease.")

    return score

In [ ]:
def integrated_gradients_batch(
    model,
    x,
    baseline,
    state_t,
    n_steps=32
):
    """
    Integrated Gradients for one batch of cells
    Returns a tensor with same shape as x
    """

    model.eval()

    x = x.to(DEVICE)
    baseline = baseline.to(DEVICE)

    delta = x - baseline

    total_grads = torch.zeros_like(x)

    alphas = torch.linspace(
        1.0 / n_steps,
        1.0,
        n_steps,
        device=DEVICE
    )

    for alpha in alphas:

        x_interp = baseline + alpha * delta
        x_interp.requires_grad_(True)

        score = risk_scalar_from_input(
            model=model,
            x=x_interp,
            state_t=state_t
        ).sum()

        grad = torch.autograd.grad(
            score,
            x_interp
        )[0]

        total_grads += grad.detach()

    avg_grads = total_grads / n_steps

    attributions = delta * avg_grads

    return attributions

In [ ]:
def attribute_one_seed_one_disease(
    seed,
    disease,
    max_cells_per_patient=64,
    batch_size=32,
    ig_steps=32,
    random_seed=20260811
):
    """
    Patient-balanced gene attribution for one disease and one seed
    """

    model = load_frozen_model(seed)

    state = build_disease_state(
        seed=seed,
        disease=disease
    )

    state_t = prepare_state_for_torch(
        state
    )

    risk_idx_full = get_anchor_indices(
        disease,
        "risk"
    )

    risk_idx = select_balanced_cells(
        idx=risk_idx_full,
        max_cells_per_patient=max_cells_per_patient,
        random_seed=random_seed + seed
    )

    patient_sums = {}
    patient_counts = {}

    for start in range(
        0,
        len(risk_idx),
        batch_size
    ):
        end = min(
            start + batch_size,
            len(risk_idx)
        )

        batch_idx = risk_idx[start:end]

        x_np = rows_to_dense_float32(
            adata_m.X,
            batch_idx
        )

        x_t = torch.from_numpy(
            x_np
        ).to(DEVICE)

        attr_t = integrated_gradients_batch(
            model=model,
            x=x_t,
            baseline=state_t["baseline_t"],
            state_t=state_t,
            n_steps=ig_steps
        )

        attr_np = (
            attr_t
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32, copy=False)
        )

        batch_patients = patient_arr[
            batch_idx
        ]

        for i, patient in enumerate(batch_patients):

            if patient not in patient_sums:
                patient_sums[patient] = np.zeros(
                    attr_np.shape[1],
                    dtype=np.float32
                )
                patient_counts[patient] = 0

            patient_sums[patient] += attr_np[i]
            patient_counts[patient] += 1

        del x_t, attr_t
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Equal-patient aggregation
    patient_vectors = []

    for patient in sorted(patient_sums.keys()):

        patient_mean = (
            patient_sums[patient]
            / patient_counts[patient]
        )

        patient_vectors.append(
            patient_mean
        )

    patient_vectors = np.vstack(
        patient_vectors
    )

    disease_vector = np.mean(
        patient_vectors,
        axis=0
    ).astype(np.float32)

    result = {
        "seed": seed,
        "disease": disease,
        "n_patients": patient_vectors.shape[0],
        "n_cells": len(risk_idx),
        "gene_attribution": disease_vector,
        "patient_vectors": patient_vectors
    }

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

## Helper: safe KDE

In [ ]:
# Patient-balanced ridge density
def safe_kde(values, x_grid, bw_method="scott"):
    """
    Estimate one patient's cell-level score distribution.

    Each patient's KDE is subsequently normalized to area=1,
    so patients contribute equally regardless of cell number.
    """

    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 5:
        return None

    # Completely constant values cannot support Gaussian KDE
    if np.nanstd(values) < 1e-8:
        return None

    kde = gaussian_kde(
        values,
        bw_method=bw_method,
    )

    density = kde(x_grid)

    area = np.trapz(
        density,
        x_grid,
    )

    if area <= 0:
        return None

    return density / area

## Helper: patient-balanced density

In [ ]:
def patient_balanced_density(
    data,
    x_grid,
    bw_method="scott",
):
    """
    Construct one ridge density.

    Step 1:
        KDE is estimated separately for every patient.

    Step 2:
        Patient densities are averaged equally within each
        clinical component.

    Step 3:
        If the group contains multiple reference components
        (e.g. COVID moderate progression + severe convalescence),
        component densities are averaged equally.

    Thus:
        cell number cannot dominate patient weight, and
        one reference context cannot dominate another simply
        because it contains more observations.
    """

    component_densities = []

    for component, component_df in data.groupby(
        "anchor_component",
        observed=True,
    ):

        patient_densities = []

        for patient, patient_df in component_df.groupby(
            "patient_id",
            observed=True,
        ):

            density = safe_kde(
                patient_df["risk_score"].values,
                x_grid,
                bw_method=bw_method,
            )

            if density is not None:
                patient_densities.append(density)

        if len(patient_densities) == 0:
            continue

        component_density = np.mean(
            np.vstack(patient_densities),
            axis=0,
        )

        component_densities.append(
            component_density
        )

    if len(component_densities) == 0:
        raise ValueError(
            "No valid patient KDEs could be generated."
        )

    # Equal clinical-component weighting.
    final_density = np.mean(
        np.vstack(component_densities),
        axis=0,
    )

    # Normalize again for clean visualization.
    final_density /= np.trapz(
        final_density,
        x_grid,
    )

    return final_density

In [ ]:
def patient_balanced_sample(indices, patient_ids, n, rng):
    """
    Sample up to n indices while spreading the sample across patients as evenly as possible
    This prevents one large patient from visually dominating a disease
    """
    indices = np.asarray(indices)
    patient_ids = np.asarray(patient_ids)

    if len(indices) <= n:
        return np.sort(indices)

    unique_patients = pd.unique(patient_ids)
    n_patients = len(unique_patients)

    # Base quota per patient
    base_quota = max(1, n // n_patients)

    chosen = []
    leftovers = []

    for p in unique_patients:
        p_idx = indices[patient_ids == p]
        take = min(base_quota, len(p_idx))

        if take > 0:
            pick = rng.choice(p_idx, size=take, replace=False)
            chosen.append(pick)

            remaining = np.setdiff1d(p_idx, pick, assume_unique=False)
            if len(remaining) > 0:
                leftovers.append(remaining)
        else:
            leftovers.append(p_idx)

    chosen = np.concatenate(chosen) if len(chosen) > 0 else np.array([], dtype=int)

    # Fill remaining slots from the leftover pool
    need = n - len(chosen)
    if need > 0 and len(leftovers) > 0:
        pool = np.concatenate(leftovers)
        extra_n = min(need, len(pool))
        extra = rng.choice(pool, size=extra_n, replace=False)
        chosen = np.concatenate([chosen, extra])

    # Final safeguard
    if len(chosen) > n:
        chosen = rng.choice(chosen, size=n, replace=False)

    return np.sort(chosen)


In [ ]:
# Min-Max transformation for visualization
def minmax_scale(x):

    x = np.asarray(
        x,
        dtype=float,
    )

    xmin = np.nanmin(x)
    xmax = np.nanmax(x)

    if np.isclose(
        xmin,
        xmax,
    ):
        raise ValueError(
            "Cannot min-max scale a constant variable."
        )

    return (
        x - xmin
    ) / (
        xmax - xmin
    )

## Helper: linear-origin model

In [ ]:
# Linear vs saturating model comparison
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit


def linear_origin(
    K,
    slope
):
    """
    Linear accumulation constrained through E(0)=0
    """

    return slope * K


## Helper: saturation-origin model

In [ ]:
def saturating_origin(
    K,
    amplitude,
    tau
):
    """
    Exponential saturation constrained through E(0)=0

    amplitude:
        asymptotic efficacy

    tau:
        characteristic feature-number scale
    """

    return amplitude * (
        1.0
        - np.exp(
            -K / tau
        )
    )

## Helper: AICc

In [ ]:
def calculate_aicc(
    y,
    yhat,
    n_parameters
):
    """
    Corrected Akaike Information Criterion

    Appropriate here because each titration curve
    contains only a small number of K values
    """

    y = np.asarray(
        y,
        dtype=float
    )

    yhat = np.asarray(
        yhat,
        dtype=float
    )

    n = len(y)
    k = n_parameters

    residual = (
        y - yhat
    )

    sse = np.sum(
        residual ** 2
    )

    # Prevent log(0) in an essentially perfect fit
    sse = max(
        float(sse),
        1e-12
    )

    aic = (
        n
        * np.log(
            sse / n
        )
        +
        2 * k
    )

    if (
        n - k - 1
    ) <= 0:

        return np.inf

    aicc = (
        aic
        +
        (
            2 * k * (k + 1)
        )
        /
        (
            n - k - 1
        )
    )

    return float(
        aicc
    )


In [ ]:
def compare_titration_models(
    curve_df
):
    """
    Compare continuing linear accumulation against
    exponential saturation

    Uses median_top_efficacy as the response

    Returns one summary row plus fitted curves
    """

    tmp = (
        curve_df[
            [
                "K",
                "median_top_efficacy"
            ]
        ]
        .dropna()
        .sort_values("K")
        .copy()
    )

    K = tmp[
        "K"
    ].to_numpy(
        dtype=float
    )

    y = tmp[
        "median_top_efficacy"
    ].to_numpy(
        dtype=float
    )

    # Model 1: linear
    linear_params, _ = curve_fit(
        linear_origin,
        K,
        y,
        p0=[
            max(
                y[-1] / K[-1],
                1e-6
            )
        ],
        bounds=(
            [0],
            [np.inf]
        ),
        maxfev=10000
    )

    yhat_linear = linear_origin(
        K,
        *linear_params
    )

    linear_aicc = calculate_aicc(
        y,
        yhat_linear,
        n_parameters=1
    )

    # Model 2: saturating
    initial_amplitude = max(
        np.max(y) * 1.2,
        1e-3
    )

    initial_tau = np.median(
        K
    )

    try:

        sat_params, _ = curve_fit(
            saturating_origin,
            K,
            y,
            p0=[
                initial_amplitude,
                initial_tau
            ],
            bounds=(
                [
                    0,
                    1e-6
                ],
                [
                    np.inf,
                    np.inf
                ]
            ),
            maxfev=50000
        )

        amplitude = float(
            sat_params[0]
        )

        tau = float(
            sat_params[1]
        )

        yhat_sat = saturating_origin(
            K,
            *sat_params
        )


        sat_aicc = calculate_aicc(
            y,
            yhat_sat,
            n_parameters=2
        )

        K90 = (
            tau
            * np.log(10)
        )

    except Exception as e:

        amplitude = np.nan
        tau = np.nan
        K90 = np.nan

        sat_aicc = np.inf

        yhat_sat = np.full_like(
            y,
            np.nan
        )

    # Model comparison

    # Positive:
    # saturation model has LOWER / better AICc
    delta_aicc = (
        linear_aicc
        - sat_aicc
    )

    if np.isfinite(
        delta_aicc
    ):

        saturation_weight = (
            1
            /
            (
                1
                +
                np.exp(
                    -0.5
                    * delta_aicc
                )
            )
        )

    else:

        saturation_weight = np.nan

    # Descriptive evidence category
    if not np.isfinite(
        sat_aicc
    ):

        conclusion = (
            "saturation_fit_failed"
        )

    elif delta_aicc >= 10:

        if K90 <= K.max():

            conclusion = (
                "strong_saturation_within_range"
            )

        else:

            conclusion = (
                "strong_saturation_beyond_range"
            )


    elif delta_aicc >= 4:

        if K90 <= K.max():

            conclusion = (
                "moderate_saturation_within_range"
            )

        else:

            conclusion = (
                "moderate_saturation_beyond_range"
            )


    elif delta_aicc <= -4:

        conclusion = (
            "linear_accumulation_favored"
        )

    else:

        conclusion = (
            "no_clear_model_preference"
        )

    return {

        "linear_slope":
            float(
                linear_params[0]
            ),

        "linear_AICc":
            linear_aicc,

        "sat_amplitude":
            amplitude,

        "sat_tau":
            tau,

        "sat_K90":
            K90,

        "sat_AICc":
            sat_aicc,

        "delta_AICc_linear_minus_sat":
            delta_aicc,

        "saturation_model_weight":
            saturation_weight,

        "K90_within_tested_range":
            (
                bool(
                    K90 <= K.max()
                )
                if np.isfinite(K90)
                else False
            ),

        "conclusion":
            conclusion,

        "observed_K":
            K,

        "observed_y":
            y,

        "linear_fit":
            yhat_linear,

        "saturation_fit":
            yhat_sat
    }

In [ ]:
# Construct conserved intersection evidence table

def extract_disease_evidence(
    disease,
    prefix
):

    df = (
        full_attr_tables[
            disease
        ]
        .sort_values(
            "median_attribution",
            ascending=False
        )
        .reset_index(
            drop=True
        )
        .copy()
    )

    df[
        f"{prefix}_rank"
    ] = np.arange(
        1,
        len(df) + 1
    )


    keep_cols = [
        "gene",
        "median_attribution",
        "attr_iqr",
        "positive_seed_fraction",
        f"{prefix}_rank"
    ]

    df = df[
        keep_cols
    ].copy()


    df = df.rename(
        columns={
            "median_attribution":
                f"{prefix}_attribution",

            "attr_iqr":
                f"{prefix}_attr_iqr",

            "positive_seed_fraction":
                f"{prefix}_positive_seed_fraction"
        }
    )

    return df

# 3. Clinical risk localization

In [ ]:
from pynndescent import NNDescent

# Metadata columns
DISEASE_COL = "disease"
PATIENT_COL = "patient_id"
SAMPLE_COL = "sample_id"
SEVERITY_COL = "severity"
STAGE_COL = "sampling_time"

# Exact disease labels
CART = "CAR-T_CRS"
COVID = "COVID19"
SLE = "SLE"

# Construct formal anchor labels

n_cells = adata_m.n_obs

disease_arr = (
    adata_m.obs[DISEASE_COL]
    .astype(str)
    .to_numpy()
)

severity_arr = (
    adata_m.obs[SEVERITY_COL]
    .astype("string")
    .fillna("NA")
    .astype(str)
    .to_numpy()
)

stage_arr = (
    adata_m.obs[STAGE_COL]
    .astype("string")
    .fillna("NA")
    .astype(str)
    .to_numpy()
)


anchor_group = np.full(
    n_cells,
    "query_only",
    dtype=object
)


# ============================================================
# CAR-T CRS
#
# Risk:
#   Severe × progression
#
# Stage reference:
#   No_CRS × progression
#
# Temporal reference:
#   Severe × before/convalescence
# ============================================================

mask = (
    (disease_arr == CART)
    & (severity_arr == "Severe")
    & (stage_arr == "CAR-T_CRS_pro")
)
anchor_group[mask] = "risk"


mask = (
    (disease_arr == CART)
    & (severity_arr == "No_CRS")
    & (stage_arr == "CAR-T_CRS_pro")
)
anchor_group[mask] = "stage_ref"


mask = (
    (disease_arr == CART)
    & (severity_arr == "Severe")
    & np.isin(
        stage_arr,
        [
            "CAR-T_CRS_before",
            "CAR-T_CRS_con"
        ]
    )
)
anchor_group[mask] = "temporal_ref"


# ============================================================
# COVID-19
#
# Risk:
#   Severe × progression
#
# Stage reference:
#   Moderate × progression
#
# Temporal reference:
#   Severe × convalescence
# ============================================================

mask = (
    (disease_arr == COVID)
    & (severity_arr == "Severe")
    & (stage_arr == "COVID19_pro")
)
anchor_group[mask] = "risk"


mask = (
    (disease_arr == COVID)
    & (severity_arr == "Moderate")
    & (stage_arr == "COVID19_pro")
)
anchor_group[mask] = "stage_ref"


mask = (
    (disease_arr == COVID)
    & (severity_arr == "Severe")
    & (stage_arr == "COVID19_con")
)
anchor_group[mask] = "temporal_ref"


# ============================================================
# SLE
#
# Risk:
#   Severe
#
# Reference:
#   Moderate
#
# Severity-missing cells are retained as query-only cells.
# ============================================================

mask = (
    (disease_arr == SLE)
    & (severity_arr == "Severe")
)
anchor_group[mask] = "risk"


mask = (
    (disease_arr == SLE)
    & (severity_arr == "Moderate")
)
anchor_group[mask] = "severity_ref"


adata_m.obs["Fig6_anchor_group"] = pd.Categorical(
    anchor_group,
    categories=[
        "query_only",
        "risk",
        "stage_ref",
        "temporal_ref",
        "severity_ref"
    ]
)

In [ ]:
# Check anchor structure

anchor_summary = (
    adata_m.obs[
        adata_m.obs["Fig6_anchor_group"]
        != "query_only"
    ]
    .groupby(
        [
            DISEASE_COL,
            "Fig6_anchor_group"
        ],
        observed=True
    )
    .agg(
        n_patients=(
            PATIENT_COL,
            "nunique"
        ),
        n_samples=(
            SAMPLE_COL,
            "nunique"
        ),
        n_cells=(
            PATIENT_COL,
            "size"
        )
    )
    .reset_index()
)

anchor_summary

In [ ]:
# Fixed localization parameters
ANCHOR_CELLS_PER_PATIENT_GROUP = 500

K_CANDIDATE = 100

CAP_PER_PATIENT_GROUP = 3

INDEX_N_NEIGHBORS = 50

QUERY_BATCH_SIZE = 20000

QUERY_EPSILON = 0.10

ANCHOR_SEED = 20260811
ANN_SEED = 20260811

In [ ]:
# Create a globally unique disease/patient key
patient_key = (
    adata_m.obs[DISEASE_COL].astype(str)
    + "||"
    + adata_m.obs[PATIENT_COL].astype(str)
)

patient_codes, patient_levels = pd.factorize(
    patient_key,
    sort=True
)

patient_codes = patient_codes.astype(
    np.int32
)

In [ ]:
# Build one fixed balanced anchor set, so the same cell indices will be reused for all five representation seeds
def build_balanced_anchor_pool(
    adata,
    max_cells_per_patient_group=500,
    random_seed=20260811
):

    rng = np.random.default_rng(
        random_seed
    )

    tmp = adata.obs[
        [
            DISEASE_COL,
            PATIENT_COL,
            SAMPLE_COL,
            "Fig6_anchor_group"
        ]
    ].copy()

    tmp["_global_idx"] = np.arange(
        adata.n_obs,
        dtype=np.int64
    )

    tmp = tmp[
        tmp["Fig6_anchor_group"]
        .astype(str)
        != "query_only"
    ]


    selected = []

    for (
        disease,
        anchor,
        patient
    ), df in tmp.groupby(
        [
            DISEASE_COL,
            "Fig6_anchor_group",
            PATIENT_COL
        ],
        observed=True,
        sort=False
    ):

        idx = df[
            "_global_idx"
        ].to_numpy(
            dtype=np.int64
        )

        n_take = min(
            len(idx),
            max_cells_per_patient_group
        )

        if n_take < len(idx):

            idx = rng.choice(
                idx,
                size=n_take,
                replace=False
            )

        selected.append(idx)

    return np.concatenate(
        selected
    )

balanced_anchor_idx = (
    build_balanced_anchor_pool(
        adata_m,
        max_cells_per_patient_group=
            ANCHOR_CELLS_PER_PATIENT_GROUP,
        random_seed=ANCHOR_SEED
    )
)

print(
    f"Balanced anchor cells: "
    f"{len(balanced_anchor_idx):,}"
)

In [ ]:
# Check
balanced_anchor_meta = (
    adata_m.obs.iloc[
        balanced_anchor_idx
    ][
        [
            DISEASE_COL,
            PATIENT_COL,
            "Fig6_anchor_group"
        ]
    ]
    .copy()
)

balanced_anchor_check = (
    balanced_anchor_meta
    .groupby(
        [
            DISEASE_COL,
            "Fig6_anchor_group",
            PATIENT_COL
        ],
        observed=True
    )
    .size()
    .rename("n_anchor_cells")
    .reset_index()
)

balanced_anchor_check.groupby(
    [
        DISEASE_COL,
        "Fig6_anchor_group"
    ],
    observed=True
)["n_anchor_cells"].describe()

In [ ]:
GROUP_CODE = {
    "risk": 0,
    "stage_ref": 1,
    "temporal_ref": 2,
    "severity_ref": 3
}

DISEASE_CONFIG = {

    CART: {
        "groups": [
            "risk",
            "stage_ref",
            "temporal_ref"
        ]
    },

    COVID: {
        "groups": [
            "risk",
            "stage_ref",
            "temporal_ref"
        ]
    },

    SLE: {
        "groups": [
            "risk",
            "severity_ref"
        ]
    }
}

In [ ]:
# The scoring function:

def localize_risk_one_seed_one_disease(
    seed,
    disease,
    verbose=True
):

    # Frozen latent representation
    Z = Z_by_seed[seed]

    # Query cells = ALL cells from this disease
    query_idx = np.where(
        disease_arr == disease
    )[0]

    # Balanced clinical-anchor pool for this disease
    ref_idx = balanced_anchor_idx[
        disease_arr[
            balanced_anchor_idx
        ] == disease
    ]

    Z_ref = np.asarray(
        Z[ref_idx, :],
        dtype=np.float32
    )

    ref_patient = (
        patient_codes[
            ref_idx
        ]
    )

    ref_group_string = (
        anchor_group[
            ref_idx
        ]
    )

    ref_group_code = np.array(
        [
            GROUP_CODE[g]
            for g in ref_group_string
        ],
        dtype=np.int16
    )

    groups = (
        DISEASE_CONFIG[
            disease
        ]["groups"]
    )

    # Which patients are available in each clinical group?

    group_patient_sets = {}

    for group in groups:

        code = GROUP_CODE[
            group
        ]

        group_patient_sets[group] = set(
            np.unique(
                ref_patient[
                    ref_group_code == code
                ]
            ).tolist()
        )

    if verbose:

        print(
            f"\n{disease} | "
            f"query cells={len(query_idx):,} | "
            f"anchor cells={len(ref_idx):,}"
        )

        for group in groups:

            print(
                f"  {group}: "
                f"{len(group_patient_sets[group])} "
                f"patients"
            )

    # Approximate-neighbor index
    index = NNDescent(
        Z_ref,
        metric="euclidean",
        n_neighbors=
            INDEX_N_NEIGHBORS,
        random_state=
            ANN_SEED,
        n_jobs=-1,
        low_memory=True,
        parallel_batch_queries=True,
        verbose=verbose
    )

    # Output arrays
    support = {
        group: np.zeros(
            len(query_idx),
            dtype=np.float32
        )
        for group in groups
    }

    # Composite integer key:
    # group_code * PATIENT_MOD + patient_code
    PATIENT_MOD = (
        int(patient_codes.max())
        + 1
    )

    # Query in manageable batches
    for start in range(
        0,
        len(query_idx),
        QUERY_BATCH_SIZE
    ):

        end = min(
            start + QUERY_BATCH_SIZE,
            len(query_idx)
        )

        q_global = query_idx[
            start:end
        ]

        q_Z = np.asarray(
            Z[q_global, :],
            dtype=np.float32
        )

        k_use = min(
            K_CANDIDATE,
            len(ref_idx)
        )

        nn_idx, _ = index.query(
            q_Z,
            k=k_use,
            epsilon=QUERY_EPSILON
        )

        candidate_patient = (
            ref_patient[
                nn_idx
            ]
        )

        candidate_group = (
            ref_group_code[
                nn_idx
            ]
        )

        query_patient = (
            patient_codes[
                q_global
            ]
        )

        # Score each query cell
        for local_i in range(
            len(q_global)
        ):

            qp = int(
                query_patient[
                    local_i
                ]
            )

            # COMPLETE leave-one-patient-out exclusion
            valid = (
                candidate_patient[
                    local_i
                ]
                != qp
            )

            cand_p = (
                candidate_patient[
                    local_i,
                    valid
                ]
            )

            cand_g = (
                candidate_group[
                    local_i,
                    valid
                ]
            )

            # Count neighbors for each:
            # clinical group × patient

            composite = (
                cand_g.astype(
                    np.int64
                )
                * PATIENT_MOD
                + cand_p.astype(
                    np.int64
                )
            )

            unique_key, counts = (
                np.unique(
                    composite,
                    return_counts=True
                )
            )

            unique_group = (
                unique_key
                // PATIENT_MOD
            )

            # Patient-balanced support per group
            for group in groups:

                code = GROUP_CODE[
                    group
                ]

                # Number of eligible patients after
                # excluding query patient
                n_available = len(
                    group_patient_sets[
                        group
                    ]
                )

                if (
                    qp
                    in group_patient_sets[
                        group
                    ]
                ):
                    n_available -= 1


                if n_available <= 0:

                    support[group][
                        start + local_i
                    ] = np.nan

                    continue

                group_counts = counts[
                    unique_group
                    == code
                ]

                # Each patient/context contributes:
                #
                # 0/3, 1/3, 2/3, or 3/3 maximum.
                capped_evidence = (
                    np.minimum(
                        group_counts,
                        CAP_PER_PATIENT_GROUP
                    )
                    .sum()
                )

                support[group][
                    start + local_i
                ] = (
                    capped_evidence
                    /
                    (
                        CAP_PER_PATIENT_GROUP
                        * n_available
                    )
                )

        if verbose:

            print(
                f"  queried "
                f"{end:,}/{len(query_idx):,}"
            )

    # Convert clinical supports into risk axes
    result = pd.DataFrame({
        "global_idx": query_idx,
        "seed": seed,
        "disease": disease,
        "risk_support":
            support["risk"]
    })

    if disease in [
        CART,
        COVID
    ]:

        result[
            "stage_ref_support"
        ] = support[
            "stage_ref"
        ]

        result[
            "temporal_ref_support"
        ] = support[
            "temporal_ref"
        ]


        result[
            "stage_axis"
        ] = (
            support["risk"]
            - support["stage_ref"]
        )


        result[
            "temporal_axis"
        ] = (
            support["risk"]
            - support["temporal_ref"]
        )


        # Equal weight to the two independent
        # clinical contrasts.
        result[
            "risk_score"
        ] = (
            result["stage_axis"]
            + result["temporal_axis"]
        ) / 2


    else:

        result[
            "severity_ref_support"
        ] = support[
            "severity_ref"
        ]


        result[
            "severity_axis"
        ] = (
            support["risk"]
            - support[
                "severity_ref"
            ]
        )


        result[
            "risk_score"
        ] = result[
            "severity_axis"
        ]


    return result

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import gc

# Output directory
LOCALIZATION_DIR = Path("Fig6_risk_localization")
LOCALIZATION_DIR.mkdir(exist_ok=True)

# Fixed disease order
LOCALIZATION_DISEASES = [
    CART,
    COVID,
    SLE
]

# Allocate matrices for all five seeds
# Rows = cells within disease
# Columns = independent MAE seeds

localization_store = {}

for disease in LOCALIZATION_DISEASES:

    disease_idx = np.where(
        disease_arr == disease
    )[0]

    localization_store[disease] = {
        "global_idx": disease_idx,

        "risk_score": np.full(
            (len(disease_idx), len(FINAL_SEEDS)),
            np.nan,
            dtype=np.float32
        )
    }

    # CAR-T/COVID have two independent axes
    if disease in [CART, COVID]:

        localization_store[disease][
            "stage_axis"
        ] = np.full(
            (len(disease_idx), len(FINAL_SEEDS)),
            np.nan,
            dtype=np.float32
        )

        localization_store[disease][
            "temporal_axis"
        ] = np.full(
            (len(disease_idx), len(FINAL_SEEDS)),
            np.nan,
            dtype=np.float32
        )

    # SLE has one severity axis
    else:

        localization_store[disease][
            "severity_axis"
        ] = np.full(
            (len(disease_idx), len(FINAL_SEEDS)),
            np.nan,
            dtype=np.float32
        )

In [ ]:
# Full five-seed localization

for seed_i, seed in enumerate(FINAL_SEEDS):

    print(
        "\n"
        + "#" * 75
    )
    print(
        f"LOCALIZATION SEED {seed} "
        f"({seed_i + 1}/{len(FINAL_SEEDS)})"
    )
    print(
        "#" * 75
    )

    for disease in LOCALIZATION_DISEASES:

        print(
            f"\nRunning {disease}..."
        )

        result = (
            localize_risk_one_seed_one_disease(
                seed=seed,
                disease=disease,
                verbose=True
            )
        )

        # Critical row-alignment check
        expected_idx = (
            localization_store[
                disease
            ]["global_idx"]
        )

        result_idx = (
            result["global_idx"]
            .to_numpy()
        )

        if not np.array_equal(
            expected_idx,
            result_idx
        ):
            raise RuntimeError(
                f"Cell-order mismatch for "
                f"{disease}, seed {seed}."
            )

        # Store risk score
        localization_store[
            disease
        ]["risk_score"][
            :,
            seed_i
        ] = (
            result["risk_score"]
            .to_numpy(
                dtype=np.float32
            )
        )

        # Store disease-specific component axes
        if disease in [
            CART,
            COVID
        ]:

            localization_store[
                disease
            ]["stage_axis"][
                :,
                seed_i
            ] = (
                result["stage_axis"]
                .to_numpy(
                    dtype=np.float32
                )
            )

            localization_store[
                disease
            ]["temporal_axis"][
                :,
                seed_i
            ] = (
                result["temporal_axis"]
                .to_numpy(
                    dtype=np.float32
                )
            )

        else:

            localization_store[
                disease
            ]["severity_axis"][
                :,
                seed_i
            ] = (
                result["severity_axis"]
                .to_numpy(
                    dtype=np.float32
                )
            )

        # Save this individual seed immediately
        # This means a kernel crash later does NOT require
        # rerunning already-completed localization seeds.
        save_dict = {
            "global_idx":
                result_idx,

            "risk_score":
                result[
                    "risk_score"
                ].to_numpy(
                    dtype=np.float32
                )
        }

        if disease in [
            CART,
            COVID
        ]:

            save_dict[
                "stage_axis"
            ] = result[
                "stage_axis"
            ].to_numpy(
                dtype=np.float32
            )

            save_dict[
                "temporal_axis"
            ] = result[
                "temporal_axis"
            ].to_numpy(
                dtype=np.float32
            )

        else:

            save_dict[
                "severity_axis"
            ] = result[
                "severity_axis"
            ].to_numpy(
                dtype=np.float32
            )


        np.savez_compressed(
            LOCALIZATION_DIR
            /
            f"Fig6_localization_"
            f"{disease}_seed_{seed}.npz",

            **save_dict
        )


        del result
        gc.collect()

In [ ]:
# Five-seed consensus
consensus_localization = {}


for disease in LOCALIZATION_DISEASES:

    store = localization_store[
        disease
    ]

    risk_matrix = store[
        "risk_score"
    ]

    # Primary consensus score
    median_risk = np.nanmedian(
        risk_matrix,
        axis=1
    )

    # Seed variability
    # IQR is robust to one unusual seed.
    q25_risk = np.nanquantile(
        risk_matrix,
        0.25,
        axis=1
    )

    q75_risk = np.nanquantile(
        risk_matrix,
        0.75,
        axis=1
    )

    risk_iqr = (
        q75_risk
        - q25_risk
    )


    consensus = {
        "global_idx":
            store["global_idx"],

        "risk_score":
            median_risk.astype(
                np.float32
            ),    # we use median, rather than mean, to reduce the influence of one unusual seed

        "risk_seed_iqr":
            risk_iqr.astype(
                np.float32
            )
    }

    # Consensus component axes
    if disease in [
        CART,
        COVID
    ]:

        consensus[
            "stage_axis"
        ] = np.nanmedian(
            store["stage_axis"],
            axis=1
        ).astype(
            np.float32
        )

        consensus[
            "temporal_axis"
        ] = np.nanmedian(
            store["temporal_axis"],
            axis=1
        ).astype(
            np.float32
        )

    else:

        consensus[
            "severity_axis"
        ] = np.nanmedian(
            store["severity_axis"],
            axis=1
        ).astype(
            np.float32
        )


    consensus_localization[
        disease
    ] = consensus

In [ ]:
# Save consensus arrays

for disease in LOCALIZATION_DISEASES:

    np.savez_compressed(
        LOCALIZATION_DIR
        /
        f"Fig6_localization_{disease}_CONSENSUS.npz",

        **consensus_localization[
            disease
        ]
    )


print(
    "Five-seed consensus localization saved."
)

In [ ]:
# Add disease-specific consensus scores to adata_m.obs
adata_m.obs[
    "Fig6_risk_score"
] = np.nan

adata_m.obs[
    "Fig6_risk_seed_iqr"
] = np.nan


# Separate component axes
adata_m.obs[
    "Fig6_stage_axis"
] = np.nan

adata_m.obs[
    "Fig6_temporal_axis"
] = np.nan

adata_m.obs[
    "Fig6_severity_axis"
] = np.nan


for disease in LOCALIZATION_DISEASES:

    result = (
        consensus_localization[
            disease
        ]
    )

    idx = result[
        "global_idx"
    ]


    adata_m.obs.iloc[
        idx,
        adata_m.obs.columns.get_loc(
            "Fig6_risk_score"
        )
    ] = result[
        "risk_score"
    ]


    adata_m.obs.iloc[
        idx,
        adata_m.obs.columns.get_loc(
            "Fig6_risk_seed_iqr"
        )
    ] = result[
        "risk_seed_iqr"
    ]


    if disease in [
        CART,
        COVID
    ]:

        adata_m.obs.iloc[
            idx,
            adata_m.obs.columns.get_loc(
                "Fig6_stage_axis"
            )
        ] = result[
            "stage_axis"
        ]

        adata_m.obs.iloc[
            idx,
            adata_m.obs.columns.get_loc(
                "Fig6_temporal_axis"
            )
        ] = result[
            "temporal_axis"
        ]

    else:

        adata_m.obs.iloc[
            idx,
            adata_m.obs.columns.get_loc(
                "Fig6_severity_axis"
            )
        ] = result[
            "severity_axis"
        ]

In [ ]:
from scipy.stats import spearmanr
from itertools import combinations

seed_stability_records = []


for disease in LOCALIZATION_DISEASES:

    matrix = (
        localization_store[
            disease
        ]["risk_score"]
    )


    for i, j in combinations(
        range(len(FINAL_SEEDS)),
        2
    ):

        rho, _ = spearmanr(
            matrix[:, i],
            matrix[:, j],
            nan_policy="omit"
        )


        seed_stability_records.append({

            "disease":
                disease,

            "seed_a":
                FINAL_SEEDS[i],

            "seed_b":
                FINAL_SEEDS[j],

            "spearman_r":
                rho
        })


risk_seed_stability = pd.DataFrame(
    seed_stability_records
)


risk_seed_stability

In [ ]:
risk_seed_stability.groupby(
    "disease",
    observed=True
)["spearman_r"].describe()

In [ ]:
risk_seed_stability.to_csv(
    LOCALIZATION_DIR
    /
    "Fig6_risk_localization_seed_stability.csv",
    index=False
)

In [ ]:
# One row per sample
sample_consensus = (
    adata_m.obs[
        [
            SAMPLE_COL,
            PATIENT_COL,
            DISEASE_COL,
            SEVERITY_COL,
            STAGE_COL,
            "Fig6_risk_score",
            "Fig6_risk_seed_iqr"
        ]
    ]
    .groupby(
        [
            SAMPLE_COL,
            PATIENT_COL,
            DISEASE_COL,
            SEVERITY_COL,
            STAGE_COL
        ],
        observed=True,
        dropna=False
    )
    .agg(

        mean_risk_score=(
            "Fig6_risk_score",
            "mean"
        ),

        median_risk_score=(
            "Fig6_risk_score",
            "median"
        ),

        mean_seed_iqr=(
            "Fig6_risk_seed_iqr",
            "mean"
        ),

        n_monocytes=(
            "Fig6_risk_score",
            "size"
        )
    )
    .reset_index()
)


# One row per disease / severity / sampling-time context
context_consensus = (
    adata_m.obs[
        [
            DISEASE_COL,
            SEVERITY_COL,
            STAGE_COL,
            "Fig6_risk_score",
            "Fig6_risk_seed_iqr",
        ]
    ]
    .groupby(
        [
            DISEASE_COL,
            SEVERITY_COL,
            STAGE_COL,
        ],
        observed=True,
        dropna=False,
    )
    .agg(
        mean_risk_score=("Fig6_risk_score", "mean"),
        median_risk_score=("Fig6_risk_score", "median"),
        mean_seed_iqr=("Fig6_risk_seed_iqr", "mean"),
        n_monocytes=("Fig6_risk_score", "size"),
    )
    .reset_index()
)



In [ ]:
sample_consensus.to_csv(
    LOCALIZATION_DIR
    /
    "Fig6_consensus_sample_risk_scores.csv",
    index=False
)

context_consensus.to_csv(
    LOCALIZATION_DIR
    /
    "Fig6_consensus_context_summary.csv",
    index=False
)

# 4. Fig. 6B MAE training convergence

In [ ]:
from pathlib import Path
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

ROOT = resolve_analysis_path(os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai")))
HISTORY_DIR = ROOT / "Fig6_MAE_history"
OUT_DIR = ROOT / "Fig6"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [
    20260810,
    20260811,
    20260812,
    20260813,
    20260814,
]

N_EPOCHS = 30

LOSS_KEY_CANDIDATES = [
    "epoch_loss",
    "epoch_losses",
    "train_loss",
    "train_losses",
    "training_loss",
    "training_losses",
    "mean_train_loss",
    "mean_epoch_loss",
    "loss",
    "losses",
]

In [ ]:
def find_history_file(seed):
    """Locate one saved epoch-level training-history file for a final MAE seed."""

    seed_text = str(seed)
    supported_suffixes = {".csv", ".json", ".npy", ".npz"}

    matches = [
        path
        for path in HISTORY_DIR.rglob("*")
        if (
            path.is_file()
            and seed_text in str(path)
            and path.suffix.lower() in supported_suffixes
        )
    ]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No MAE history file found for seed {seed} under:\n{HISTORY_DIR}"
        )

    matches = sorted(
        matches,
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    if len(matches) > 1:
        print(
            f"Multiple histories found for seed {seed}; using newest:\n{matches[0]}"
        )

    return matches[0]


In [ ]:
def valid_epoch_vector(values):
    """Return a finite 30-epoch loss vector, or None if the object is not valid."""

    try:
        arr = np.asarray(values, dtype=float).reshape(-1)
    except Exception:
        return None

    if len(arr) != N_EPOCHS:
        return None

    if not np.all(np.isfinite(arr)):
        return None

    return arr


def extract_from_mapping(obj):
    """Recursively identify a saved training-loss vector in a mapping."""

    if not isinstance(obj, dict):
        return None

    lower_to_original = {str(key).lower(): key for key in obj.keys()}

    for candidate in LOSS_KEY_CANDIDATES:
        if candidate in lower_to_original:
            key = lower_to_original[candidate]
            arr = valid_epoch_vector(obj[key])
            if arr is not None:
                return arr, str(key)

    for key, value in obj.items():
        if isinstance(value, dict):
            result = extract_from_mapping(value)
            if result is not None:
                arr, nested_key = result
                return arr, f"{key}.{nested_key}"

    return None


In [ ]:
def load_epoch_losses(path):
    """Read a 30-epoch training-loss vector from CSV, JSON, NPY, or NPZ history."""

    suffix = path.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(path)
        lower_to_original = {col.lower(): col for col in df.columns}

        for candidate in LOSS_KEY_CANDIDATES:
            if candidate in lower_to_original:
                col = lower_to_original[candidate]
                arr = valid_epoch_vector(df[col])
                if arr is not None:
                    return arr, col

        raise ValueError(
            f"Could not identify a 30-epoch training-loss column in:\n{path}\n"
            f"Columns:\n{df.columns.tolist()}"
        )

    if suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        result = extract_from_mapping(obj)
        if result is not None:
            return result

        arr = valid_epoch_vector(obj)
        if arr is not None:
            return arr, "<root list>"

        raise ValueError(f"Could not identify a 30-epoch history in:\n{path}")

    if suffix == ".npy":
        obj = np.load(path, allow_pickle=True)

        if obj.shape == () and obj.dtype == object:
            result = extract_from_mapping(obj.item())
            if result is not None:
                return result

        arr = valid_epoch_vector(obj)
        if arr is not None:
            return arr, "<root array>"

        raise ValueError(f"Could not identify a 30-epoch history in:\n{path}")

    if suffix == ".npz":
        npz = np.load(path, allow_pickle=True)
        lower_to_original = {key.lower(): key for key in npz.files}

        for candidate in LOSS_KEY_CANDIDATES:
            if candidate in lower_to_original:
                key = lower_to_original[candidate]
                arr = valid_epoch_vector(npz[key])
                if arr is not None:
                    return arr, key

        raise ValueError(
            f"Could not identify a 30-epoch training-loss history in:\n{path}\n"
            f"NPZ keys:\n{npz.files}"
        )

    raise ValueError(f"Unsupported history format:\n{path}")


In [ ]:
history_rows = []

for seed in SEEDS:
    path = find_history_file(seed)
    losses, source_field = load_epoch_losses(path)

    print(f"\nSeed {seed}")
    print(f"  source: {path}")
    print(f"  field: {source_field}")
    print(f"  epoch 1 MSE : {losses[0]:.8f}")
    print(f"  epoch 30 MSE: {losses[-1]:.8f}")

    for epoch, loss in enumerate(losses, start=1):
        history_rows.append({
            "seed": seed,
            "epoch": epoch,
            "training_loss": float(loss),
            "source_file": str(path),
            "source_field": source_field,
        })

history_df = pd.DataFrame(history_rows)


In [ ]:
epoch_counts = history_df.groupby("seed")["epoch"].nunique()

if not (epoch_counts == N_EPOCHS).all():
    raise RuntimeError(
        "At least one seed does not contain exactly 30 epoch-level training losses."
    )

if set(history_df["seed"].unique()) != set(SEEDS):
    raise RuntimeError("Loaded seed IDs do not match the five final MAE seeds.")

seed_stats = []

for seed in SEEDS:
    sub = history_df.loc[history_df["seed"] == seed].sort_values("epoch")
    mse_epoch1 = float(sub.loc[sub["epoch"] == 1, "training_loss"].iloc[0])
    mse_epoch30 = float(sub.loc[sub["epoch"] == 30, "training_loss"].iloc[0])
    mse_reduction_pct = (mse_epoch1 - mse_epoch30) / mse_epoch1 * 100

    seed_stats.append({
        "seed": seed,
        "mse_epoch1": mse_epoch1,
        "mse_epoch30": mse_epoch30,
        "mse_reduction_pct": mse_reduction_pct,
    })

seed_stats_df = pd.DataFrame(seed_stats)

summary_text = (
    f"Epoch 30 MSE:  {seed_stats_df['mse_epoch30'].median():.4f} "
    f"({seed_stats_df['mse_epoch30'].min():.4f}-{seed_stats_df['mse_epoch30'].max():.4f})"
    "\n"
    f"MSE reduction:  {seed_stats_df['mse_reduction_pct'].median():.1f}% "
    f"({seed_stats_df['mse_reduction_pct'].min():.1f}-{seed_stats_df['mse_reduction_pct'].max():.1f}%)"
)

print("\n" + "=" * 60)
print("CROSS-SEED TRAINING SUMMARY")
print("=" * 60)
print(summary_text)


In [ ]:
CURVE_RANK_START_EPOCH = 5

curve_position_rows = []

for seed in SEEDS:
    sub = history_df.loc[history_df["seed"] == seed].sort_values("epoch")
    stable_mean_loss = float(
        sub.loc[sub["epoch"] >= CURVE_RANK_START_EPOCH, "training_loss"].mean()
    )
    curve_position_rows.append({
        "seed": seed,
        "stable_mean_loss": stable_mean_loss,
    })

curve_position_df = (
    pd.DataFrame(curve_position_rows)
    .sort_values("stable_mean_loss", ascending=True)
    .reset_index(drop=True)
)

curve_position_df["alpha"] = [0.50, 0.60, 0.70, 0.80, 0.90]
curve_position_df["linewidth"] = [3.40, 3.80, 4.20, 4.60, 5.00]
curve_position_df["zorder"] = [5, 4, 3, 2, 1]

alpha_map = dict(zip(curve_position_df["seed"], curve_position_df["alpha"]))
linewidth_map = dict(zip(curve_position_df["seed"], curve_position_df["linewidth"]))
zorder_map = dict(zip(curve_position_df["seed"], curve_position_df["zorder"]))

print("\nCurve visibility assignment: BOTTOM -> TOP")
print(curve_position_df)


In [ ]:
seed_colors = {
    20260810: "#4C78A8",
    20260811: "#F58518",
    20260812: "#C43753",
    20260813: "#54A24B",
    20260814: "#C881FF",
}

fig, ax = plt.subplots(figsize=(5.6, 4.1))

plot_order_top_to_bottom = curve_position_df["seed"].iloc[::-1].tolist()

for seed in plot_order_top_to_bottom:
    seed = int(seed)
    sub = history_df.loc[history_df["seed"] == seed].sort_values("epoch")

    ax.plot(
        sub["epoch"],
        sub["training_loss"],
        color=seed_colors[seed],
        linewidth=linewidth_map[seed],
        alpha=alpha_map[seed],
        zorder=zorder_map[seed],
        label=str(seed),
    )

ax.text(
    0.50,
    1.03,
    summary_text,
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=8.5,
    linespacing=1.4,
    bbox=dict(
        boxstyle="round,pad=0.45",
        facecolor="white",
        edgecolor="0.70",
        linewidth=0.8,
    ),
)


In [ ]:
ax.set_xlim(1, N_EPOCHS)
ax.set_xticks([1, 5, 10, 15, 20, 25, 30])
ax.set_xlabel("Training epoch")
ax.set_ylabel("Masked-expression reconstruction loss (MSE)")
ax.set_title(
    "MAE training convergence across five seeds",
    fontweight="bold",
    pad=48,
)
ax.grid(axis="y", linewidth=0.5, alpha=0.22)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

legend_handles = []
legend_labels = []

for seed in SEEDS:
    legend_handles.append(
        plt.Line2D([0], [0], color=seed_colors[seed], linewidth=1.7, alpha=1.0)
    )
    legend_labels.append(str(seed))

ax.legend(
    legend_handles,
    legend_labels,
    title="Seed",
    frameon=False,
    fontsize=7.5,
    title_fontsize=8,
    loc="upper right",
)


In [ ]:
ZOOM_EPOCH_START = 5
ZOOM_EPOCH_END = 30
ZOOM_PADDING_FRACTION = 0.12
ZOOM_MIN_PADDING = 0.00015
INSET_LINEWIDTH_SCALE = 0.88

zoom_df = history_df.loc[
    history_df["epoch"].between(ZOOM_EPOCH_START, ZOOM_EPOCH_END)
].copy()

zoom_ymin = float(zoom_df["training_loss"].min())
zoom_ymax = float(zoom_df["training_loss"].max())
zoom_ypad = max(
    (zoom_ymax - zoom_ymin) * ZOOM_PADDING_FRACTION,
    ZOOM_MIN_PADDING,
)

zoom_rectangle = plt.Rectangle(
    (ZOOM_EPOCH_START, zoom_ymin - zoom_ypad),
    ZOOM_EPOCH_END - ZOOM_EPOCH_START,
    (zoom_ymax + zoom_ypad) - (zoom_ymin - zoom_ypad),
    fill=False,
    edgecolor="0.60",
    linewidth=0.8,
    linestyle="--",
    alpha=0.65,
    zorder=0,
)
ax.add_patch(zoom_rectangle)

axins = inset_axes(
    ax,
    width="45%",
    height="38%",
    loc="center right",
    bbox_to_anchor=(0.00, 0.02, 0.98, 0.98),
    bbox_transform=ax.transAxes,
    borderpad=1.1,
)

for seed in plot_order_top_to_bottom:
    seed = int(seed)
    sub = history_df.loc[history_df["seed"] == seed].sort_values("epoch")
    axins.plot(
        sub["epoch"],
        sub["training_loss"],
        color=seed_colors[seed],
        linewidth=linewidth_map[seed] * INSET_LINEWIDTH_SCALE,
        alpha=alpha_map[seed],
        zorder=zorder_map[seed],
    )

axins.set_xlim(ZOOM_EPOCH_START, ZOOM_EPOCH_END)
axins.set_ylim(zoom_ymin - zoom_ypad, zoom_ymax + zoom_ypad)
axins.set_xticks([5, 10, 15, 20, 25, 30])
axins.tick_params(axis="both", labelsize=6)
axins.set_title("Late-epoch zoom", fontsize=7, pad=2)
axins.grid(axis="y", linewidth=0.4, alpha=0.20)

for spine in axins.spines.values():
    spine.set_linewidth(0.8)
    spine.set_color("0.40")


In [ ]:
history_df.to_csv(
    OUT_DIR / "Fig6B_MAE_training_convergence_source_data.csv",
    index=False,
)

seed_stats_df.to_csv(
    OUT_DIR / "Fig6B_MAE_training_convergence_statistics.csv",
    index=False,
)

curve_position_df.to_csv(
    OUT_DIR / "Fig6B_MAE_training_convergence_curve_visibility.csv",
    index=False,
)

fig.subplots_adjust(top=0.72)

fig.savefig(
    OUT_DIR / "Fig6B_MAE_training_convergence.pdf",
    bbox_inches="tight",
)

fig.savefig(
    OUT_DIR / "Fig6B_MAE_training_convergence.png",
    dpi=600,
    bbox_inches="tight",
)

plt.show()


# 5. Fig. 6C clinical anchoring ridge plots

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path

from scipy.stats import gaussian_kde, mannwhitneyu
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import scanpy as sc

WORKDIR = resolve_analysis_path(os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai")))

CELL_STATE_PATH = (
    WORKDIR
    / "Fig6_FREEZE_20260811"
    / "tables"
    / "Fig6_cell_level_state.csv.gz"
)

# backed="r" below avoids loading the large expression matrix
ADATA_PATH = WORKDIR / "adata_mono_MAE_HVG5000.h5ad"

OUTDIR = WORKDIR / "Fig6_risk_localization" / "Fig6C_ridge"
OUTDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# plotting parameters

# Number of x coordinates used to draw each smooth density
N_GRID = 500

# Moderate fixed bandwidth for display smoothing
# This affects visualization only, not statistics
KDE_BW = 0.30

# Only used to define the displayed x-range
# No observations are removed from density estimation or testing
XLIM_QUANTILES = (0.005, 0.995)

# Common vertical scale for ridges within each disease panel
RIDGE_HEIGHT = 1.00

REFERENCE_ALPHA = 0.60
RISK_ALPHA = 0.72

# Colors
DISEASE_COLORS = {
    "CAR-T_CRS": "#4C78A8",
    "COVID19": "#F58518",
    "SLE": "#54A24B",
}

REFERENCE_COLOR = "#BDBDBD"

DISEASE_DISPLAY = {
    "CAR-T_CRS": "CAR-T CRS",
    "COVID19": "COVID-19",
    "SLE": "SLE",
}

FIGSIZE = (4.0, 2.2)

In [ ]:
# Load frozen risk scores and exact metadata
state = pd.read_csv(
    CELL_STATE_PATH,
    compression="gzip",
)

adata_b = sc.read_h5ad(
    ADATA_PATH,
    backed="r",
)

print("Cell-state rows:", len(state))
print("AnnData cells:", adata_b.n_obs)

if len(state) != adata_b.n_obs:
    raise ValueError(
        "Cell-state table and AnnData do not have the same number "
        "of rows. Do not assume row-wise alignment."
    )

obs = adata_b.obs.copy()
adata_b.file.close()

required_obs = [
    "disease",
    "patient_id",
    "sample_id",
    "severity",
    "sampling_time",
]

missing = [x for x in required_obs if x not in obs.columns]

if missing:
    raise ValueError(
        f"Required AnnData metadata missing: {missing}"
    )


In [ ]:
# Combine frozen risk scores with exact AnnData metadata
df = pd.DataFrame({
    "cell_id": state["cell_id"].values,

    "risk_score":
        pd.to_numeric(
            state["Fig6_primary_risk_score"],
            errors="coerce"
        ).values,

    "disease":
        obs["disease"].astype(str).values,

    "patient_id":
        obs["patient_id"].astype(str).values,

    "sample_id":
        obs["sample_id"].astype(str).values,

    "severity":
        obs["severity"].astype(str).values,

    "sampling_time":
        obs["sampling_time"].astype(str).values,
})

expected_diseases = {"CAR-T_CRS", "COVID19", "SLE"}
unexpected_diseases = sorted(set(df["disease"].dropna()) - expected_diseases)

if unexpected_diseases:
    raise ValueError(
        f"Unexpected disease labels in AnnData metadata: {unexpected_diseases}"
    )


In [ ]:
# Six ridge populations
df["anchor_group"] = pd.NA
df["anchor_component"] = pd.NA


# CAR-T CRS

mask = (
    (df["disease"] == "CAR-T_CRS")
    & (df["severity"] == "Severe")
    & (df["sampling_time"] == "CAR-T_CRS_pro")
)

df.loc[mask, "anchor_group"] = "Risk"
df.loc[mask, "anchor_component"] = "Severe progression"


mask = (
    (df["disease"] == "CAR-T_CRS")
    & (df["severity"] == "Severe")
    & (df["sampling_time"] == "CAR-T_CRS_before")
)

df.loc[mask, "anchor_group"] = "Reference"
df.loc[mask, "anchor_component"] = "Severe before"


mask = (
    (df["disease"] == "CAR-T_CRS")
    & (df["severity"] == "Severe")
    & (df["sampling_time"] == "CAR-T_CRS_con")
)

df.loc[mask, "anchor_group"] = "Reference"
df.loc[mask, "anchor_component"] = "Severe convalescence"


# COVID-19

mask = (
    (df["disease"] == "COVID19")
    & (df["severity"] == "Severe")
    & (df["sampling_time"] == "COVID19_pro")
)

df.loc[mask, "anchor_group"] = "Risk"
df.loc[mask, "anchor_component"] = "Severe progression"


mask = (
    (df["disease"] == "COVID19")
    & (df["severity"] == "Moderate")
    & (df["sampling_time"] == "COVID19_pro")
)

df.loc[mask, "anchor_group"] = "Reference"
df.loc[mask, "anchor_component"] = "Moderate progression"


mask = (
    (df["disease"] == "COVID19")
    & (df["severity"] == "Severe")
    & (df["sampling_time"] == "COVID19_con")
)

df.loc[mask, "anchor_group"] = "Reference"
df.loc[mask, "anchor_component"] = "Severe convalescence"


# SLE

mask = (
    (df["disease"] == "SLE")
    & (df["severity"] == "Severe")
)

df.loc[mask, "anchor_group"] = "Risk"
df.loc[mask, "anchor_component"] = "Severe"


mask = (
    (df["disease"] == "SLE")
    & (df["severity"] == "Moderate")
)

df.loc[mask, "anchor_group"] = "Reference"
df.loc[mask, "anchor_component"] = "Moderate"


df = df.loc[
    df["anchor_group"].notna()
    & df["risk_score"].notna()
].copy()


In [ ]:
def plot_disease_ridge(
    disease,
    data,
):

    disease_df = data.loc[
        data["disease"] == disease
    ].copy()

    if disease_df.empty:
        raise ValueError(
            f"No cells found for {disease}"
        )

    # Disease-specific x-axis display range
    x_min, x_max = np.quantile(
        disease_df["risk_score"],
        XLIM_QUANTILES,
    )

    padding = 0.06 * (
        x_max - x_min
    )

    x_grid = np.linspace(
        x_min - padding,
        x_max + padding,
        N_GRID,
    )
    # Separate risk and reference cells
    risk_df = disease_df.loc[
        disease_df["anchor_group"] == "Risk"
    ]

    ref_df = disease_df.loc[
        disease_df["anchor_group"] == "Reference"
    ]

    # Patient-balanced densities
    risk_density = patient_balanced_density(
        risk_df,
        x_grid,
        bw_method=KDE_BW,
    )

    ref_density = patient_balanced_density(
        ref_df,
        x_grid,
        bw_method=KDE_BW,
    )

    # Scale both densities to a common within-panel maximum
    max_density = max(
        risk_density.max(),
        ref_density.max(),
    )

    risk_density = (
        risk_density
        / max_density
        * RIDGE_HEIGHT
    )

    ref_density = (
        ref_density
        / max_density
        * RIDGE_HEIGHT
    )


    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    # Reference ridge (draw first)
    ax.fill_between(
        x_grid,
        0,
        ref_density,
        color=REFERENCE_COLOR,
        alpha=REFERENCE_ALPHA,
        linewidth=0,
        zorder=1,
    )

    ax.plot(
        x_grid,
        ref_density,
        color=REFERENCE_COLOR,
        linewidth=1.1,
        zorder=2,
    )

    # Risk ridge (draw on top)
    disease_color = DISEASE_COLORS[disease]

    ax.fill_between(
        x_grid,
        0,
        risk_density,
        color=disease_color,
        alpha=RISK_ALPHA,
        linewidth=0,
        zorder=3,
    )

    ax.plot(
        x_grid,
        risk_density,
        color=disease_color,
        linewidth=1.2,
        zorder=4,
    )

    # Minimal formatting
    # Remove all text except the x-axis score annotation.
    ax.set_xlabel(
        "Primary risk score",
        fontsize=10.5,
    )

    ax.set_ylabel("")

    ax.set_yticks([])

    ax.set_title("")


    ax.set_xlim(
        x_grid.min(),
        x_grid.max(),
    )

    ax.set_ylim(
        0,
        RIDGE_HEIGHT * 1.05,
    )


    # Keep only bottom axis for later Illustrator assembly.
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    ax.tick_params(
        axis="y",
        length=0,
        labelleft=False,
    )

    ax.tick_params(
        axis="x",
        labelsize=9,
    )


    plt.tight_layout()

    # Save
    safe_name = disease.replace(
        "-",
        ""
    )

    out_pdf = (
        OUTDIR
        / f"Fig6C_ridge_{safe_name}.pdf"
    )

    out_png = (
        OUTDIR
        / f"Fig6C_ridge_{safe_name}.png"
    )


    fig.savefig(
        out_pdf,
        bbox_inches="tight",
    )

    fig.savefig(
        out_png,
        dpi=600,
        bbox_inches="tight",
    )

    plt.show()


for disease in [
    "CAR-T_CRS",
    "COVID19",
    "SLE",
]:
    plot_disease_ridge(
        disease,
        df,
    )

# 6. Fig. 6D CS-score concordance

In [ ]:
# ============================================================
# Fig. 6D
# Patient-level concordance between learned monocyte
# risk burden and cytokine-storm transcriptional activity
#
# Visualization:
#   within-disease min-max scaling to [0, 1]
#
# Statistics:
#   Spearman correlation on ORIGINAL unscaled values
#   + BH correction across the three diseases
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

In [ ]:
import os

analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path

WORKDIR = resolve_analysis_path(os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai")))

INPUT_PATH = (
    WORKDIR
    / "Fig6_CS_concordance"
    / "Fig6_CS_concordance_patient_level.csv"
)

OUTDIR = (
    WORKDIR
    / "Fig6_CS_concordance"
    / "Fig6D_plot"
)

OUTDIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUT_STATS = (
    OUTDIR
    / "Fig6D_patient_level_correlation_statistics.csv"
)

OUT_PLOTDATA = (
    OUTDIR
    / "Fig6D_patient_level_scaled_plotdata.csv"
)

In [ ]:
# Plotting parameters
DISEASE_COLORS = {
    "CAR-T_CRS": "#4C78A8",
    "COVID19": "#F58518",
    "SLE": "#54A24B",
}

DISEASE_ORDER = [
    "CAR-T_CRS",
    "COVID19",
    "SLE",
]

FIGSIZE = (3.2, 3.2)

POINT_SIZE = 24
POINT_ALPHA = 0.48

LINE_WIDTH = 2.2

AXIS_MIN = 0.0
AXIS_MAX = 1.0

AXIS_TICKS = [
    0.0,
    0.2,
    0.4,
    0.6,
    0.8,
    1.0,
]


In [ ]:
# Load existing patient-level concordance table
patient_df = pd.read_csv(
    INPUT_PATH
)

print("Table shape:", patient_df.shape)
display(patient_df.head())


In [ ]:
DISEASE_COL = "disease"
PATIENT_COL = "patient_id"

# Primary concordance analysis:
# patient-level mean learned risk burden
# versus patient-level mean CS score
RISK_COL = "risk_burden_mean"
CS_COL = "CS_score_mean"


required_cols = [
    DISEASE_COL,
    PATIENT_COL,
    RISK_COL,
    CS_COL,
]

missing_cols = [
    col for col in required_cols
    if col not in patient_df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

print("Using primary concordance variables:")
print("Risk burden:", RISK_COL)
print("CS score:", CS_COL)

In [ ]:
# Use exact disease labels from the frozen concordance table
plot_df = patient_df[
    [
        DISEASE_COL,
        PATIENT_COL,
        RISK_COL,
        CS_COL,
    ]
].copy()

plot_df.columns = [
    "disease",
    "patient_id",
    "risk_burden_raw",
    "CS_score_raw",
]

plot_df["disease"] = plot_df["disease"].astype(str)

unexpected_diseases = sorted(
    set(plot_df["disease"].dropna())
    - set(DISEASE_ORDER)
)

if unexpected_diseases:
    raise ValueError(
        f"Unexpected disease labels in concordance table: {unexpected_diseases}"
    )

plot_df["risk_burden_raw"] = pd.to_numeric(
    plot_df["risk_burden_raw"],
    errors="coerce",
)

plot_df["CS_score_raw"] = pd.to_numeric(
    plot_df["CS_score_raw"],
    errors="coerce",
)

plot_df = plot_df.loc[
    plot_df["disease"].isin(DISEASE_ORDER)
    & plot_df["risk_burden_raw"].notna()
    & plot_df["CS_score_raw"].notna()
].copy()

print("\nPatients included:")
print(
    plot_df.groupby("disease")["patient_id"].nunique()
)


In [ ]:
plot_df["risk_burden_scaled"] = np.nan
plot_df["CS_score_scaled"] = np.nan


for disease in DISEASE_ORDER:

    idx = (
        plot_df["disease"] == disease
    )

    plot_df.loc[
        idx,
        "risk_burden_scaled"
    ] = minmax_scale(
        plot_df.loc[
            idx,
            "risk_burden_raw"
        ]
    )


    plot_df.loc[
        idx,
        "CS_score_scaled"
    ] = minmax_scale(
        plot_df.loc[
            idx,
            "CS_score_raw"
        ]
    )


plot_df.to_csv(
    OUT_PLOTDATA,
    index=False,
)


In [ ]:
# Calculate statistics on the original values
# Spearman statistics on untransformed values
stats_results = []

for disease in DISEASE_ORDER:

    sub = plot_df.loc[
        plot_df["disease"] == disease
    ].copy()


    rho, p_value = spearmanr(
        sub["risk_burden_raw"],
        sub["CS_score_raw"],
    )


    stats_results.append({
        "disease": disease,

        "n_patients":
            sub["patient_id"].nunique(),

        "spearman_rho":
            rho,

        "p_value_raw":
            p_value,
    })

stats_df = pd.DataFrame(
    stats_results
)

# BH correction across the THREE disease-level correlations
stats_df["p_adj_BH"] = multipletests(
    stats_df["p_value_raw"].values,
    method="fdr_bh",
)[1]


def significance_label(p):

    if p < 0.0001:
        return "****"

    if p < 0.001:
        return "***"

    if p < 0.01:
        return "**"

    if p < 0.05:
        return "*"

    return "ns"


stats_df["BH_significance"] = (
    stats_df["p_adj_BH"]
    .apply(significance_label)
)


stats_df.to_csv(
    OUT_STATS,
    index=False,
)


print("\nCorrelation statistics:")
display(stats_df)

In [ ]:
# Plot
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

for disease in DISEASE_ORDER:

    sub = (
        plot_df.loc[
            plot_df["disease"] == disease
        ]
        .copy()
    )


    x = (
        sub["risk_burden_scaled"]
        .to_numpy(dtype=float)
    )

    y = (
        sub["CS_score_scaled"]
        .to_numpy(dtype=float)
    )

    slope, intercept = np.polyfit(
        x,
        y,
        deg=1,
    )

    x_line = np.linspace(
        AXIS_MIN,
        AXIS_MAX,
        200,
    )

    y_line = (
        slope * x_line
        + intercept
    )


    color = DISEASE_COLORS[
        disease
    ]


    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )


    # Patient-level points
    ax.scatter(
        x,
        y,

        s=POINT_SIZE,
        color=color,
        alpha=POINT_ALPHA,

        edgecolors="none",

        zorder=2,
    )


    # Visualization-only regression line
    ax.plot(
        x_line,
        y_line,

        color=color,
        linewidth=LINE_WIDTH,

        zorder=3,
    )

    # Identical axes across diseases
    ax.set_xlim(
        AXIS_MIN,
        AXIS_MAX,
    )

    ax.set_ylim(
        AXIS_MIN,
        AXIS_MAX,
    )


    ax.set_xticks(
        AXIS_TICKS
    )

    ax.set_yticks(
        AXIS_TICKS
    )


    ax.set_xlabel(
        "Learned risk burden (scaled)",
        fontsize=10,
    )

    ax.set_ylabel(
        "CS score (scaled)",
        fontsize=10,
    )
    ax.set_title("")


    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)

    ax.tick_params(
        axis="both",
        labelsize=8.5,
        width=0.8,
        length=3.5,
    )

    ax.set_aspect(
        "equal",
        adjustable="box",
    )

    plt.tight_layout()


    safe_name = (
        disease
        .replace("-", "")
    )

    out_pdf = (
        OUTDIR
        / f"Fig6D_correlation_{safe_name}.pdf"
    )

    out_png = (
        OUTDIR
        / f"Fig6D_correlation_{safe_name}.png"
    )

    fig.savefig(
        out_pdf,
        bbox_inches="tight",
    )

    fig.savefig(
        out_png,
        dpi=600,
        bbox_inches="tight",
    )

    plt.show()

print("\nSaved statistics:")
print(OUT_STATS)

print("\nSaved plotting data:")
print(OUT_PLOTDATA)

# 7. Fig. 6E atlas-state alignment

In [ ]:
# ============================================================
# Fig. 6E
# Post-hoc enrichment of atlas-defined monocyte states
# in the learned risk representation
# ============================================================

import numpy as np
import pandas as pd
import scanpy as sc

from pathlib import Path
import os
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path



MAE_DIR = resolve_analysis_path(os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai")))

H5AD_PATH = (
    MAE_DIR
    / "adata_mono_MAE_HVG5000.h5ad"
)


OUTDIR = (
    MAE_DIR
    / "Fig6_state_risk_alignment"
)

OUTDIR.mkdir(
    parents=True,
    exist_ok=True,
)

ANNOT_COL = "subcluster_anno"
PATIENT_COL = "patient_id"
DISEASE_COL = "disease"

RISK_COL = "Fig6_primary_risk_score"

MIN_STATE_CELLS = 30
MIN_OTHER_CELLS = 100

# Require at least this many informative patients
MIN_PATIENTS = 3

In [ ]:
# Metadata and exact risk-score column

adata = sc.read_h5ad(
    H5AD_PATH,
    backed="r",
)

required_obs = [
    ANNOT_COL,
    PATIENT_COL,
    DISEASE_COL,
    RISK_COL,
]

missing = [
    x for x in required_obs
    if x not in adata.obs.columns
]

if missing:
    raise ValueError(
        f"Missing required obs columns: {missing}"
    )


meta = (
    adata.obs[required_obs]
    .copy()
)

meta["cell_id"] = adata.obs_names.astype(str)


print("n cells:", len(meta))

print("\nAnnotation counts:")
print(
    meta[ANNOT_COL]
    .value_counts(dropna=False)
)

print("\nDisease counts:")
print(
    meta[DISEASE_COL]
    .value_counts(dropna=False)
)


In [ ]:
meta[RISK_COL] = (
    meta[RISK_COL]
    .astype(float)
)

print("Risk score loaded from exact AnnData obs column:", RISK_COL)


In [ ]:
# Use exact disease labels from AnnData metadata
meta["disease"] = meta[DISEASE_COL].astype(str)

expected_diseases = {"CAR-T_CRS", "COVID19", "SLE"}
unexpected_diseases = sorted(set(meta["disease"].dropna()) - expected_diseases)

if unexpected_diseases:
    raise ValueError(
        f"Unexpected disease labels in AnnData metadata: {unexpected_diseases}"
    )

meta = meta.loc[
    meta["disease"].isin(expected_diseases)
    & meta[RISK_COL].notna()
    & meta[ANNOT_COL].notna()
    & meta[PATIENT_COL].notna()
].copy()


In [ ]:
# Exclude previously identified sample-biased subclusters
# from POST-HOC state-alignment analysis only
EXCLUDED_STATES = [
    "Mono_CD14_IFI44",
    "Mono_CD14_CD16",
]

meta_E = meta.loc[
    ~meta[ANNOT_COL].isin(EXCLUDED_STATES)
].copy()

print(
    meta_E[ANNOT_COL]
    .value_counts()
)

In [ ]:
# Disease-wise risk-score standardization
meta_E["risk_z"] = np.nan

for disease, idx in meta_E.groupby(
    "disease"
).groups.items():

    x = (
        meta_E.loc[
            idx,
            RISK_COL,
        ]
        .astype(float)
    )

    sd = x.std(ddof=0)

    if sd == 0:
        raise ValueError(
            f"Zero variance in risk score for {disease}"
        )

    meta_E.loc[
        idx,
        "risk_z"
    ] = (
        (x - x.mean())
        / sd
    )

print(
    meta_E.groupby("disease")["risk_z"]
    .agg(["mean", "std"])
)


In [ ]:
# Patient-level state enrichment
# Effect:
#
# mean risk_z in annotated state
# minus
# mean risk_z in all OTHER monocytes
# from the SAME disease and SAME patient

records = []

for disease, disease_df in meta_E.groupby(
        "disease",
        observed=True,
    ):
        for patient, patient_df in disease_df.groupby(
            PATIENT_COL,
            observed=True,
        ):
            annotations = (
                patient_df[ANNOT_COL]
                .dropna()
                .unique()
            )

            for state in annotations:
                in_state = (
                    patient_df[ANNOT_COL]
                    == state
                )

                n_state = int(in_state.sum())
                n_other = int((~in_state).sum())

                if n_state < MIN_STATE_CELLS:
                    continue

                if n_other < MIN_OTHER_CELLS:
                    continue

                state_mean = (
                    patient_df.loc[
                        in_state,
                        "risk_z",
                    ]
                    .mean()
                )

                other_mean = (
                    patient_df.loc[
                        ~in_state,
                        "risk_z",
                    ]
                    .mean()
                )

                delta = state_mean - other_mean

                records.append(
                    {
                        "disease": disease,
                        "patient_id": patient,
                        "state": state,
                        "n_state_cells": n_state,
                        "n_other_cells": n_other,
                        "state_mean_risk_z": state_mean,
                        "other_mean_risk_z": other_mean,
                        "delta_risk_z": delta,
                    }
                )


patient_state = pd.DataFrame(
    records
)


print(
    "Patient × state comparisons:",
    len(patient_state)
)

display(
    patient_state.head()
)


In [ ]:
# State-level summary
summary_records = []


for (disease, state), df in patient_state.groupby(
    [
        "disease",
        "state",
    ],
    observed=True,
):

    values = (
        df["delta_risk_z"]
        .dropna()
        .to_numpy()
    )

    n_patients = len(values)
    pvalue = np.nan


    if n_patients >= MIN_PATIENTS:

        if np.allclose(
            values,
            0,
        ):
            pvalue = 1.0

        else:
            try:
                pvalue = wilcoxon(
                    values,
                    alternative="two-sided",
                    zero_method="wilcox",
                ).pvalue

            except ValueError:
                pvalue = np.nan

    else:
        pvalue = np.nan


    summary_records.append(
        {
            "disease": disease,
            "state": state,

            "n_patients": n_patients,

            "mean_delta_risk_z": (
                np.mean(values)
                if n_patients > 0
                else np.nan
            ),

            "median_delta_risk_z": (
                np.median(values)
                if n_patients > 0
                else np.nan
            ),

            "fraction_positive": (
                np.mean(values > 0)
                if n_patients > 0
                else np.nan
            ),

            "min_delta": (
                np.min(values)
                if n_patients > 0
                else np.nan
            ),

            "max_delta": (
                np.max(values)
                if n_patients > 0
                else np.nan
            ),

            "pvalue": pvalue,
        }
    )


state_summary = pd.DataFrame(
    summary_records
)

In [ ]:
# BH correction within disease
state_summary["p_adj"] = np.nan


for disease, idx in state_summary.groupby(
    "disease"
).groups.items():

    valid_idx = [
        i
        for i in idx
        if pd.notna(
            state_summary.loc[
                i,
                "pvalue",
            ]
        )
    ]


    if len(valid_idx) == 0:
        continue


    pvals = (
        state_summary.loc[
            valid_idx,
            "pvalue",
        ]
        .to_numpy()
    )


    state_summary.loc[
        valid_idx,
        "p_adj",
    ] = multipletests(
        pvals,
        method="fdr_bh",
    )[1]

In [ ]:
state_summary = (
    state_summary
    .sort_values(
        [
            "disease",
            "mean_delta_risk_z",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


display(
    state_summary
)

In [ ]:
# Fig. 6E: Bubble plot of state-level risk alignment

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# input

INFILE = state_summary

OUTDIR = Path(
    "Fig6_state_risk_alignment"
)

OUT_PDF = OUTDIR / "Fig6E_state_risk_alignment_bubble.pdf"
OUT_PNG = OUTDIR / "Fig6E_state_risk_alignment_bubble.png"

# Keep the 3 canonical states only, with fixed order

df = INFILE.copy()

state_order = [
    "Mono_CD14_IL1B",
    "Mono_CD14_S100A8",
    "Mono_CD16_LST1",
]

disease_order = [
    "CAR-T_CRS",
    "COVID19",
    "SLE",
]

df = df.loc[
    df["state"].isin(state_order)
    & df["disease"].isin(disease_order)
].copy()

state_label_map = {
    "Mono_CD14_IL1B": "Inflammatory CD14\n(IL1B)",
    "Mono_CD14_S100A8": "Emergency CD14\n(S100A8)",
    "Mono_CD16_LST1": "CD16 sensing\n(LST1)",
}

disease_label_map = {
    "CAR-T_CRS": "CAR-T CRS",
    "COVID19": "COVID-19",
    "SLE": "SLE",
}

df["state_label"] = df["state"].map(state_label_map)
df["disease_label"] = df["disease"].map(disease_label_map)

# Significance handling
df["pvalue_plot"] = df["pvalue"].fillna(1.0)

df["neglog10_pvalue"] = -np.log10(
    df["pvalue_plot"].clip(lower=1e-300)
)

# Cap size encoding at p = 1e-4
SIG_CAP = 4.0

df["neglog10_pvalue_capped"] = (
    df["neglog10_pvalue"]
    .clip(upper=SIG_CAP)
)

SIZE_MIN = 80
SIZE_MAX = 1050


def significance_to_size(p):
    """
    Convert p-value to bubble area.

    p >= 1       -> SIZE_MIN
    p <= 1e-4    -> SIZE_MAX
    intermediate values scale with -log10(p).
    """
    score = -np.log10(max(p, 1e-300))
    score = np.clip(score, 0, SIG_CAP)

    return (
        SIZE_MIN
        + (score / SIG_CAP)
        * (SIZE_MAX - SIZE_MIN)
    )


df["bubble_size"] = (
    df["pvalue_plot"]
    .apply(significance_to_size)
)

def p_to_star(p):
    if pd.isna(p):
        return ""
    if p < 1e-4:
        return "****"
    elif p < 1e-3:
        return "***"
    elif p < 1e-2:
        return "**"
    elif p < 5e-2:
        return "*"
    else:
        return ""

df["sig_star"] = df["pvalue_plot"].apply(p_to_star)

# Coordinates
state_to_y = {state: i for i, state in enumerate(state_order[::-1])}

# reverse so first state is top row
disease_to_x = {disease: i for i, disease in enumerate(disease_order)}

df["x"] = df["disease"].map(disease_to_x)
df["y"] = df["state"].map(state_to_y)

# Plot
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

fig, ax = plt.subplots(figsize=(5.6, 4.2))

# symmetric color range around 0
vmax = np.max(np.abs(df["mean_delta_risk_z"]))
vmax = max(vmax, 0.30)   # ensure a stable lower bound for visual contrast

sc = ax.scatter(
    df["x"],
    df["y"],
    s=df["bubble_size"],
    c=df["mean_delta_risk_z"],
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
    edgecolor="black",
    linewidth=0.8,
    zorder=3,
)

# light grid to create matrix feeling
for xi in [-0.5, 0.5, 1.5, 2.5]:
    ax.axvline(xi, color="0.88", lw=1, zorder=0)
for yi in [-0.5, 0.5, 1.5, 2.5]:
    ax.axhline(yi, color="0.88", lw=1, zorder=0)

# significance stars
for _, row in df.iterrows():
    if row["sig_star"] != "":
        ax.text(
            row["x"],
            row["y"],
            row["sig_star"],
            ha="center",
            va="center",
            fontsize=10,
            fontweight="bold",
            color="black",
            zorder=4,
        )

# Bubble-size legend for adjusted p-values
legend_pvalues = [
    0.05,
    0.01,
    0.001,
    0.0001,
]

legend_handles = []

for p in legend_pvalues:

    size = significance_to_size(p)

    handle = ax.scatter(
        [],
        [],
        s=size,
        facecolor="white",
        edgecolor="0.25",
        linewidth=0.8,
        label=(
            r"$P_{adj}=0.05$" if p == 0.05 else
            r"$P_{adj}=0.01$" if p == 0.01 else
            r"$P_{adj}=0.001$" if p == 0.001 else
            r"$P_{adj}\leq10^{-4}$"
        ),
    )

    legend_handles.append(handle)


size_legend = ax.legend(
    handles=legend_handles,
    title="BH-adjusted P",
    frameon=False,

    # Put to the right of the matrix.
    # Adjust manually depending on your colorbar position.
    bbox_to_anchor=(1.32, 0.72),
    loc="center left",

    labelspacing=1.25,
    handletextpad=1.0,
    borderaxespad=0,

    fontsize=9,
    title_fontsize=9.5,
)

ax.add_artist(size_legend)

# axes
ax.set_xticks([disease_to_x[d] for d in disease_order])
ax.set_xticklabels([disease_label_map[d] for d in disease_order], fontsize=10)

ax.set_yticks([state_to_y[s] for s in state_order[::-1]])
ax.set_yticklabels([state_label_map[s] for s in state_order[::-1]], fontsize=10)

ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.5, 2.5)

ax.set_xlabel("")
ax.set_ylabel("")

# remove tick marks
ax.tick_params(length=0)

# clean border
for spine in ax.spines.values():
    spine.set_visible(False)

# colorbar
cbar = plt.colorbar(sc, ax=ax, fraction=0.05, pad=0.04)
cbar.set_label("Mean Δ risk score (state vs other monocytes)", fontsize=10)

plt.tight_layout()

fig.savefig(OUT_PDF, bbox_inches="tight")
fig.savefig(OUT_PNG, dpi=600, bbox_inches="tight")

plt.show()

print("Saved:")
print(OUT_PDF)
print(OUT_PNG)

# 8. Integrated Gradients gene attribution

In [ ]:
# restore the frozen model architecture
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import gc
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_DIR = Path("Fig6_MAE_models")
ATTR_DIR = Path("Fig6_gene_attribution")
ATTR_DIR.mkdir(exist_ok=True)

GENE_NAMES = adata_m.var_names.to_numpy()

LOCALIZATION_DISEASES = [
    CART,
    COVID,
    SLE
]

INPUT_DIM = adata_m.n_vars
LATENT_DIM = 64

In [ ]:
# Attribution parameters
ATTR_CELLS_PER_PATIENT = 64
ATTR_BATCH_SIZE = 32
IG_STEPS = 32
ATTR_RANDOM_SEED = 20260811

In [ ]:
disease_arr = (
    adata_m.obs[DISEASE_COL]
    .astype(str)
    .to_numpy()
)

severity_arr = (
    adata_m.obs[SEVERITY_COL]
    .astype("string")
    .fillna("NA")
    .astype(str)
    .to_numpy()
)

stage_arr = (
    adata_m.obs[STAGE_COL]
    .astype("string")
    .fillna("NA")
    .astype(str)
    .to_numpy()
)

patient_arr = (
    adata_m.obs[PATIENT_COL]
    .astype(str)
    .to_numpy()
)

In [ ]:
attr_results = []

for disease in LOCALIZATION_DISEASES:

    for seed in FINAL_SEEDS:

        print(
            f"\nAttribution | {disease} | seed {seed}"
        )

        res = attribute_one_seed_one_disease(
            seed=seed,
            disease=disease,
            max_cells_per_patient=ATTR_CELLS_PER_PATIENT,
            batch_size=ATTR_BATCH_SIZE,
            ig_steps=IG_STEPS,
            random_seed=ATTR_RANDOM_SEED
        )

        np.save(
            ATTR_DIR /
            f"Fig6_attr_{disease}_seed_{seed}.npy",
            res["gene_attribution"]
        )

        attr_results.append({
            "disease": disease,
            "seed": seed,
            "n_patients": res["n_patients"],
            "n_cells": res["n_cells"]
        })

attr_results = pd.DataFrame(attr_results)
attr_results

In [ ]:
consensus_attr_tables = {}
consensus_attr_matrix = {}

for disease in LOCALIZATION_DISEASES:

    seed_vectors = []

    for seed in FINAL_SEEDS:

        vec = np.load(
            ATTR_DIR /
            f"Fig6_attr_{disease}_seed_{seed}.npy"
        )

        seed_vectors.append(vec.astype(np.float32))

    seed_matrix = np.vstack(seed_vectors)

    median_attr = np.median(
        seed_matrix,
        axis=0
    )

    q25_attr = np.quantile(
        seed_matrix,
        0.25,
        axis=0
    )

    q75_attr = np.quantile(
        seed_matrix,
        0.75,
        axis=0
    )

    iqr_attr = q75_attr - q25_attr

    positive_seed_fraction = np.mean(
        seed_matrix > 0,
        axis=0
    )

    df = pd.DataFrame({
        "gene": GENE_NAMES,
        "median_attribution": median_attr,
        "attr_iqr": iqr_attr,
        "positive_seed_fraction": positive_seed_fraction
    })

    df = df.sort_values(
        "median_attribution",
        ascending=False
    ).reset_index(drop=True)

    consensus_attr_tables[disease] = df
    consensus_attr_matrix[disease] = seed_matrix

    df.to_csv(
        ATTR_DIR /
        f"Fig6_attr_consensus_{disease}.csv",
        index=False
    )

In [ ]:
from scipy.stats import spearmanr
from itertools import combinations

attr_stability_records = []

for disease in LOCALIZATION_DISEASES:

    seed_matrix = consensus_attr_matrix[disease]

    for i, j in combinations(range(len(FINAL_SEEDS)), 2):

        rho, _ = spearmanr(
            seed_matrix[i, :],
            seed_matrix[j, :]
        )

        attr_stability_records.append({
            "disease": disease,
            "seed_a": FINAL_SEEDS[i],
            "seed_b": FINAL_SEEDS[j],
            "spearman_r": rho
        })

attr_stability_df = pd.DataFrame(attr_stability_records)

attr_stability_df

In [ ]:
attr_stability_df.to_csv(
    ATTR_DIR / "Fig6_attr_seed_stability.csv",
    index=False
)

In [ ]:
# Build complete gene-attribution tables

ATTR_TABLE_DIR = ATTR_DIR / "tables"
ATTR_TABLE_DIR.mkdir(exist_ok=True)


full_attr_tables = {}


for disease in LOCALIZATION_DISEASES:

    # Shape:
    # 5 seeds × 5000 genes

    seed_matrix = (
        consensus_attr_matrix[
            disease
        ]
    )

    assert seed_matrix.shape == (
        len(FINAL_SEEDS),
        len(GENE_NAMES)
    )

    # Start with gene names
    df = pd.DataFrame({
        "gene": GENE_NAMES
    })

    # Store EVERY seed explicitly
    for i, seed in enumerate(
        FINAL_SEEDS
    ):

        df[
            f"attr_seed_{seed}"
        ] = seed_matrix[i, :]

    # Consensus statistics across seeds
    df["median_attribution"] = np.median(
        seed_matrix,
        axis=0
    )

    df["mean_attribution"] = np.mean(
        seed_matrix,
        axis=0
    )

    df["attr_q25"] = np.quantile(
        seed_matrix,
        0.25,
        axis=0
    )

    df["attr_q75"] = np.quantile(
        seed_matrix,
        0.75,
        axis=0
    )

    df["attr_iqr"] = (
        df["attr_q75"]
        - df["attr_q25"]
    )

    df["attr_min"] = np.min(
        seed_matrix,
        axis=0
    )

    df["attr_max"] = np.max(
        seed_matrix,
        axis=0
    )

    # Directional stability
    df["positive_seed_fraction"] = np.mean(
        seed_matrix > 0,
        axis=0
    )

    df["negative_seed_fraction"] = np.mean(
        seed_matrix < 0,
        axis=0
    )

    # Absolute magnitude
    df["abs_median_attribution"] = np.abs(
        df["median_attribution"]
    )

    # Human-readable direction
    df["direction"] = np.where(
        df["median_attribution"] > 0,
        "positive",
        np.where(
            df["median_attribution"] < 0,
            "negative",
            "neutral"
        )
    )

    # Ranks
    # rank_positive:
    #   strongest positive driver = 1
    #
    # rank_negative:
    #   strongest negative driver = 1
    #
    # rank_absolute:
    #   strongest attribution regardless of sign = 1
    df["rank_positive"] = (
        df["median_attribution"]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

    df["rank_negative"] = (
        df["median_attribution"]
        .rank(
            ascending=True,
            method="min"
        )
        .astype(int)
    )

    df["rank_absolute"] = (
        df["abs_median_attribution"]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

    # Sort primarily by signed attribution
    df = df.sort_values(
        "median_attribution",
        ascending=False
    ).reset_index(drop=True)


    full_attr_tables[
        disease
    ] = df

    # Save complete disease-specific table
    df.to_csv(
        ATTR_TABLE_DIR /
        f"Fig6_gene_attribution_FULL_{disease}.csv",
        index=False
    )

In [ ]:
# Save top/bottom driver tables
N_TOP_EXPORT = 100


for disease in LOCALIZATION_DISEASES:

    df = full_attr_tables[
        disease
    ]


    # Strongest positive drivers
    top_positive = (
        df
        .sort_values(
            "median_attribution",
            ascending=False
        )
        .head(N_TOP_EXPORT)
        .copy()
    )


    # Strongest negative drivers
    top_negative = (
        df
        .sort_values(
            "median_attribution",
            ascending=True
        )
        .head(N_TOP_EXPORT)
        .copy()
    )


    top_positive.to_csv(
        ATTR_TABLE_DIR /
        f"Fig6_gene_attribution_TOP_POSITIVE_{disease}.csv",
        index=False
    )


    top_negative.to_csv(
        ATTR_TABLE_DIR /
        f"Fig6_gene_attribution_TOP_NEGATIVE_{disease}.csv",
        index=False
    )

In [ ]:
# Combined cross-disease consensus table

combined_attr = []


for disease in LOCALIZATION_DISEASES:

    tmp = full_attr_tables[
        disease
    ].copy()

    tmp.insert(
        0,
        "disease",
        disease
    )

    combined_attr.append(
        tmp
    )


combined_attr = pd.concat(
    combined_attr,
    axis=0,
    ignore_index=True
)


combined_attr.to_csv(
    ATTR_TABLE_DIR /
    "Fig6_gene_attribution_ALL_DISEASES.csv",
    index=False
)


print(
    combined_attr.shape
)

In [ ]:
# Seed-level long-format attribution table

long_records = []


for disease in LOCALIZATION_DISEASES:

    seed_matrix = (
        consensus_attr_matrix[
            disease
        ]
    )


    for i, seed in enumerate(
        FINAL_SEEDS
    ):

        tmp = pd.DataFrame({
            "disease": disease,
            "seed": seed,
            "gene": GENE_NAMES,
            "attribution": seed_matrix[i, :]
        })

        long_records.append(
            tmp
        )


attr_seed_long = pd.concat(
    long_records,
    axis=0,
    ignore_index=True
)


attr_seed_long.to_csv(
    ATTR_TABLE_DIR /
    "Fig6_gene_attribution_SEED_LEVEL_LONG.csv",
    index=False
)


print(
    attr_seed_long.shape
)

In [ ]:
# Gene × disease attribution matrix

attr_wide = (
    combined_attr[
        [
            "gene",
            "disease",
            "median_attribution"
        ]
    ]
    .pivot(
        index="gene",
        columns="disease",
        values="median_attribution"
    )
)


attr_wide.to_csv(
    ATTR_TABLE_DIR /
    "Fig6_gene_attribution_MEDIAN_MATRIX.csv"
)


attr_wide.head()

## Conserved attribution-consensus profile

In [ ]:
from scipy.stats import rankdata

CONSERVED_DIR = Path("Fig6_conserved_profile")
CONSERVED_DIR.mkdir(exist_ok=True)


def signed_absolute_rank_score(attribution):
    """
    Convert one raw gene-attribution vector into a signed within-profile rank score.
    """
    attribution = np.asarray(attribution, dtype=np.float64)
    n_genes = len(attribution)

    abs_rank = rankdata(
        np.abs(attribution),
        method="average",
    )

    magnitude_percentile = (abs_rank - 1) / (n_genes - 1)
    return np.sign(attribution) * magnitude_percentile


In [ ]:
signed_rank_by_seed = {}

for seed_idx, seed in enumerate(FINAL_SEEDS):
    signed_rank_by_seed[seed] = {}

    for disease in LOCALIZATION_DISEASES:
        raw_attr = consensus_attr_matrix[disease][seed_idx, :]
        signed_rank_by_seed[seed][disease] = signed_absolute_rank_score(raw_attr)


conserved_score_by_seed = {}

for seed in FINAL_SEEDS:
    disease_scores = np.stack(
        [
            signed_rank_by_seed[seed][disease]
            for disease in LOCALIZATION_DISEASES
        ],
        axis=0,
    )

    conserved_score = disease_scores.mean(axis=0)
    conserved_score_by_seed[seed] = conserved_score.astype(np.float32)

    np.save(
        CONSERVED_DIR / f"Fig6_conserved_rank_score_seed_{seed}.npy",
        conserved_score,
    )


conserved_score_matrix = np.stack(
    [
        conserved_score_by_seed[seed]
        for seed in FINAL_SEEDS
    ],
    axis=0,
)

print(conserved_score_matrix.shape)


In [ ]:
conserved_profile_df = pd.DataFrame({
    "gene": GENE_NAMES,
    "median_conserved_score": np.median(conserved_score_matrix, axis=0),
    "mean_conserved_score": np.mean(conserved_score_matrix, axis=0),
    "conserved_score_q25": np.quantile(conserved_score_matrix, 0.25, axis=0),
    "conserved_score_q75": np.quantile(conserved_score_matrix, 0.75, axis=0),
    "positive_seed_fraction": np.mean(conserved_score_matrix > 0, axis=0),
})

conserved_profile_df["conserved_score_iqr"] = (
    conserved_profile_df["conserved_score_q75"]
    - conserved_profile_df["conserved_score_q25"]
)

for disease in LOCALIZATION_DISEASES:
    disease_table = (
        full_attr_tables[disease][["gene", "median_attribution"]]
        .copy()
        .rename(columns={"median_attribution": f"{disease}_attribution"})
    )

    conserved_profile_df = conserved_profile_df.merge(
        disease_table,
        on="gene",
        how="left",
    )

disease_attr_cols = [
    f"{disease}_attribution"
    for disease in LOCALIZATION_DISEASES
]

conserved_profile_df["n_diseases_positive"] = (
    conserved_profile_df[disease_attr_cols] > 0
).sum(axis=1)


In [ ]:
disease_rank_cols = []

for disease in LOCALIZATION_DISEASES:
    attr_col = f"{disease}_attribution"
    rank_col = f"{disease}_positive_rank_percentile"

    conserved_profile_df[rank_col] = (
        conserved_profile_df[attr_col]
        .rank(
            ascending=True,
            pct=True,
        )
    )

    disease_rank_cols.append(rank_col)

conserved_profile_df["mean_disease_rank_percentile"] = (
    conserved_profile_df[disease_rank_cols]
    .mean(axis=1)
)

conserved_profile_df["minimum_disease_rank_percentile"] = (
    conserved_profile_df[disease_rank_cols]
    .min(axis=1)
)

conserved_profile_df["rank_conserved"] = (
    conserved_profile_df["median_conserved_score"]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)

conserved_profile_df = (
    conserved_profile_df
    .sort_values("median_conserved_score", ascending=False)
    .reset_index(drop=True)
)


# 9. Fig. 6G reference-state feature reversion titration

## Titration control imports and settings

In [ ]:
# ============================================================
# Purpose:
#   Rebuild all nested matched-control trajectories from
#   scratch for the extended K range
#
# Upstream dependencies:
#   1. adata_m
#   2. full_attr_tables
#   3. conserved_profile_df
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import json

from sklearn.neighbors import NearestNeighbors


# Formal settings

CART = "CAR-T_CRS"
COVID = "COVID19"
SLE = "SLE"

DISEASES = [
    CART,
    COVID,
    SLE
]


# Extended grid:
# COVID's previous fitted K90 (~644) now lies between
# directly observed K=600 and K=700.
K_GRID = [
    5,
    10,
    20,
    30,
    50,
    75,
    100,
    150,
    200,
    300,
    400,
    500,
    600,
    700,
    800,
    900,
    1000
]

K_MAX = max(K_GRID)


# Sampling / computation
TITRATION_CELLS_PER_PATIENT = 64
TITRATION_BATCH_SIZE = 64
PERTURBATION_SET_CHUNK = 8

# Number of matched null trajectories per control family.
N_TITRATION_CONTROLS = 100


# Low-attribution control pool:
# genes in the bottom 25% by absolute attribution magnitude.
LOW_ATTR_QUANTILE = 0.25


# For each target gene, randomly sample from its nearest
# expression/detection-matched available candidates.
MATCH_NEAREST_N = 50


# Precompute this many nearest candidates per target.
# Larger than MATCH_NEAREST_N because candidates are removed
# without replacement as a trajectory grows toward K_MAX.
MATCH_PRECOMPUTE_N = 1000


TITRATION_RANDOM_SEED = 20260812

# Specificity definition
SPECIFICITY_ALPHA = 0.05
MIN_PASSING_SEEDS = 4


RR_DIR = Path(
    "Fig6_reference_reversion_titration"
)

RR_DIR.mkdir(
    exist_ok=True
)


CONTROL_FILE = (
    RR_DIR /
    f"Fig6_titration_control_trajectories_Top{K_MAX}.json"
)


print(
    f"K_MAX = {K_MAX}"
)

print(
    f"Controls per family = {N_TITRATION_CONTROLS}"
)

print(
    f"Output = {CONTROL_FILE}"
)

# Verify only the core upstream objects
required_objects = [
    "adata_m",
    "full_attr_tables",
    "conserved_profile_df"
]


missing_objects = [
    x
    for x in required_objects
    if x not in globals()
]


if missing_objects:

    raise RuntimeError(
        "Missing essential upstream result objects: "
        f"{missing_objects}\n"
        "These are required scientific outputs, not old "
        "titration helper objects."
    )


## Reconstruct molecular profiles

In [ ]:
# Reconstruct the four molecular profiles
FOUR_PROFILES = {}
PROFILE_POSITIVE_LIMITS = {}

# Disease-specific Integrated-Gradients rankings
for disease in DISEASES:

    if disease not in full_attr_tables:

        raise KeyError(
            f"{disease!r} missing from full_attr_tables. "
            f"Available keys: {list(full_attr_tables.keys())}"
        )

    df = (
        full_attr_tables[disease][["gene", "median_attribution"]]
        .copy()
        .rename(columns={"median_attribution": "feature_score"})
    )

    df["gene"] = df["gene"].astype(str)

    df = (
        df
        .sort_values("feature_score", ascending=False)
        .reset_index(drop=True)
    )

    FOUR_PROFILES[disease] = df

# Continuous conserved attribution-consensus ranking
conserved_df = (
    conserved_profile_df[["gene", "median_conserved_score"]]
    .copy()
    .rename(columns={"median_conserved_score": "feature_score"})
)

conserved_df["gene"] = conserved_df["gene"].astype(str)

conserved_df = (
    conserved_df
    .sort_values("feature_score", ascending=False)
    .reset_index(drop=True)
)

FOUR_PROFILES["CONSERVED"] = conserved_df

print("\nFour profiles reconstructed:")

for profile, df in FOUR_PROFILES.items():
    n_positive = int((df["feature_score"] > 0).sum())
    PROFILE_POSITIVE_LIMITS[profile] = n_positive
    effective_max = min(K_MAX, n_positive)

    print(
        f"  {profile}: {len(df)} genes | "
        f"{n_positive} positive | "
        f"effective Top-K maximum = {effective_max}"
    )

    if effective_max <= 0:
        raise RuntimeError(
            f"{profile}: no positive-attribution features are available."
        )


## Build risk-anchor indices

In [ ]:
obs_disease = adata_m.obs["disease"].astype(str)
obs_severity = adata_m.obs["severity"].astype(str)
obs_stage = adata_m.obs["sampling_time"].astype(str)


def get_risk_indices_local(disease):
    """
    Reconstruct the final risk-anchor definitions with exact metadata labels.
    """
    disease_mask = obs_disease == disease
    severe_mask = obs_severity == "Severe"

    if disease == CART:
        mask = disease_mask & severe_mask & (obs_stage == "CAR-T_CRS_pro")
    elif disease == COVID:
        mask = disease_mask & severe_mask & (obs_stage == "COVID19_pro")
    elif disease == SLE:
        mask = disease_mask & severe_mask
    else:
        raise ValueError(disease)

    return np.flatnonzero(mask.to_numpy())


risk_indices_by_disease = {}

print("\nRisk-anchor reconstruction:")

for disease in DISEASES:
    idx = get_risk_indices_local(disease)
    risk_indices_by_disease[disease] = idx

    n_patients = adata_m.obs.iloc[idx]["patient_id"].nunique()

    print(
        f"  {disease}: "
        f"{len(idx):,} cells | "
        f"{n_patients} patients"
    )

    if len(idx) == 0:
        raise RuntimeError(
            f"No risk-anchor cells found for {disease}. "
            "Check exact severity/sampling_time labels."
        )


## Compute gene-matching statistics

In [ ]:
def patient_balanced_gene_stats_local(
    disease
):
    """
    Calculate equal-patient expression and detection
    statistics within disease-specific risk-anchor cells.
    """

    idx = (
        risk_indices_by_disease[
            disease
        ]
    )


    X = adata_m.X[
        idx
    ]


    if sp.issparse(
        X
    ):

        X = X.tocsr()


    patients = (
        adata_m.obs.iloc[
            idx
        ][
            "patient_id"
        ]
        .astype(str)
        .to_numpy()
    )


    unique_patients = np.unique(
        patients
    )


    patient_mean_expr = []

    patient_detection = []


    for patient in unique_patients:

        patient_mask = (
            patients == patient
        )


        Xp = X[
            patient_mask
        ]

        # Mean log1p expression per gene
        mean_expr = np.asarray(
            Xp.mean(
                axis=0
            )
        ).ravel()

        # Detection fraction per gene
        if sp.issparse(
            Xp
        ):

            detected = (
                Xp.copy()
            )

            detected.data = np.ones_like(
                detected.data,
                dtype=np.float32
            )

            detection_rate = (
                np.asarray(
                    detected.mean(
                        axis=0
                    )
                )
                .ravel()
            )

        else:

            detection_rate = (
                Xp > 0
            ).mean(
                axis=0
            )

        patient_mean_expr.append(
            mean_expr
        )

        patient_detection.append(
            detection_rate
        )

    patient_mean_expr = np.vstack(
        patient_mean_expr
    )


    patient_detection = np.vstack(
        patient_detection
    )

    # Equal weighting across patients
    mean_expr = (
        patient_mean_expr.mean(
            axis=0
        )
    )

    detection_rate = (
        patient_detection.mean(
            axis=0
        )
    )

    return pd.DataFrame({

        "gene":
            adata_m.var_names.astype(
                str
            ),

        "mean_expr":
            mean_expr,

        "detection_rate":
            detection_rate
    })

gene_stats_by_disease = {}

print(
    "\nComputing patient-balanced matching statistics..."
)

for disease in DISEASES:

    gene_stats_by_disease[
        disease
    ] = (
        patient_balanced_gene_stats_local(
            disease
        )
    )


    print(
        f"  {disease}: done"
    )


## Build titration tracks

In [ ]:
def build_matching_table_local(
    profile,
    eval_disease
):
    """
    Join:

    1. expression/detection characteristics measured in
       eval_disease risk-anchor cells;

    2. molecular importance from the ranking being tested.
    """

    stats = (
        gene_stats_by_disease[
            eval_disease
        ]
        .copy()
        .set_index(
            "gene"
        )
    )

    scores = (
        FOUR_PROFILES[
            profile
        ][
            [
                "gene",
                "feature_score"
            ]
        ]
        .copy()
        .set_index(
            "gene"
        )
    )

    df = stats.join(
        scores,
        how="inner"
    )


    if len(df) < K_MAX:

        raise RuntimeError(
            f"{profile} -> {eval_disease}: "
            f"only {len(df)} overlapping genes."
        )

    df[
        "abs_feature_score"
    ] = np.abs(
        df[
            "feature_score"
        ]
    )

    # log1p makes mean-expression matching less dominated
    # by a small number of highly expressed genes.
    df[
        "log_mean_expr"
    ] = np.log1p(
        df[
            "mean_expr"
        ]
    )

    # Standardize both matching dimensions so expression
    # and detection contribute on comparable scales.
    for col in [
        "log_mean_expr",
        "detection_rate"
    ]:

        sd = float(
            df[
                col
            ].std()
        )

        if (
            not np.isfinite(sd)
            or sd == 0
        ):

            sd = 1.0

        df[
            f"{col}_z"
        ] = (
            df[
                col
            ]
            - df[
                col
            ].mean()
        ) / sd

    return df

# Define the six tracks

tracks = []


for disease in DISEASES:

    # Disease-specific ranking evaluated in its own disease
    tracks.append({

        "profile":
            disease,

        "eval_disease":
            disease
    })


    # Conserved ranking evaluated in the same disease
    tracks.append({

        "profile":
            "CONSERVED",

        "eval_disease":
            disease
    })

TITRATION_TRACKS = tracks

print(
    "\nSix tracks:"
)

for track in tracks:

    print(
        " ",
        track[
            "profile"
        ],
        "->",
        track[
            "eval_disease"
        ]
    )


## Define matched-trajectory generator

In [ ]:
def generate_matched_trajectories_fast(
    matching_df,
    target_genes,
    candidate_genes,
    n_controls,
    random_seed,
    nearest_n=50,
    precompute_n=1000
):
    """
    Generate nested expression/detection-matched trajectories.

    Each returned trajectory has the same length as the supplied target gene list.

    Therefore, for a track-specific maximum M:
        trajectory[:5]   -> K=5 control
        trajectory[:100] -> K=100 control
        trajectory[:M]   -> largest available control
    """

    target_genes = [
        str(x)
        for x in target_genes
    ]


    candidate_genes = sorted(
        [
            str(x)
            for x in candidate_genes
        ]
    )


    if len(
        candidate_genes
    ) < len(
        target_genes
    ):

        raise RuntimeError(
            "Candidate pool smaller than target trajectory."
        )


    feature_cols = [
        "log_mean_expr_z",
        "detection_rate_z"
    ]


    target_matrix = (
        matching_df.loc[
            target_genes,
            feature_cols
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    candidate_matrix = (
        matching_df.loc[
            candidate_genes,
            feature_cols
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    # Precompute a large nearest-neighbor neighborhood.
    # We need >50 because candidates are progressively used
    # up as a trajectory grows toward the track-specific maximum K
    n_precompute = min(
        len(candidate_genes),
        max(
            precompute_n,
            nearest_n
        )
    )


    nn = NearestNeighbors(
        n_neighbors=n_precompute,
        metric="euclidean",
        algorithm="auto"
    )


    nn.fit(
        candidate_matrix
    )


    _, neighbor_idx = (
        nn.kneighbors(
            target_matrix,
            return_distance=True
        )
    )


    rng = np.random.default_rng(
        random_seed
    )


    trajectories = []


    n_candidates = len(
        candidate_genes
    )


    for control_id in range(
        n_controls
    ):

        used = np.zeros(
            n_candidates,
            dtype=bool
        )


        trajectory = []


        for target_i in range(
            len(target_genes)
        ):

            local_neighbors = (
                neighbor_idx[
                    target_i
                ]
            )


            available_local = (
                local_neighbors[
                    ~used[
                        local_neighbors
                    ]
                ]
            )

            # Prefer the nearest N currently available genes
            if len(
                available_local
            ) > 0:

                pool = (
                    available_local[
                        :nearest_n
                    ]
                )


            else:

                pool = np.flatnonzero(
                    ~used
                )


            if len(
                pool
            ) == 0:

                raise RuntimeError(
                    f"Candidate pool exhausted at "
                    f"target {target_i}."
                )


            chosen_idx = int(
                rng.choice(
                    pool
                )
            )


            used[
                chosen_idx
            ] = True


            trajectory.append(
                candidate_genes[
                    chosen_idx
                ]
            )


        trajectories.append(
            trajectory
        )


    return trajectories


## Generate titration controls

In [ ]:
titration_controls = {}

for track_i, track in enumerate(tracks):
    profile = track["profile"]
    eval_disease = track["eval_disease"]
    track_key = f"{profile}__IN__{eval_disease}"

    print("\n" + "=" * 85)
    print("Preparing:", track_key)
    print("=" * 85)

    matching_df = build_matching_table_local(profile, eval_disease)

    ranked_profile = (
        FOUR_PROFILES[profile]
        .loc[lambda x: x["gene"].isin(matching_df.index)]
        .copy()
    )

    ranked_positive = ranked_profile.loc[
        ranked_profile["feature_score"] > 0
    ].copy()

    track_k_max = min(K_MAX, len(ranked_positive))

    if track_k_max <= 0:
        raise RuntimeError(
            f"{track_key}: no positive-attribution target genes are available."
        )

    target_genes = (
        ranked_positive
        .head(track_k_max)["gene"]
        .astype(str)
        .tolist()
    )

    target_scores = matching_df.loc[target_genes, "feature_score"]

    if not (target_scores > 0).all():
        raise RuntimeError(
            f"{track_key}: target set contains non-positive features after truncation."
        )

    target_gene_set = set(target_genes)

    # Matched random candidate pool. Exclude all true targets for this track.
    random_candidates = [
        str(gene)
        for gene in matching_df.index
        if str(gene) not in target_gene_set
    ]

    # Matched low-attribution candidate pool: bottom 25% by absolute feature score.
    low_cutoff = matching_df["abs_feature_score"].quantile(LOW_ATTR_QUANTILE)

    low_candidates = [
        str(gene)
        for gene in matching_df[
            matching_df["abs_feature_score"] <= low_cutoff
        ].index
        if str(gene) not in target_gene_set
    ]

    print(f"  matching genes: {len(matching_df)}")
    print(f"  requested K_MAX: {K_MAX}")
    print(f"  track-specific K max: {track_k_max}")
    print(f"  target genes: {len(target_genes)}")
    print(f"  random pool: {len(random_candidates)}")
    print(f"  low-attribution pool: {len(low_candidates)}")

    # Required because each trajectory samples without replacement internally.
    if len(random_candidates) < track_k_max:
        raise RuntimeError(
            f"{track_key}: random candidate pool contains only "
            f"{len(random_candidates)} genes; cannot build a Top-{track_k_max} trajectory."
        )

    if len(low_candidates) < track_k_max:
        raise RuntimeError(
            f"{track_key}: low-attribution pool contains only "
            f"{len(low_candidates)} genes; cannot build a Top-{track_k_max} trajectory.\n"
            "Do NOT automatically enlarge LOW_ATTR_QUANTILE."
        )

    random_trajectories = generate_matched_trajectories_fast(
        matching_df=matching_df,
        target_genes=target_genes,
        candidate_genes=random_candidates,
        n_controls=N_TITRATION_CONTROLS,
        random_seed=TITRATION_RANDOM_SEED + track_i * 1000 + 1,
        nearest_n=MATCH_NEAREST_N,
        precompute_n=MATCH_PRECOMPUTE_N,
    )

    print("  random trajectories: done")

    low_trajectories = generate_matched_trajectories_fast(
        matching_df=matching_df,
        target_genes=target_genes,
        candidate_genes=low_candidates,
        n_controls=N_TITRATION_CONTROLS,
        random_seed=TITRATION_RANDOM_SEED + track_i * 1000 + 2,
        nearest_n=MATCH_NEAREST_N,
        precompute_n=MATCH_PRECOMPUTE_N,
    )

    print("  low-attribution trajectories: done")

    titration_controls[track_key] = {
        "profile": profile,
        "eval_disease": eval_disease,
        "requested_K_MAX": K_MAX,
        "track_K_MAX": track_k_max,
        "target_genes": target_genes,
        "random": random_trajectories,
        "low": low_trajectories,
    }


## Save and validate titration controls

In [ ]:
# Save the fresh track-specific Top-K control object

with open(CONTROL_FILE, "w") as f:
    json.dump(titration_controls, f)

# Final self-contained validation
expected_keys = [
    f"{disease}__IN__{disease}"
    for disease in DISEASES
] + [
    f"CONSERVED__IN__{disease}"
    for disease in DISEASES
]

for key in expected_keys:
    if key not in titration_controls:
        raise RuntimeError(f"Missing track: {key}")

    obj = titration_controls[key]
    track_k_max = int(obj["track_K_MAX"])

    assert 0 < track_k_max <= K_MAX
    assert len(obj["target_genes"]) == track_k_max
    assert len(obj["random"]) == N_TITRATION_CONTROLS
    assert len(obj["low"]) == N_TITRATION_CONTROLS
    assert all(len(x) == track_k_max for x in obj["random"])
    assert all(len(x) == track_k_max for x in obj["low"])

print("\n" + "=" * 85)
print(
    f"SUCCESS: all six control tracks were rebuilt with "
    f"track-specific maxima up to Top-{K_MAX}."
)
print(f"Saved to:\n{CONTROL_FILE}")
print("\nThe following objects have also been recreated for compatibility with later cells:")
print("  K_GRID")
print("  K_MAX")
print("  CART / COVID / SLE")
print("  DISEASES")
print("  FOUR_PROFILES")
print("  gene_stats_by_disease")
print("  TITRATION_TRACKS")
print("  titration_controls")
print("=" * 85)


In [ ]:
# frozen-model loader
def load_model_for_seed(
    seed
):

    try:

        model = load_frozen_model(
            int(seed)
        )


    except (
        AttributeError,
        TypeError
    ):

        checkpoint_path = (
            MODEL_DIR /
            f"Fig6_MAE_seed_{int(seed)}.pt"
        )

        model = load_frozen_model(
            checkpoint_path
        )


    return model

## Reference-reversion gene index

In [ ]:
# Evaluate multiple reference-reversion gene sets
GENE_TO_IDX = {

    gene: i

    for i, gene
    in enumerate(
        GENE_NAMES
    )
}


## Define reference-reversion set evaluator

In [ ]:
def evaluate_reference_reversion_sets(
    model,
    seed,
    state,
    cell_idx,
    gene_sets,
    reference_baseline,
    batch_size=64,
    set_chunk_size=8
):
    """
    Evaluate several gene sets under reference-state reversion.

    For selected genes:
        observed risk-cell expression
                    ->
        disease-specific clinical-reference expression

    Unselected genes remain unchanged.

    Returns
    -------
    disease_effects
        Equal-patient mean risk reduction for each gene set.

    patient_effects
        Matrix [gene sets × patients].

    unique_patients
        Patient order corresponding to patient_effects.

    Interpretation
    --------------
    Positive:
        moves cell away from learned risk representation.

    Negative:
        moves cell toward learned risk representation.
    """

    cell_idx = np.asarray(
        cell_idx,
        dtype=int
    )

    # Gene names -> indices
    gene_set_indices = []


    for genes in gene_sets:

        gene_set_indices.append(

            np.asarray(
                [
                    GENE_TO_IDX[g]
                    for g in genes
                ],
                dtype=np.int64
            )
        )


    n_sets = len(
        gene_sets
    )

    # Clean latent risk scores already exist
    clean_z = np.asarray(
        Z_by_seed[
            seed
        ][
            cell_idx
        ],
        dtype=np.float32
    )


    clean_scores = (
        risk_score_from_latent(
            clean_z,
            state
        )
    )

    # Patient indexing
    patients = patient_arr[
        cell_idx
    ]

    unique_patients = np.unique(
        patients
    )


    patient_to_code = {

        patient: i

        for i, patient
        in enumerate(
            unique_patients
        )
    }

    patient_codes = np.asarray(
        [
            patient_to_code[p]
            for p in patients
        ],
        dtype=int
    )


    n_patients = len(
        unique_patients
    )

    reduction_sums = np.zeros(
        (
            n_sets,
            n_patients
        ),
        dtype=np.float64
    )

    patient_counts = np.bincount(
        patient_codes,
        minlength=n_patients
    ).astype(
        np.float64
    )

    model.eval()

    with torch.inference_mode():

        for cell_start in range(
            0,
            len(cell_idx),
            batch_size
        ):

            cell_end = min(
                cell_start
                + batch_size,
                len(cell_idx)
            )


            batch_idx = cell_idx[
                cell_start:
                cell_end
            ]


            x_base = (
                rows_to_dense_float32(
                    adata_m.X,
                    batch_idx
                )
            )


            clean_batch = clean_scores[
                cell_start:
                cell_end
            ]

            codes_batch = patient_codes[
                cell_start:
                cell_end
            ]

            # Process several counterfactual sets per network pass to reduce GPU overhead
            for set_start in range(
                0,
                n_sets,
                set_chunk_size
            ):

                set_end = min(
                    set_start
                    + set_chunk_size,
                    n_sets
                )


                local_indices = (
                    gene_set_indices[
                        set_start:
                        set_end
                    ]
                )


                n_local_sets = len(
                    local_indices
                )


                # [sets, cells, genes]
                x_cf = np.repeat(
                    x_base[
                        None,
                        :,
                        :
                    ],
                    n_local_sets,
                    axis=0
                )

                # REFERENCE-STATE REVERSION
                for local_i, gene_idx in enumerate(
                    local_indices
                ):

                    x_cf[
                        local_i
                    ][
                        :,
                        gene_idx
                    ] = (
                        reference_baseline[
                            gene_idx
                        ][
                            None,
                            :
                        ]
                    )

                # Encode counterfactual cells
                n_batch_cells = (
                    x_base.shape[0]
                )


                x_flat = (
                    x_cf.reshape(
                        n_local_sets
                        * n_batch_cells,
                        adata_m.n_vars
                    )
                )

                x_tensor = (
                    torch.from_numpy(
                        x_flat
                    )
                    .to(
                        DEVICE
                    )
                )

                z_cf = (
                    model
                    .encode(
                        x_tensor
                    )
                    .detach()
                    .cpu()
                    .numpy()
                )

                cf_scores = (
                    risk_score_from_latent(
                        z_cf,
                        state
                    )
                    .reshape(
                        n_local_sets,
                        n_batch_cells
                    )
                )

                reductions = (
                    clean_batch[
                        None,
                        :
                    ]
                    -
                    cf_scores
                )

                # Equal-patient aggregation
                for local_i in range(
                    n_local_sets
                ):

                    global_i = (
                        set_start
                        + local_i
                    )


                    np.add.at(
                        reduction_sums[
                            global_i
                        ],
                        codes_batch,
                        reductions[
                            local_i
                        ]
                    )

    patient_effects = (
        reduction_sums
        /
        patient_counts[
            None,
            :
        ]
    )

    disease_effects = (
        patient_effects.mean(
            axis=1
        )
    )

    return (
        disease_effects,
        patient_effects,
        unique_patients
    )


## Reference-reversion titration output state

In [ ]:
# Full reference-reversion titration

RAW_RESULT_FILE = (
    RR_DIR /
    "Fig6_RR_titration_RAW.csv"
)

PATIENT_RESULT_FILE = (
    RR_DIR /
    "Fig6_RR_titration_TOP_patient_effects.csv"
)

# Resume support
if RAW_RESULT_FILE.exists():

    rr_raw = pd.read_csv(
        RAW_RESULT_FILE
    )

else:

    rr_raw = pd.DataFrame()


if PATIENT_RESULT_FILE.exists():

    rr_patient = pd.read_csv(
        PATIENT_RESULT_FILE
    )

else:

    rr_patient = pd.DataFrame()


## Run full reference-reversion titration

## Clinical reference baselines and gaps

In [ ]:
def get_reference_expression_baseline(disease):
    """
    Patient-balanced clinical-reference expression vector using exact final anchors.
    """
    if disease == CART:
        ref_idx = get_anchor_indices(CART, "temporal_ref")
        baseline = patient_balanced_mean(adata_m.X, ref_idx)
    elif disease == COVID:
        stage_ref_idx = get_anchor_indices(COVID, "stage_ref")
        temporal_ref_idx = get_anchor_indices(COVID, "temporal_ref")
        stage_baseline = patient_balanced_mean(adata_m.X, stage_ref_idx)
        temporal_baseline = patient_balanced_mean(adata_m.X, temporal_ref_idx)
        baseline = 0.5 * (stage_baseline + temporal_baseline)
    elif disease == SLE:
        ref_idx = get_anchor_indices(SLE, "severity_ref")
        baseline = patient_balanced_mean(adata_m.X, ref_idx)
    else:
        raise ValueError(disease)

    return np.asarray(baseline, dtype=np.float32).ravel()


reference_baseline_by_disease = {
    disease: get_reference_expression_baseline(disease)
    for disease in LOCALIZATION_DISEASES
}

for disease in LOCALIZATION_DISEASES:
    x = reference_baseline_by_disease[disease]
    print(disease, x.shape, np.isfinite(x).all())


In [ ]:
import os

analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path

TITRATION_DIR = resolve_analysis_path(os.environ.get("MAE_FEATURE_TITRATION_DIR", os.path.join("outputs", "ai", "Fig6_feature_titration")))

clinical_gap_df = pd.read_csv(
    TITRATION_DIR / "Fig6_titration_clinical_gaps.csv"
)

required_gap_cols = {
    "disease",
    "seed",
    "clinical_gap",
}

missing_gap_cols = required_gap_cols - set(clinical_gap_df.columns)

if missing_gap_cols:
    raise ValueError(
        f"Missing clinical-gap columns: {missing_gap_cols}"
    )

if not (clinical_gap_df["clinical_gap"] > 0).all():
    raise ValueError("A clinical gap is non-positive.")

display(
    clinical_gap_df[
        [
            "disease",
            "seed",
            "clinical_gap",
        ]
    ]
)

In [ ]:
# Disease evaluation loop
for eval_disease in LOCALIZATION_DISEASES:

    # Same exact risk-anchor cells for all K / profiles / seeds
    risk_idx = (
        select_balanced_cells(

            idx=get_anchor_indices(
                eval_disease,
                "risk"
            ),

            max_cells_per_patient=
                TITRATION_CELLS_PER_PATIENT,

            random_seed=
                TITRATION_RANDOM_SEED
        )
    )

    reference_baseline = (
        reference_baseline_by_disease[
            eval_disease
        ]
    )

    # Each disease evaluates:
    #   1. its own profile
    #   2. the conserved profile
    profiles_to_test = [
        eval_disease,
        "CONSERVED"
    ]

    for seed in FINAL_SEEDS:

        print(
            "\n"
            + "=" * 90
        )

        print(
            "Reference reversion |",
            eval_disease,
            "| seed =",
            seed
        )

        print(
            "=" * 90
        )


        model = load_model_for_seed(
            seed
        )


        state = build_disease_state(
            seed=seed,
            disease=eval_disease
        )


        clinical_gap = (
            clinical_gap_df[
                (
                    clinical_gap_df[
                        "disease"
                    ] == eval_disease
                )
                &
                (
                    clinical_gap_df[
                        "seed"
                    ] == seed
                )
            ][
                "clinical_gap"
            ]
            .iloc[0]
        )


        for profile in profiles_to_test:

            track_key = (
                f"{profile}__IN__{eval_disease}"
            )


            controls = titration_controls[
                track_key
            ]

            track_k_max = int(
                controls.get(
                    "track_K_MAX",
                    len(controls["target_genes"])
                )
            )

            for K in K_GRID:

                if K > track_k_max:
                    print(
                        f"Skip unavailable K: {profile} -> {eval_disease} | "
                        f"K={K} exceeds track-specific maximum {track_k_max}"
                    )
                    continue

                # ============================================
                # Resume protection
                # ============================================

                if len(
                    rr_raw
                ) > 0:

                    existing = rr_raw[
                        (
                            rr_raw[
                                "profile"
                            ] == profile
                        )
                        &
                        (
                            rr_raw[
                                "eval_disease"
                            ] == eval_disease
                        )
                        &
                        (
                            rr_raw[
                                "seed"
                            ] == seed
                        )
                        &
                        (
                            rr_raw[
                                "K"
                            ] == K
                        )
                    ]

                else:

                    existing = pd.DataFrame()


                expected_n = (
                    1
                    +
                    N_TITRATION_CONTROLS
                    +
                    N_TITRATION_CONTROLS
                )


                if len(
                    existing
                ) == expected_n:

                    print(
                        "Skip completed:",
                        profile,
                        eval_disease,
                        K,
                        seed
                    )

                    continue

                # Remove incomplete unit from interrupted run
                if len(
                    existing
                ) > 0:

                    keep = ~(
                        (
                            rr_raw[
                                "profile"
                            ] == profile
                        )
                        &
                        (
                            rr_raw[
                                "eval_disease"
                            ] == eval_disease
                        )
                        &
                        (
                            rr_raw[
                                "seed"
                            ] == seed
                        )
                        &
                        (
                            rr_raw[
                                "K"
                            ] == K
                        )
                    )

                    rr_raw = (
                        rr_raw[
                            keep
                        ]
                        .copy()
                    )


                print(
                    f"Running "
                    f"{profile} -> {eval_disease} | "
                    f"K={K}"
                )

                # Nested gene sets
                top_set = (
                    controls[
                        "target_genes"
                    ][
                        :K
                    ]
                )


                random_sets = [

                    trajectory[
                        :K
                    ]

                    for trajectory
                    in controls[
                        "random"
                    ][
                        :N_TITRATION_CONTROLS
                    ]
                ]


                low_sets = [

                    trajectory[
                        :K
                    ]

                    for trajectory
                    in controls[
                        "low"
                    ][
                        :N_TITRATION_CONTROLS
                    ]
                ]


                all_sets = (
                    [top_set]
                    +
                    random_sets
                    +
                    low_sets
                )

                # Reference-state counterfactual
                # load_model_for_seed() may return (model, metadata) in some setups
                # Unwrap the actual torch model before calling evaluate_reference_reversion_sets()
                model_obj = model
                if isinstance(model_obj, tuple):
                    if len(model_obj) == 0:
                        raise ValueError("Loaded model tuple is empty.")
                    model_obj = model_obj[0]

                if not hasattr(model_obj, "eval"):
                    raise TypeError(
                        f"Expected a torch model, got {type(model_obj)}. "
                        "If load_model_for_seed() returns (model, ...), unwrap the model first."
                    )

                model = model_obj

                result = evaluate_reference_reversion_sets(
                    model=model,
                    seed=seed,
                    state=state,
                    cell_idx=risk_idx,
                    gene_sets=all_sets,
                    reference_baseline=reference_baseline,
                    batch_size=TITRATION_BATCH_SIZE,
                    set_chunk_size=PERTURBATION_SET_CHUNK,
                )

                if not isinstance(result, tuple) or len(result) != 3:
                    raise TypeError(
                        "evaluate_reference_reversion_sets() must return "
                        "(effects, patient_effects, patients)."
                    )

                effects, patient_effects, patients = result

                normalized = (
                    effects
                    / clinical_gap
                )

                # Labels
                labels = (
                    ["top_attribution"]
                    +
                    [
                        "matched_random"
                    ]
                    * N_TITRATION_CONTROLS
                    +
                    [
                        "low_attribution"
                    ]
                    * N_TITRATION_CONTROLS
                )

                control_ids = (
                    [0]
                    +
                    list(
                        range(
                            N_TITRATION_CONTROLS
                        )
                    )
                    +
                    list(
                        range(
                            N_TITRATION_CONTROLS
                        )
                    )
                )

                unit_records = []

                for i in range(
                    len(effects)
                ):

                    unit_records.append({

                        "perturbation_method":
                            "reference_state_reversion",

                        "profile":
                            profile,

                        "eval_disease":
                            eval_disease,

                        "seed":
                            seed,

                        "K":
                            K,

                        "control_type":
                            labels[i],

                        "control_id":
                            control_ids[i],

                        "raw_risk_reduction":
                            float(
                                effects[i]
                            ),

                        "clinical_gap":
                            float(
                                clinical_gap
                            ),

                        "normalized_efficacy":
                            float(
                                normalized[i]
                            )
                    })

                rr_raw = pd.concat(
                    [
                        rr_raw,
                        pd.DataFrame(
                            unit_records
                        )
                    ],
                    ignore_index=True
                )

                # Patient-level TOP feature effects
                top_patient_normalized = (
                    patient_effects[
                        0
                    ]
                    / clinical_gap
                )

                patient_records = [

                    {
                        "profile":
                            profile,

                        "eval_disease":
                            eval_disease,

                        "seed":
                            seed,

                        "K":
                            K,

                        "patient_id":
                            patient,

                        "normalized_efficacy":
                            float(value)
                    }

                    for patient, value
                    in zip(
                        patients,
                        top_patient_normalized
                    )
                ]

                # Remove duplicate patient block
                if len(
                    rr_patient
                ) > 0:

                    keep = ~(
                        (
                            rr_patient[
                                "profile"
                            ] == profile
                        )
                        &
                        (
                            rr_patient[
                                "eval_disease"
                            ] == eval_disease
                        )
                        &
                        (
                            rr_patient[
                                "seed"
                            ] == seed
                        )
                        &
                        (
                            rr_patient[
                                "K"
                            ] == K
                        )
                    )


                    rr_patient = (
                        rr_patient[
                            keep
                        ]
                        .copy()
                    )

                rr_patient = pd.concat(
                    [
                        rr_patient,
                        pd.DataFrame(
                            patient_records
                        )
                    ],
                    ignore_index=True
                )

                # Immediate checkpoint
                rr_raw.to_csv(
                    RAW_RESULT_FILE,
                    index=False
                )


                rr_patient.to_csv(
                    PATIENT_RESULT_FILE,
                    index=False
                )

        del model

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()


In [ ]:
# Specificity + excess efficacy
rr_specificity_records = []

for (
    profile,
    eval_disease,
    seed,
    K
), tmp in rr_raw.groupby(

    [
        "profile",
        "eval_disease",
        "seed",
        "K"
    ],

    observed=True
):

    top = (
        tmp[
            tmp[
                "control_type"
            ] == "top_attribution"
        ][
            "normalized_efficacy"
        ]
        .iloc[0]
    )


    random_null = (
        tmp[
            tmp[
                "control_type"
            ] == "matched_random"
        ][
            "normalized_efficacy"
        ]
        .to_numpy()
    )

    low_null = (
        tmp[
            tmp[
                "control_type"
            ] == "low_attribution"
        ][
            "normalized_efficacy"
        ]
        .to_numpy()
    )

    random_median = np.median(
        random_null
    )

    low_median = np.median(
        low_null
    )

    # Exact empirical upper-tail probabilities
    random_tail = (
        1
        +
        np.sum(
            random_null >= top
        )
    ) / (
        1
        + len(
            random_null
        )
    )


    low_tail = (
        1
        +
        np.sum(
            low_null >= top
        )
    ) / (
        1
        + len(
            low_null
        )
    )

    # Excess effect beyond matched controls
    excess_random = (
        top
        - random_median
    )

    excess_low = (
        top
        - low_median
    )

    # Conservative:
    # must outperform the harder of the two controls
    conservative_excess = min(
        excess_random,
        excess_low
    )

    specificity_pass = bool(

        (top > 0)

        and

        (
            random_tail
            <= SPECIFICITY_ALPHA
        )

        and

        (
            low_tail
            <= SPECIFICITY_ALPHA
        )
    )

    rr_specificity_records.append({

        "profile":
            profile,

        "eval_disease":
            eval_disease,

        "seed":
            seed,

        "K":
            K,

        "top_efficacy":
            top,

        "random_null_median":
            random_median,

        "low_null_median":
            low_median,

        "excess_vs_random":
            excess_random,

        "excess_vs_low":
            excess_low,

        "conservative_excess":
            conservative_excess,

        "random_tail_probability":
            random_tail,

        "low_tail_probability":
            low_tail,

        "specificity_pass":
            specificity_pass
    })

rr_specificity = pd.DataFrame(
    rr_specificity_records
)

rr_specificity.to_csv(
    RR_DIR /
    "Fig6_RR_specificity_by_seed.csv",
    index=False
)

In [ ]:
# Aggregate across five neural-network seeds
rr_curve = (
    rr_specificity
    .groupby(
        [
            "profile",
            "eval_disease",
            "K"
        ],
        observed=True
    )
    .agg(

        median_top_efficacy=(
            "top_efficacy",
            "median"
        ),

        q25_top_efficacy=(
            "top_efficacy",
            lambda x:
                np.quantile(
                    x,
                    0.25
                )
        ),

        q75_top_efficacy=(
            "top_efficacy",
            lambda x:
                np.quantile(
                    x,
                    0.75
                )
        ),

        median_excess_vs_random=(
            "excess_vs_random",
            "median"
        ),

        median_excess_vs_low=(
            "excess_vs_low",
            "median"
        ),

        median_conservative_excess=(
            "conservative_excess",
            "median"
        ),

        q25_conservative_excess=(
            "conservative_excess",
            lambda x:
                np.quantile(
                    x,
                    0.25
                )
        ),

        q75_conservative_excess=(
            "conservative_excess",
            lambda x:
                np.quantile(
                    x,
                    0.75
                )
        ),

        n_specific_seeds=(
            "specificity_pass",
            "sum"
        )
    )
    .reset_index()
)

rr_curve[
    "specificity_pass"
] = (
    rr_curve[
        "n_specific_seeds"
    ]
    >= MIN_PASSING_SEEDS
)

rr_curve.to_csv(
    RR_DIR /
    "Fig6_RR_titration_curve_summary.csv",
    index=False
)

display(
    rr_curve
)

# 10. Saturation fitting, K90, and conserved 34-gene intersection

In [ ]:
# Six titration track
MODEL_COMPARISON_TRACKS = [

    (CART, CART),
    (COVID, COVID),
    (SLE, SLE),

    ("CONSERVED", CART),
    ("CONSERVED", COVID),
    ("CONSERVED", SLE)
]

model_comparison_records = []

model_comparison_fits = {}

for profile, eval_disease in (
    MODEL_COMPARISON_TRACKS
):

    tmp = rr_curve[
        (
            rr_curve[
                "profile"
            ] == profile
        )
        &
        (
            rr_curve[
                "eval_disease"
            ] == eval_disease
        )
    ].copy()


    result = (
        compare_titration_models(
            tmp
        )
    )


    track_name = (
        f"{profile}__IN__{eval_disease}"
    )


    model_comparison_fits[
        track_name
    ] = result


    model_comparison_records.append({

        "profile":
            profile,

        "eval_disease":
            eval_disease,

        "linear_slope":
            result[
                "linear_slope"
            ],

        "linear_AICc":
            result[
                "linear_AICc"
            ],

        "sat_amplitude":
            result[
                "sat_amplitude"
            ],

        "sat_tau":
            result[
                "sat_tau"
            ],

        "sat_K90":
            result[
                "sat_K90"
            ],

        "sat_AICc":
            result[
                "sat_AICc"
            ],

        "delta_AICc_linear_minus_sat":
            result[
                "delta_AICc_linear_minus_sat"
            ],

        "saturation_model_weight":
            result[
                "saturation_model_weight"
            ],

        "K90_within_tested_range":
            result[
                "K90_within_tested_range"
            ],

        "conclusion":
            result[
                "conclusion"
            ]
    })


titration_model_comparison = (
    pd.DataFrame(
        model_comparison_records
    )
)


display(
    titration_model_comparison
)

In [ ]:
# Check whether all K90-defined genes have positive attribution
K90_COUNTS = {
    CART: 229,
    COVID: 649,
    SLE: 116
}

for disease, K in K90_COUNTS.items():

    df = (
        full_attr_tables[disease]
        .sort_values(
            "median_attribution",
            ascending=False
        )
        .reset_index(drop=True)
    )

    selected = df.head(K)

    print(
        disease,
        "| K90 =", K,
        "| minimum attribution =",
        selected["median_attribution"].min(),
        "| positive genes =",
        int(
            (
                selected["median_attribution"] > 0
            ).sum()
        ),
        "/",
        K
    )

In [ ]:
# Combined disease-specific titration plot
# Three diseases in one field, with fitted K90 shown

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Select the three disease-specific tracks only
disease_order = [CART, COVID, SLE]

# Extract fitted K90 values from the existing model-comparison table
disease_k90_map = {}

for disease in disease_order:
    row = titration_model_comparison[
        (titration_model_comparison["profile"] == disease) &
        (titration_model_comparison["eval_disease"] == disease)
    ].iloc[0]

    disease_k90_map[disease] = float(row["sat_K90"])

print("Disease-specific K90:", disease_k90_map)

# Decide the plotting range

observed_k_max = float(rr_curve["K"].max())
fitted_k90_max = max(disease_k90_map.values())

x_plot_max = 1000

print("Observed K max =", observed_k_max)
print("Plotting x-axis max =", x_plot_max)

# Draw one combined plot
fig, ax = plt.subplots(figsize=(8.5, 6))

for disease in disease_order:
    # observed titration curve
    tmp = (
        rr_curve[
            (rr_curve["profile"] == disease) &
            (rr_curve["eval_disease"] == disease)
        ]
        .sort_values("K")
        .copy()
    )

    x_obs = tmp["K"].to_numpy(dtype=float)
    y_obs = tmp["median_top_efficacy"].to_numpy(dtype=float)

    # Use q25/q75 if they exist; otherwise fall back to points only
    has_q25 = "q25_top_efficacy" in tmp.columns
    has_q75 = "q75_top_efficacy" in tmp.columns

    if has_q25 and has_q75:
        y_q25 = tmp["q25_top_efficacy"].to_numpy(dtype=float)
        y_q75 = tmp["q75_top_efficacy"].to_numpy(dtype=float)

        yerr = np.vstack([
            y_obs - y_q25,
            y_q75 - y_obs
        ])

        ax.errorbar(
            x_obs,
            y_obs,
            yerr=yerr,
            fmt='o',
            capsize=3,
            alpha=0.9,
            label=f"{disease} observed"
        )
    else:
        ax.scatter(
            x_obs,
            y_obs,
            s=40,
            alpha=0.9,
            label=f"{disease} observed"
        )

    # fitted saturation curve
    fit_key = f"{disease}__IN__{disease}"
    fit_res = model_comparison_fits[fit_key]

    K_smooth = np.linspace(0, x_plot_max, 500)

    y_fit = saturating_origin(
        K_smooth,
        fit_res["sat_amplitude"],
        fit_res["sat_tau"]
    )

    ax.plot(
        K_smooth,
        y_fit,
        linewidth=2,
        label=f"{disease} fit"
    )

    # K90 marker
    k90 = disease_k90_map[disease]

    ax.axvline(
        k90,
        linestyle="--",
        alpha=0.7
    )

    y_k90 = saturating_origin(
        np.array([k90]),
        fit_res["sat_amplitude"],
        fit_res["sat_tau"]
    )[0]

    ax.text(
        k90,
        y_k90,
        f"  {disease} K90={k90:.0f}",
        va="bottom",
        ha="left"
    )

# Cosmetics
ax.set_title("Disease-specific reference-reversion titration")
ax.set_xlabel("Top-K reverted genes")
ax.set_ylabel("Normalized efficacy\n(fraction of clinical-risk separation erased)")
ax.set_xlim(0, x_plot_max)
ax.grid(True, alpha=0.3)

ax.legend(
    frameon=False,
    ncol=2,
    fontsize=9
)

fig.tight_layout()
plt.show()

In [ ]:
# K90-DEFINED RISK-ASSOCIATED GENE SETS
from pathlib import Path
import numpy as np
import pandas as pd

GENESET_DIR = (
    RR_DIR /
    "K90_defined_gene_sets"
)

GENESET_DIR.mkdir(
    exist_ok=True
)

# Extract disease-specific fitted K90

K90_BY_DISEASE = {}

for disease in [
    CART,
    COVID,
    SLE
]:

    row = (
        titration_model_comparison[
            (
                titration_model_comparison[
                    "profile"
                ] == disease
            )
            &
            (
                titration_model_comparison[
                    "eval_disease"
                ] == disease
            )
        ]
        .iloc[0]
    )

    # K must be an integer gene count
    K90_BY_DISEASE[
        disease
    ] = int(
        np.rint(
            row[
                "sat_K90"
            ]
        )
    )

print(
    "K90-defined feature counts:"
)

for disease, K in (
    K90_BY_DISEASE.items()
):

    print(
        f"  {disease}: {K}"
    )

In [ ]:
# Disease-specific K90 sets
k90_gene_tables = {}
k90_gene_sets = {}


for disease, K in (
    K90_BY_DISEASE.items()
):

    df = (
        full_attr_tables[
            disease
        ]
        .sort_values(
            "median_attribution",
            ascending=False
        )
        .reset_index(
            drop=True
        )
        .copy()
    )

    # Explicit disease-specific rank
    df[
        "disease_rank"
    ] = (
        np.arange(
            1,
            len(df) + 1
        )
    )

    selected = (
        df
        .head(K)
        .copy()
    )

    # Critical directionality check
    n_positive = int(
        (
            selected[
                "median_attribution"
            ] > 0
        ).sum()
    )


    print(
        "\n",
        disease,
        sep=""
    )

    print(
        "  fitted K90:",
        K
    )

    print(
        "  positive-attribution genes:",
        f"{n_positive}/{K}"
    )

    print(
        "  attribution at K90 boundary:",
        selected[
            "median_attribution"
        ].iloc[-1]
    )


    if n_positive != K:

        raise RuntimeError(
            f"{disease}: the K90-defined set "
            f"contains {K - n_positive} "
            "non-positive-attribution genes. "
            "Stop before defining the final set."
        )

    # Provenance columns
    selected[
        "K90"
    ] = K


    selected[
        "within_K90"
    ] = True


    selected[
        "normalized_K90_rank"
    ] = (
        selected[
            "disease_rank"
        ]
        / K
    )

    # COVID K90 extends beyond the empirical K<=500 range.
    selected[
        "K90_extrapolated_beyond_tested_range"
    ] = bool(
        K > max(K_GRID)
    )

    k90_gene_tables[
        disease
    ] = selected

    k90_gene_sets[
        disease
    ] = set(
        selected[
            "gene"
        ].astype(str)
    )

In [ ]:
# Export disease-specific K90 lists
for disease, df in (
    k90_gene_tables.items()
):

    K = K90_BY_DISEASE[
        disease
    ]

    df.to_csv(
        GENESET_DIR /
        f"Fig6_{disease}_K90_Top{K}_risk_features.csv",
        index=False
    )

    df[
        ["gene"]
    ].to_csv(
        GENESET_DIR /
        f"Fig6_{disease}_K90_Top{K}_genes.txt",
        index=False,
        header=False
    )

In [ ]:
# K90-set overlap
cart_set = (
    k90_gene_sets[
        CART
    ]
)

covid_set = (
    k90_gene_sets[
        COVID
    ]
)

sle_set = (
    k90_gene_sets[
        SLE
    ]
)


cart_covid = (
    cart_set
    & covid_set
)

cart_sle = (
    cart_set
    & sle_set
)

covid_sle = (
    covid_set
    & sle_set
)


conserved_set = (
    cart_set
    & covid_set
    & sle_set
)


overlap_summary = pd.DataFrame({

    "comparison": [
        "CAR-T ∩ COVID19",
        "CAR-T ∩ SLE",
        "COVID19 ∩ SLE",
        "CAR-T ∩ COVID19 ∩ SLE"
    ],

    "n_genes": [
        len(cart_covid),
        len(cart_sle),
        len(covid_sle),
        len(conserved_set)
    ]
})


display(
    overlap_summary
)


print(
    "\nConserved K90-defined genes:",
    len(conserved_set)
)

In [ ]:
cart_evidence = (
    extract_disease_evidence(
        CART,
        "CART"
    )
)

covid_evidence = (
    extract_disease_evidence(
        COVID,
        "COVID19"
    )
)

sle_evidence = (
    extract_disease_evidence(
        SLE,
        "SLE"
    )
)


conserved_genes_df = pd.DataFrame({
    "gene":
        sorted(
            conserved_set
        )
})


conserved_genes_df = (
    conserved_genes_df
    .merge(
        cart_evidence,
        on="gene",
        how="left"
    )
    .merge(
        covid_evidence,
        on="gene",
        how="left"
    )
    .merge(
        sle_evidence,
        on="gene",
        how="left"
    )
)

In [ ]:
# Relative K90 ranks
conserved_genes_df[
    "CART_K90_rank_fraction"
] = (
    conserved_genes_df[
        "CART_rank"
    ]
    /
    K90_BY_DISEASE[
        CART
    ]
)


conserved_genes_df[
    "COVID19_K90_rank_fraction"
] = (
    conserved_genes_df[
        "COVID19_rank"
    ]
    /
    K90_BY_DISEASE[
        COVID
    ]
)


conserved_genes_df[
    "SLE_K90_rank_fraction"
] = (
    conserved_genes_df[
        "SLE_rank"
    ]
    /
    K90_BY_DISEASE[
        SLE
    ]
)

In [ ]:
rank_fraction_cols = [
    "CART_K90_rank_fraction",
    "COVID19_K90_rank_fraction",
    "SLE_K90_rank_fraction"
]


conserved_genes_df[
    "mean_K90_rank_fraction"
] = (
    conserved_genes_df[
        rank_fraction_cols
    ]
    .mean(axis=1)
)


conserved_genes_df[
    "worst_K90_rank_fraction"
] = (
    conserved_genes_df[
        rank_fraction_cols
    ]
    .max(axis=1)
)

In [ ]:
# Add continuous conserved consensus evidence
conserved_extra_cols = [
    "gene",
    "median_conserved_score",
    "conserved_score_iqr",
    "positive_seed_fraction",
    "n_diseases_positive",
    "mean_disease_rank_percentile",
    "minimum_disease_rank_percentile",
    "rank_conserved",
]

missing_conserved_cols = [
    col
    for col in conserved_extra_cols
    if col not in conserved_profile_df.columns
]

if missing_conserved_cols:
    raise ValueError(
        f"Missing conserved-profile columns: {missing_conserved_cols}"
    )

conserved_genes_df = conserved_genes_df.merge(
    conserved_profile_df[conserved_extra_cols],
    on="gene",
    how="left",
)


In [ ]:
if (
    "median_conserved_score"
    in conserved_genes_df.columns
):

    conserved_genes_df = (
        conserved_genes_df
        .sort_values(
            "median_conserved_score",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

else:

    conserved_genes_df = (
        conserved_genes_df
        .sort_values(
            "mean_K90_rank_fraction",
            ascending=True
        )
        .reset_index(
            drop=True
        )
    )


conserved_genes_df[
    "conserved_intersection_rank"
] = (
    np.arange(
        1,
        len(
            conserved_genes_df
        ) + 1
    )
)

In [ ]:
# Export conserved K90 intersection
N_CONSERVED = len(
    conserved_genes_df
)


conserved_genes_df.to_csv(
    GENESET_DIR /
    (
        "Fig6_CONSERVED_K90_intersection_"
        f"{N_CONSERVED}genes_FULL.csv"
    ),
    index=False
)


conserved_genes_df[
    ["gene"]
].to_csv(
    GENESET_DIR /
    (
        "Fig6_CONSERVED_K90_intersection_"
        f"{N_CONSERVED}genes.txt"
    ),
    index=False,
    header=False
)


overlap_summary.to_csv(
    GENESET_DIR /
    "Fig6_K90_gene_set_overlap_summary.csv",
    index=False
)

In [ ]:
# Final gene-set manifest
gene_set_manifest = pd.DataFrame({

    "gene_set": [
        CART,
        COVID,
        SLE,
        "CONSERVED"
    ],

    "definition": [
        "Top-K90 disease-specific IG ranking",
        "Top-K90 disease-specific IG ranking",
        "Top-K90 disease-specific IG ranking",
        "Intersection of all three disease-specific K90 sets"
    ],

    "n_genes": [
        K90_BY_DISEASE[CART],
        K90_BY_DISEASE[COVID],
        K90_BY_DISEASE[SLE],
        len(conserved_set)
    ],

    "fitted_K90": [
        K90_BY_DISEASE[CART],
        K90_BY_DISEASE[COVID],
        K90_BY_DISEASE[SLE],
        np.nan
    ],

    "K90_extrapolated": [
        K90_BY_DISEASE[CART] > max(K_GRID),
        K90_BY_DISEASE[COVID] > max(K_GRID),
        K90_BY_DISEASE[SLE] > max(K_GRID),
        False
    ]
})


gene_set_manifest.to_csv(
    GENESET_DIR /
    "Fig6_K90_gene_set_manifest.csv",
    index=False
)


display(
    gene_set_manifest
)

In [ ]:
display(
    conserved_genes_df[
        [
            col
            for col in [
                "conserved_intersection_rank",
                "gene",

                "CART_rank",
                "COVID19_rank",
                "SLE_rank",

                "CART_attribution",
                "COVID19_attribution",
                "SLE_attribution",

                "median_conserved_score",
                "conserved_score_iqr",

                "mean_K90_rank_fraction",
                "worst_K90_rank_fraction"
            ]
            if col in conserved_genes_df.columns
        ]
    ].head(50)
)

# 11. Fig. 6G K90 display curves

In [ ]:
# Fig. 6G
# Disease-specific reference-state reversion titration

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [ ]:
import os

analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path

WORKDIR = resolve_analysis_path(os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai")))

RR_DIR = (
    WORKDIR
    / "Fig6_reference_reversion_titration"
)

CURVE_PATH = (
    RR_DIR
    / "Fig6_RR_titration_curve_summary.csv"
)

OUTDIR = (
    RR_DIR
    / "Fig6E_plot"
)

OUTDIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUT_PDF = (
    OUTDIR
    / "Fig6E_reference_reversion_K90.pdf"
)

OUT_PNG = (
    OUTDIR
    / "Fig6E_reference_reversion_K90.png"
)

OUT_CURVES = (
    OUTDIR
    / "Fig6E_reference_reversion_display_curves.csv"
)

OUT_SUMMARY = (
    OUTDIR
    / "Fig6E_reference_reversion_display_fit_summary.csv"
)

In [ ]:
DISEASE_ORDER = [
    "CAR-T_CRS",
    "COVID19",
    "SLE",
]

DISEASE_DISPLAY = {
    "CAR-T_CRS": "CAR-T CRS",
    "COVID19": "COVID-19",
    "SLE": "SLE",
}


# Frozen finalized K90 values
K90 = {
    "CAR-T_CRS": 229,
    "COVID19": 649,
    "SLE": 116,
}

DISEASE_COLORS = {
    "CAR-T_CRS": "#4C78A8",
    "COVID19": "#F58518",
    "SLE": "#54A24B",
}


# Plot geometry
FIGSIZE = (7.4, 5.1)

X_MAX = 1000

N_CURVE_POINTS = 800

# Point / line aesthetics
POINT_SIZE = 44
POINT_ALPHA = 0.92

ERRORBAR_LINEWIDTH = 1.15
ERRORBAR_CAPSIZE = 3.2

FIT_LINEWIDTH = 2.5

K90_LINEWIDTH = 1.5
K90_ALPHA = 0.62

GRID_ALPHA = 0.15

In [ ]:
curve_summary = pd.read_csv(
    CURVE_PATH
)

print("Input table shape:", curve_summary.shape)

print("\nProfiles:")
print(
    curve_summary["profile"]
    .value_counts()
)

display(curve_summary.head())

In [ ]:
# Restrict to the THREE disease-specific profiles

plot_df = (
    curve_summary.loc[
        curve_summary["profile"].isin(
            DISEASE_ORDER
        )
    ]
    .copy()
)


required_cols = [
    "profile",
    "K",
    "median_top_efficacy",
    "q25_top_efficacy",
    "q75_top_efficacy",
]

missing = [
    col
    for col in required_cols
    if col not in plot_df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )


for col in [
    "K",
    "median_top_efficacy",
    "q25_top_efficacy",
    "q75_top_efficacy",
]:

    plot_df[col] = pd.to_numeric(
        plot_df[col],
        errors="coerce",
    )


plot_df = (
    plot_df
    .dropna(
        subset=[
            "K",
            "median_top_efficacy",
            "q25_top_efficacy",
            "q75_top_efficacy",
        ]
    )
    .sort_values(
        [
            "profile",
            "K",
        ]
    )
    .reset_index(drop=True)
)


print("\nRows retained:")
print(
    plot_df.groupby(
        "profile"
    ).size()
)

display(plot_df.head())

In [ ]:
# Reconstruct the finalized saturation curves for display
# ============================================================
#
# Final model:
#
#     E(K) = A * [1 - exp(-K / tau)]
#
# and
#
#     K90 = tau * ln(10)
#
#
# The saved curve-summary table contains observed summarized
# efficacy values but not the original fitted A and tau.
#
# Therefore:
#
#   1. tau is FIXED from the finalized frozen K90;
#   2. only A is estimated from median_top_efficacy.
#
# This preserves the finalized K90 exactly while providing
# a smooth publication-grade representation of the fitted curve.
# ============================================================

def saturation_basis(K, tau):

    K = np.asarray(
        K,
        dtype=float,
    )

    return (
        1.0
        - np.exp(
            -K / tau
        )
    )


def estimate_A_fixed_tau(
    K,
    efficacy,
    tau,
):
    """
    Least-squares estimate of A when tau is fixed.

    Since:
        y = A * f(K)

    the closed-form solution is:
        A = sum(f*y) / sum(f^2)
    """

    f = saturation_basis(
        K,
        tau,
    )

    denominator = np.sum(
        f ** 2
    )

    if denominator <= 0:
        raise ValueError(
            "Invalid saturation basis."
        )

    A = (
        np.sum(
            f * efficacy
        )
        / denominator
    )

    return A


fit_rows = []
curve_rows = []


for disease in DISEASE_ORDER:

    sub = plot_df.loc[
        plot_df["profile"] == disease
    ].copy()


    x_obs = (
        sub["K"]
        .to_numpy(dtype=float)
    )

    y_obs = (
        sub["median_top_efficacy"]
        .to_numpy(dtype=float)
    )

    # Convert finalized K90 into tau
    tau = (
        K90[disease]
        / np.log(10)
    )

    # Estimate plateau A only
    A = estimate_A_fixed_tau(
        x_obs,
        y_obs,
        tau,
    )

    # Generate smooth display curve
    x_curve = np.linspace(
        0,
        X_MAX,
        N_CURVE_POINTS,
    )

    y_curve = (
        A
        * saturation_basis(
            x_curve,
            tau,
        )
    )

    fit_rows.append({
        "profile": disease,
        "K90": K90[disease],
        "tau_fixed": tau,
        "A_display_fit": A,
        "max_observed_K": x_obs.max(),
    })

    curve_rows.append(
        pd.DataFrame({
            "profile": disease,
            "K": x_curve,
            "fitted_efficacy": y_curve,
        })
    )

fit_df = pd.DataFrame(
    fit_rows
)

curve_df = pd.concat(
    curve_rows,
    ignore_index=True,
)

print("\nDisplay-curve parameters:")
display(fit_df)

In [ ]:
# Prepare asymmetric IQR error bars

# ============================================================
#
# Observed center:
#   median_top_efficacy
#
# Lower error:
#   median - q25
#
# Upper error:
#   q75 - median
#
# These are IQR-based dispersion bars, NOT SEM/SD.
# ============================================================

plot_df["err_lower"] = (
    plot_df["median_top_efficacy"]
    - plot_df["q25_top_efficacy"]
)

plot_df["err_upper"] = (
    plot_df["q75_top_efficacy"]
    - plot_df["median_top_efficacy"]
)

plot_df["err_lower"] = (
    plot_df["err_lower"]
    .clip(lower=0)
)

plot_df["err_upper"] = (
    plot_df["err_upper"]
    .clip(lower=0)
)

In [ ]:
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42


fig, ax = plt.subplots(
    figsize=FIGSIZE
)


for disease in DISEASE_ORDER:

    color = DISEASE_COLORS[
        disease
    ]


    obs_sub = plot_df.loc[
        plot_df["profile"] == disease
    ].copy()


    curve_sub = curve_df.loc[
        curve_df["profile"] == disease
    ].copy()


    # Observed median efficacy + IQR
    yerr = np.vstack(
        [
            obs_sub["err_lower"].to_numpy(),
            obs_sub["err_upper"].to_numpy(),
        ]
    )


    ax.errorbar(
        obs_sub["K"],
        obs_sub["median_top_efficacy"],

        yerr=yerr,

        fmt="o",

        markersize=np.sqrt(
            POINT_SIZE
        ),

        color=color,
        markerfacecolor=color,
        markeredgecolor=color,

        ecolor=color,

        elinewidth=ERRORBAR_LINEWIDTH,
        capsize=ERRORBAR_CAPSIZE,

        alpha=POINT_ALPHA,

        zorder=4,
    )

    # Fitted saturation curve

    ax.plot(
    curve_sub["K"],
    curve_sub["fitted_efficacy"],

    color=color,
    linewidth=FIT_LINEWIDTH,

    linestyle="-",
    solid_capstyle="round",

    zorder=3,
)
    # K90 vertical line
    ax.axvline(
        K90[disease],

        color=color,

        linewidth=K90_LINEWIDTH,

        linestyle="--",

        alpha=K90_ALPHA,

        zorder=2,
    )

In [ ]:
label_positions = {
    "SLE": {
        "x": 120,
        "dy": 0.025,
    },

    "CAR-T_CRS": {
        "x": 225,
        "dy": 0.025,
    },

    "COVID19": {
        "x": 655,
        "dy": 0.025,
    },
}


for disease in DISEASE_ORDER:

    color = DISEASE_COLORS[
        disease
    ]

    pars = fit_df.loc[
        fit_df["profile"] == disease
    ].iloc[0]


    x_label = label_positions[
        disease
    ]["x"]


    y_label = (
        pars["A_display_fit"]
        * saturation_basis(
            x_label,
            pars["tau_fixed"],
        )
    )


    if disease == "COVID19":

        text = (
            f"COVID-19\n"
            f"K90 = {K90[disease]}*"
        )

    else:

        text = (
            f"{DISEASE_DISPLAY[disease]}\n"
            f"K90 = {K90[disease]}"
        )


    ax.text(
        x_label,
        y_label
        + label_positions[disease]["dy"],

        text,

        color=color,

        fontsize=10.5,

        ha="left",
        va="bottom",

        linespacing=1.15,

        zorder=5,
    )

In [ ]:
observed_upper = (
    plot_df["q75_top_efficacy"]
    .max()
)

curve_upper = (
    curve_df["fitted_efficacy"]
    .max()
)

Y_MAX = (
    max(
        observed_upper,
        curve_upper,
    )
    * 1.10
)

ax.set_xlim(
    0,
    X_MAX,
)

ax.set_ylim(
    0,
    Y_MAX,
)

ax.set_xlabel(
    "Top-K reverted genes",
    fontsize=11.5,
)

ax.set_ylabel(
    "Normalized reversion efficacy",
    fontsize=11.5,
)

ax.tick_params(
    axis="both",
    labelsize=9.5,
)

# Light grid only
ax.grid(
    axis="both",
    linewidth=0.7,
    alpha=GRID_ALPHA,
)


for spine in ax.spines.values():

    spine.set_linewidth(
        0.9
    )

ax.set_title("")

plt.tight_layout()

In [ ]:
fig.savefig(
    OUT_PDF,
    bbox_inches="tight",
)

fig.savefig(
    OUT_PNG,
    dpi=600,
    bbox_inches="tight",
)

# Save exact data used for the displayed smooth curves
curve_df.to_csv(
    OUT_CURVES,
    index=False,
)

fit_df.to_csv(
    OUT_SUMMARY,
    index=False,
)

plt.show()

print("\nSaved:")
print(OUT_PDF)
print(OUT_PNG)
print(OUT_CURVES)
print(OUT_SUMMARY)

# 12. Fig. 6H conserved K90 intersection

In [ ]:
# Fig. 6F: clean three-circle Venn scaffold

from pathlib import Path
import os

import matplotlib.pyplot as plt

analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path

from matplotlib.patches import Circle

# Output
WORKDIR = resolve_analysis_path(os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai")))

OUTDIR = WORKDIR / "Fig6_conserved_core"
OUTDIR.mkdir(parents=True, exist_ok=True)

OUT_PDF = OUTDIR / "Fig6F_three_circle_venn.pdf"
OUT_PNG = OUTDIR / "Fig6F_three_circle_venn.png"

# Aesthetics

DISEASE_COLORS = {
    "CAR-T_CRS": "#4C78A8",
    "COVID19": "#F58518",
    "SLE": "#54A24B",
}

FILL_ALPHA = 0.2
EDGE_WIDTH = 1.5
RADIUS = 1.5

# Symmetric three-way overlap
CENTERS = {
    "CAR-T_CRS": (-0.95, 0.45),
    "COVID19":   ( 0.95, 0.45),
    "SLE":       ( 0.00,-0.75),
}

# Draw
from matplotlib.colors import to_rgba

fig, ax = plt.subplots(
    figsize=(5.2, 4.8)
)

for disease in [
    "CAR-T_CRS",
    "COVID19",
    "SLE",
]:

    base_color = DISEASE_COLORS[disease]

    circle = Circle(
        CENTERS[disease],
        RADIUS,

        # Transparent interior only
        facecolor=to_rgba(base_color, FILL_ALPHA),

        # Fully opaque boundary
        edgecolor=base_color,

        linewidth=EDGE_WIDTH,

        alpha=None,

        antialiased=True,
    )

    ax.add_patch(circle)

# Final styling
ax.set_aspect("equal")

ax.set_xlim(-2.8, 2.8)
ax.set_ylim(-2.6, 2.5)

ax.axis("off")

plt.tight_layout()

# Save
fig.savefig(
    OUT_PDF,
    bbox_inches="tight",
    transparent=True,
)

fig.savefig(
    OUT_PNG,
    dpi=600,
    bbox_inches="tight",
    transparent=True,
)

plt.show()

# 13. Top100 atlas-program reference-state reversion

## Atlas-program imports, paths, and selection parameters

In [ ]:
# ============================================================
# Fig. 6 — Atlas-program gene-set construction
#
# DEFINITION:
#   1. Start from the COMPLETE DEG list for each Fig. 2
#      monocyte state.
#   2. Keep positively enriched genes.
#   3. If pvals_adj exists, require pvals_adj < 0.05.
#   4. Intersect with the frozen 5000-HVG MAE feature space.
#   5. Rank the AVAILABLE genes by logfoldchanges.
#   6. Select Top-100 AVAILABLE genes.
#   7. If fewer than 100 eligible genes exist, use all.
#
# IMPORTANT:
# Selection is entirely independent of:
#   - Fig. 6 IG attribution
#   - reversion efficacy
#   - disease-specific risk scores
#   - CS_score
#
# Therefore the biological gene sets remain atlas-derived.
# ============================================================


from pathlib import Path
import os

import numpy as np
import pandas as pd
import scanpy as sc

analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path


# Paths
ROOT = resolve_analysis_path(
    os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai"))
)

ADATA_FILE = (
    ROOT
    / "adata_mono_MAE_HVG5000.h5ad"
)

DEG_FILES = {

    "Inflammatory_CD14_IL1B":
        ROOT / "DEGs_Mono_CD14_IL1B_vs_Other.csv",

    "Emergency_CD14_S100A8":
        ROOT / "DEGs_Mono_CD14_S100A8_vs_Other.csv",

    "CD16_sensing_LST1":
        ROOT / "DEGs_Mono_CD16_LST1_vs_Other.csv",
}

OUT_DIR = (
    ROOT
    / "Fig6_atlas_program_reversion"
    / "Top100_available_programs"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Selection parameters
TOP_N = 100

# Only genes positively enriched
MIN_LOGFC = 0.0

# Applied only if the DEG file contains pvals_adj
PADJ_MAX = 0.05

## Load frozen feature space

In [ ]:
# Load exact frozen 5000-HVG feature space
adata_hvg = sc.read_h5ad(
    ADATA_FILE,
    backed="r", # prevent loading the entire object into memory
)

if adata_hvg.n_vars != 5000:

    raise ValueError(
        f"Expected 5000 frozen model genes, "
        f"found {adata_hvg.n_vars}"
    )

frozen_genes = pd.Index(
    adata_hvg.var_names.astype(str)
)

frozen_gene_set = set(
    frozen_genes
)

print(
    "Frozen model feature space:",
    len(frozen_genes),
    "genes",
)

adata_hvg.file.close()

## Select Top100 available programs

In [ ]:
# Construct Top-100 AVAILABLE marker programs
selected_programs = {}

selected_tables = {}

qc_rows = []

all_selected_rows = []

In [ ]:
for program, path in (
    DEG_FILES.items()
):

    filename = path.name


    print("\n")
    print("=" * 70)
    print(program)
    print("=" * 70)

    print(
        "DEG file:",
        path,
    )


    df = pd.read_csv(
        path
    )

    # Required columns
    required_cols = {
        "names",
        "logfoldchanges",
    }

    missing = (
        required_cols
        - set(
            df.columns
        )
    )

    if missing:

        raise ValueError(
            f"{filename} is missing "
            f"required columns: {missing}"
        )

    df = df.copy()

    df["names"] = (
        df["names"]
        .astype(str)
    )

    # Remove invalid / duplicate gene names
    df = df.loc[
        df["names"].notna()
        &
        df["logfoldchanges"].notna()
    ].copy()

    df = df.drop_duplicates(
        subset="names",
        keep="first",
    )

    n_original = len(
        df
    )

    # Positive atlas markers only
    df = df.loc[
        df["logfoldchanges"]
        > MIN_LOGFC
    ].copy()

    n_positive = len(
        df
    )

    # Significance filter, only if available
    if "pvals_adj" in df.columns:

        df = df.loc[
            df["pvals_adj"].notna()
            &
            (
                df["pvals_adj"]
                < PADJ_MAX
            )
        ].copy()

    n_significant_positive = len(
        df
    )

    available = df.loc[
        df["names"].isin(
            frozen_gene_set
        )
    ].copy()

    n_available = len(
        available
    )

    # Rank model-available atlas markers by enrichment
    available = (
        available
        .sort_values(
            "logfoldchanges",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )
    available[
        "available_marker_rank"
    ] = (
        np.arange(
            len(
                available
            )
        )
        + 1
    )

    # Select Top-100 available genes
    # If <100 eligible genes exist, all are retained
    n_selected = min(
        TOP_N,
        n_available,
    )

    selected = (
        available
        .head(
            n_selected
        )
        .copy()
    )

    selected[
        "program"
    ] = program

    selected[
        "selection_rule"
    ] = (
        f"Top-{TOP_N} positive DEG markers "
        f"after frozen-5000 intersection"
    )

    selected[
        "n_requested"
    ] = TOP_N

    selected[
        "n_selected"
    ] = n_selected

    selected_genes = (
        selected["names"]
        .tolist()
    )

    selected_programs[
        program
    ] = selected_genes

    selected_tables[
        program
    ] = selected

    # QC
    qc_rows.append({

        "program":
            program,

        "deg_file":
            str(path),

        "n_original_DEG_rows":
            n_original,

        "n_positive_logFC":
            n_positive,

        "n_positive_significant_if_padj_available":
            n_significant_positive,

        "n_available_in_frozen_5000":
            n_available,

        "target_N":
            TOP_N,

        "n_selected":
            n_selected,

        "used_all_available_because_lt_100":
            n_available < TOP_N,
    })

    # Save individual program
    selected.to_csv(
        OUT_DIR
        / f"Fig6_{program}_Top100_AVAILABLE.csv",
        index=False,
    )


    pd.Series(
        selected_genes,
        name="gene",
    ).to_csv(
        OUT_DIR
        / f"Fig6_{program}_Top100_AVAILABLE_genes.txt",
        index=False,
        header=False,
    )


    # Add to combined long table
    for _, row in (
        selected.iterrows()
    ):

        all_selected_rows.append({

            "program":
                program,

            "gene":
                row["names"],

            "available_marker_rank":
                int(
                    row[
                        "available_marker_rank"
                    ]
                ),

            "logfoldchanges":
                float(
                    row[
                        "logfoldchanges"
                    ]
                ),

            "score":
                (
                    float(
                        row["scores"]
                    )
                    if (
                        "scores"
                        in row.index
                        and pd.notna(
                            row["scores"]
                        )
                    )
                    else np.nan
                ),

            "pvals_adj":
                (
                    float(
                        row["pvals_adj"]
                    )
                    if (
                        "pvals_adj"
                        in row.index
                        and pd.notna(
                            row["pvals_adj"]
                        )
                    )
                    else np.nan
                ),
        })


    print(
        f"Original DEG rows:       {n_original}"
    )

    print(
        f"Positive markers:        {n_positive}"
    )

    print(
        f"Available in HVG5000:    {n_available}"
    )

    print(
        f"Selected for reversion:  {n_selected}"
    )


    print(
        "\nSelected genes:"
    )

    print(
        ", ".join(
            selected_genes
        )
    )

In [ ]:
# combined QC table and long-format table of all selected genes

qc_df = pd.DataFrame(
    qc_rows
)


combined_df = pd.DataFrame(
    all_selected_rows
)


qc_df.to_csv(
    OUT_DIR
    / "Fig6_Top100_AVAILABLE_program_QC.csv",
    index=False,
)


combined_df.to_csv(
    OUT_DIR
    / "Fig6_Top100_AVAILABLE_programs_LONG.csv",
    index=False,
)

## Program-overlap QC

In [ ]:
# Program overlap QC

overlap_rows = []

for i, program_a in enumerate(
    DEG_FILES
):

    for program_b in list(
        DEG_FILES
    )[
        i + 1:
    ]:

        A = set(
            selected_programs[
                program_a
            ]
        )

        B = set(
            selected_programs[
                program_b
            ]
        )


        shared = sorted(
            A & B
        )


        overlap_rows.append({

            "program_A":
                program_a,

            "program_B":
                program_b,

            "n_A":
                len(A),

            "n_B":
                len(B),

            "n_overlap":
                len(shared),

            "jaccard":
                (
                    len(
                        shared
                    )
                    /
                    len(
                        A | B
                    )
                ),

            "shared_genes":
                ";".join(
                    shared
                ),
        })


overlap_df = pd.DataFrame(
    overlap_rows
)


overlap_df.to_csv(
    OUT_DIR
    / "Fig6_Top100_AVAILABLE_program_overlap_QC.csv",
    index=False,
)

## Canonical marker audit

In [ ]:
# Canonical biological marker audit
# Inspect whether the canonical markers for each program are present in the Top-100 AVAILABLE selection

canonical_markers = {

    "Inflammatory_CD14_IL1B": [
        "IL1A",
        "IL1B",
        "TNF",
        "CCL3",
        "CCL4",
        "CCL20",
        "CXCL2",
        "CXCL3",
        "PTGS2",
        "OSM",
        "NFKBIA",
    ],


    "Emergency_CD14_S100A8": [
        "S100A8",
        "S100A9",
        "S100A12",
        "IL1R2",
        "VCAN",
        "FCN1",
        "LCN2",
        "CTSD",
        "CTSS",
    ],


    "CD16_sensing_LST1": [
        "FCGR3A",
        "LST1",
        "FCER1G",
        "TYROBP",
        "LILRB1",
        "LILRB2",
        "IFITM1",
        "IFITM3",
        "ISG15",
        "IFI6",
        "IFI27",
        "IFIT1",
        "IFIT2",
        "IFIT3",
        "STAT1",
        "GBP1",
        "FCGR1A",
    ],
}

audit_rows = []

for program, markers in (
    canonical_markers.items()
):

    available_table = (
        selected_tables[
            program
        ]
    )

    selected_set = set(
        selected_programs[
            program
        ]
    )

    # Reload the full DEG table to determine marker ranks
    path = DEG_FILES[
        program
    ]

    full = pd.read_csv(
        path
    )

    full["names"] = (
        full["names"]
        .astype(str)
    )


    full = full.loc[
        full["logfoldchanges"].notna()
        &
        (
            full["logfoldchanges"]
            > MIN_LOGFC
        )
    ].copy()

    if "pvals_adj" in full.columns:

        full = full.loc[
            full["pvals_adj"].notna()
            &
            (
                full["pvals_adj"]
                < PADJ_MAX
            )
        ].copy()


    full = (
        full.loc[
            full["names"].isin(
                frozen_gene_set
            )
        ]
        .sort_values(
            "logfoldchanges",
            ascending=False,
        )
        .drop_duplicates(
            "names"
        )
        .reset_index(
            drop=True
        )
    )

    full[
        "available_rank"
    ] = (
        np.arange(
            len(full)
        )
        + 1
    )


    for gene in markers:

        hit = full.loc[
            full["names"]
            == gene
        ]

        if len(hit) == 0:

            audit_rows.append({

                "program":
                    program,

                "gene":
                    gene,

                "in_frozen_5000":
                    gene
                    in frozen_gene_set,

                "positive_significant_marker":
                    False,

                "available_rank":
                    np.nan,

                "logfoldchanges":
                    np.nan,

                "selected_Top100":
                    False,
            })

        else:

            row = hit.iloc[0]


            audit_rows.append({

                "program":
                    program,

                "gene":
                    gene,

                "in_frozen_5000":
                    True,

                "positive_significant_marker":
                    True,

                "available_rank":
                    int(
                        row[
                            "available_rank"
                        ]
                    ),

                "logfoldchanges":
                    float(
                        row[
                            "logfoldchanges"
                        ]
                    ),

                "selected_Top100":
                    gene
                    in selected_set,
            })

canonical_audit = pd.DataFrame(
    audit_rows
)


canonical_audit.to_csv(
    OUT_DIR
    / "Fig6_Top100_AVAILABLE_canonical_marker_audit.csv",
    index=False,
)

## Display program construction summary

In [ ]:
# Data summary
print("\n")
print("=" * 80)
print("FINAL PROGRAM-SIZE QC")
print("=" * 80)

print(
    qc_df[
        [
            "program",
            "n_available_in_frozen_5000",
            "target_N",
            "n_selected",
            "used_all_available_because_lt_100",
        ]
    ]
    .to_string(
        index=False
    )
)

print("\n")
print("=" * 80)
print("PROGRAM OVERLAP")
print("=" * 80)

print(
    overlap_df[
        [
            "program_A",
            "program_B",
            "n_overlap",
            "jaccard",
            "shared_genes",
        ]
    ]
    .to_string(
        index=False
    )
)

print("\n")
print("=" * 80)
print("CANONICAL MARKER AUDIT")
print("=" * 80)

print(
    canonical_audit
    .to_string(
        index=False
    )
)

print(
    f"\nOutputs saved to:\n{OUT_DIR}"
)


## Top100 matched-reversion imports and configuration

In [ ]:
# ============================================================
# Fig. 6 — Top100-available atlas-program reference reversion
# with matched controls
#
# FROZEN PROGRAM DEFINITION
# -------------------------
# For each Fig. 2 monocyte state:
#
# complete positive DEG list
#   -> intersect frozen 5000 HVGs
#   -> rank by atlas logFC
#   -> Top 100 available genes
#      (or all if <100 available)
#
# IMPORTANT:
# Gene selection is already complete before this cell.
# This cell does NOT modify program membership.
#
#
# COUNTERFACTUAL
# --------------
# Atlas-program genes in risk cells
#   -> disease-specific patient-balanced reference expression
#   -> exact frozen MAE encoder
#   -> disease-specific frozen risk projection
#
#
# MATCHED CONTROLS
# ----------------
# Each biological program is compared with equal-sized random
# gene sets matched on:
#
#   1) mean expression
#   2) detection fraction
#   3) |risk - reference| expression displacement
#
# No common-N truncation is used.
# ============================================================


# Imports
from pathlib import Path
import os
import json
import gc

import numpy as np
import pandas as pd
import scanpy as sc

import torch
import torch.nn as nn

from scipy import sparse
from scipy.stats import rankdata

import matplotlib.pyplot as plt

analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path


# Configuration
ROOT = resolve_analysis_path(
    os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai"))
)

ADATA_FILE = (
    ROOT
    / "adata_mono_MAE_HVG5000.h5ad"
)

MODEL_DIR = (
    ROOT
    / "Fig6_MAE_models"
)

LATENT_DIR = (
    ROOT
    / "Fig6_MAE_latent"
)

# Newly frozen Top100-available gene definition
PROGRAM_FILE = (
    ROOT
    / "Fig6_atlas_program_reversion"
    / "Top100_available_programs"
    / "Fig6_Top100_AVAILABLE_programs_LONG.csv"
)

OUT_DIR = (
    ROOT
    / "Fig6_atlas_program_reversion_Top100_matched"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SEEDS = [
    20260810,
    20260811,
    20260812,
    20260813,
    20260814,
]

DISEASES = [
    "CAR-T_CRS",
    "COVID19",
    "SLE",
]

PROGRAM_ORDER = [
    "Inflammatory_CD14_IL1B",
    "Emergency_CD14_S100A8",
    "CD16_sensing_LST1",
]

# Matched-control parameters

# Number of independently generated null gene sets PER disease x biological program
# 100 retains consistency with your previous formal reference-reversion controls
N_MATCHED_CONTROLS = 100


# For each biological gene, candidate control genes are ranked
# by matching distance. One is randomly selected from among
# the nearest MATCH_POOL_K unused candidates
#
# Larger values:
#   more null-set diversity, slightly looser matching
#
# Smaller values:
#   tighter matching, but controls become more repetitive
MATCH_POOL_K = 50

# Equal weights because the three dimensions are converted
# to percentile ranks before distance calculation.
MATCH_WEIGHTS = np.array(
    [
        1.0,   # mean expression
        1.0,   # detection fraction
        1.0,   # risk-reference displacement
    ],
    dtype=np.float32,
)

CONTROL_RANDOM_SEED = 20260817

# GPU batch size
ENCODE_BATCH_SIZE = 2048

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    DEVICE
)

## Load AnnData and clinical labels

In [ ]:
# Load frozen AnnData
if not ADATA_FILE.exists():

    raise FileNotFoundError(
        ADATA_FILE
    )

adata = sc.read_h5ad(
    ADATA_FILE
)

if adata.shape != (
    306061,
    5000,
):

    print(
        "WARNING: observed AnnData shape:",
        adata.shape
    )

if adata.n_vars != 5000:

    raise ValueError(
        f"Expected frozen 5000-HVG input, "
        f"found {adata.n_vars} genes."
    )

required_obs = [
    "disease",
    "patient_id",
    "sample_id",
    "severity",
    "sampling_time",
]

missing_obs = [
    col
    for col in required_obs
    if col not in adata.obs.columns
]

if missing_obs:

    raise ValueError(
        f"Missing metadata columns: "
        f"{missing_obs}"
    )

genes = pd.Index(
    adata.var_names.astype(str)
)

gene_to_idx = {
    gene: i
    for i, gene
    in enumerate(genes)
}

patient_ids = (
    adata.obs["patient_id"]
    .astype(str)
    .to_numpy()
)

print(
    f"Loaded: "
    f"{adata.n_obs:,} cells x "
    f"{adata.n_vars:,} genes"
)

# Exact metadata masks
disease_meta = (
    adata.obs["disease"]
    .astype("string")
)

severity_meta = (
    adata.obs["severity"]
    .astype("string")
)

sampling_meta = (
    adata.obs["sampling_time"]
    .astype("string")
)

def exact_mask(
    series,
    value,
):

    return (
        series.eq(value)
        .fillna(False)
        .to_numpy(dtype=bool)
    )

# Disease
is_CART = exact_mask(
    disease_meta,
    "CAR-T_CRS",
)

is_COVID = exact_mask(
    disease_meta,
    "COVID19",
)

is_SLE = exact_mask(
    disease_meta,
    "SLE",
)

# Severity
is_severe = exact_mask(
    severity_meta,
    "Severe",
)

is_moderate = exact_mask(
    severity_meta,
    "Moderate",
)

# CAR-T sampling times
is_CART_pro = exact_mask(
    sampling_meta,
    "CAR-T_CRS_pro",
)

is_CART_con = exact_mask(
    sampling_meta,
    "CAR-T_CRS_con",
)

is_CART_before = exact_mask(
    sampling_meta,
    "CAR-T_CRS_before",
)


# COVID sampling times
is_COVID_pro = exact_mask(
    sampling_meta,
    "COVID19_pro",
)

is_COVID_con = exact_mask(
    sampling_meta,
    "COVID19_con",
)

## Build clinical masks

In [ ]:
def make_clinical_masks(
    disease,
):

    # --------------------------------------------------------
    # CAR-T CRS
    #
    # Risk:
    #   Severe progression
    #
    # Reference:
    #   Severe before + Severe convalescence
    # --------------------------------------------------------

    if disease == "CAR-T_CRS":

        risk = (
            is_CART
            & is_severe
            & is_CART_pro
        )


        temporal_reference = (
            is_CART
            & is_severe
            & (
                is_CART_before
                | is_CART_con
            )
        )


        return {
            "risk":
                risk,

            "reference":
                temporal_reference,

            "temporal_reference":
                temporal_reference,
        }


    # --------------------------------------------------------
    # COVID-19
    #
    # Risk:
    #   Severe progression
    #
    # Stage reference:
    #   Moderate progression
    #
    # Temporal reference:
    #   Severe convalescence
    # --------------------------------------------------------

    elif disease == "COVID19":

        risk = (
            is_COVID
            & is_severe
            & is_COVID_pro
        )


        stage_reference = (
            is_COVID
            & is_moderate
            & is_COVID_pro
        )


        temporal_reference = (
            is_COVID
            & is_severe
            & is_COVID_con
        )


        return {
            "risk":
                risk,

            "reference":
                (
                    stage_reference
                    | temporal_reference
                ),

            "stage_reference":
                stage_reference,

            "temporal_reference":
                temporal_reference,
        }


    # --------------------------------------------------------
    # SLE
    #
    # Risk:
    #   Severe
    #
    # Reference:
    #   Moderate
    # --------------------------------------------------------

    elif disease == "SLE":

        risk = (
            is_SLE
            & is_severe
        )


        severity_reference = (
            is_SLE
            & is_moderate
        )


        return {
            "risk":
                risk,

            "reference":
                severity_reference,

            "severity_reference":
                severity_reference,
        }


    raise ValueError(
        disease
    )


clinical_masks = {
    disease:
        make_clinical_masks(
            disease
        )

    for disease in DISEASES
}


print("\n")
print("=" * 70)
print("CLINICAL ANCHOR QC")
print("=" * 70)


for disease in DISEASES:

    print(
        f"\n{disease}"
    )


    for name, mask in (
        clinical_masks[
            disease
        ].items()
    ):

        print(
            f"  {name:22s}"
            f"cells={mask.sum():7,d}  "
            f"patients="
            f"{len(np.unique(patient_ids[mask]))}"
        )


## Load frozen Top100 programs

In [ ]:
# Load frozen Top100-available programs
if not PROGRAM_FILE.exists():

    raise FileNotFoundError(
        PROGRAM_FILE
    )

program_table = pd.read_csv(
    PROGRAM_FILE
)

required_program_cols = {
    "program",
    "gene",
    "available_marker_rank",
}

missing_program_cols = (
    required_program_cols
    - set(
        program_table.columns
    )
)

if missing_program_cols:

    raise ValueError(
        f"Program file missing columns: "
        f"{missing_program_cols}"
    )

program_genes = {}

for program in PROGRAM_ORDER:

    x = (
        program_table.loc[
            program_table[
                "program"
            ]
            == program
        ]
        .sort_values(
            "available_marker_rank"
        )
    )

    gs = (
        x["gene"]
        .astype(str)
        .tolist()
    )

    # Verify every selected gene is truly in the encoder input
    missing_genes = [
        gene
        for gene in gs
        if gene not in gene_to_idx
    ]

    if missing_genes:

        raise ValueError(
            f"{program} contains genes absent "
            f"from frozen model:\n"
            f"{missing_genes}"
        )

    if len(gs) == 0:

        raise ValueError(
            f"{program} has zero genes."
        )

    program_genes[
        program
    ] = gs

print("\n")
print("=" * 70)
print("FROZEN TOP100-AVAILABLE PROGRAMS")
print("=" * 70)

for program in PROGRAM_ORDER:

    print(
        f"\n{program}: "
        f"N={len(program_genes[program])}"
    )

program_sizes = pd.DataFrame([
    {
        "program":
            program,

        "n_genes":
            len(
                program_genes[
                    program
                ]
            ),
    }

    for program in PROGRAM_ORDER
])

program_sizes.to_csv(
    OUT_DIR
    / "Fig6_Top100_program_sizes.csv",
    index=False,
)

## Define encoder and expression-matching helpers

In [ ]:
class FrozenMAEEncoder(
    nn.Module
):

    def __init__(
        self,
    ):

        super().__init__()


        self.encoder = (
            nn.Sequential(

                nn.Linear(
                    5000,
                    512,
                ),

                nn.LayerNorm(
                    512,
                    eps=1e-5,
                ),

                nn.GELU(),

                nn.Linear(
                    512,
                    128,
                ),

                nn.LayerNorm(
                    128,
                    eps=1e-5,
                ),

                nn.GELU(),

                nn.Linear(
                    128,
                    64,
                ),
            )
        )


    @torch.no_grad()
    def encode(
        self,
        x,
    ):

        if not torch.is_tensor(
            x
        ):

            x = torch.as_tensor(
                x,
                dtype=torch.float32,
            )


        x = x.to(
            DEVICE,
            dtype=torch.float32,
        )


        z = self.encoder(
            x
        )


        return (
            z
            .detach()
            .cpu()
            .numpy()
        )


def load_frozen_encoder(
    seed,
):

    model_path = (
        MODEL_DIR
        / f"Fig6_MAE_seed_{seed}.pt"
    )


    checkpoint = torch.load(
        model_path,
        map_location="cpu",
        weights_only=False,
    )


    state = {
        key: value

        for key, value
        in checkpoint[
            "model_state_dict"
        ].items()

        if key.startswith(
            "encoder."
        )
    }


    model = (
        FrozenMAEEncoder()
    )


    model.load_state_dict(
        state,
        strict=True,
    )


    return (
        model
        .to(DEVICE)
        .eval()
    )

# Patient-balanced expression/detection summaries
def patient_balanced_expr_detect(
    X,
    mask,
):
    """
    Equal patient weighting.

    For each patient:
        calculate mean expression per gene
        calculate detection fraction per gene

    Then average patient summaries equally.
    """

    pts = np.unique(
        patient_ids[
            mask
        ]
    )


    expr_list = []
    detect_list = []


    for pt in pts:

        idx = np.where(
            mask
            &
            (
                patient_ids
                == pt
            )
        )[0]


        X_pt = X[
            idx
        ]


        if sparse.issparse(
            X_pt
        ):

            expr = np.asarray(
                X_pt.mean(
                    axis=0
                )
            ).ravel()


            detect = np.asarray(
                (
                    X_pt > 0
                ).mean(
                    axis=0
                )
            ).ravel()


        else:

            X_arr = np.asarray(
                X_pt
            )


            expr = (
                X_arr.mean(
                    axis=0
                )
            )


            detect = (
                (
                    X_arr > 0
                )
                .mean(
                    axis=0
                )
            )


        expr_list.append(
            expr.astype(
                np.float32
            )
        )


        detect_list.append(
            detect.astype(
                np.float32
            )
        )


    return (
        np.mean(
            np.stack(
                expr_list,
                axis=0,
            ),
            axis=0,
        ).astype(
            np.float32
        ),

        np.mean(
            np.stack(
                detect_list,
                axis=0,
            ),
            axis=0,
        ).astype(
            np.float32
        ),
    )

## Compute reference expression and matching features

In [ ]:
reference_expression = {}

matching_features = {}

for disease in DISEASES:

    print(
        f"\nPreparing matching space: "
        f"{disease}"
    )

    masks = clinical_masks[
        disease
    ]

    risk_expr, risk_detect = (
        patient_balanced_expr_detect(
            adata.X,
            masks["risk"],
        )
    )

    ref_expr, ref_detect = (
        patient_balanced_expr_detect(
            adata.X,
            masks["reference"],
        )
    )

    reference_expression[
        disease
    ] = ref_expr

    # Mean abundance context
    context_expression = (
        0.5
        * (
            risk_expr
            + ref_expr
        )
    )

    # Detection context
    context_detection = (
        0.5
        * (
            risk_detect
            + ref_detect
        )
    )

    # Size of the input perturbation caused by reversion
    abs_displacement = (
        np.abs(
            risk_expr
            - ref_expr
        )
    )

    # Convert all three quantities to percentile ranks
    n_features = (
        adata.n_vars
    )

    expr_rank = (
        rankdata(
            context_expression,
            method="average",
        )
        / n_features
    )

    detection_rank = (
        rankdata(
            context_detection,
            method="average",
        )
        / n_features
    )

    displacement_rank = (
        rankdata(
            abs_displacement,
            method="average",
        )
        / n_features
    )

    matching_features[
        disease
    ] = np.column_stack(
        [
            expr_rank,
            detection_rank,
            displacement_rank,
        ]
    ).astype(
        np.float32
    )

## Build matched-control gene sets

## Initialize matched-control candidate state

In [ ]:
all_program_genes = set(
    gene

    for gs
    in program_genes.values()

    for gene in gs
)

# Prevent control sets from borrowing genes from ANY of the three biological programs
control_candidate_indices = (
    np.array(
        [
            i
            for i, gene
            in enumerate(genes)

            if gene
            not in all_program_genes
        ],
        dtype=int,
    )
)

rng = np.random.default_rng(
    CONTROL_RANDOM_SEED
)

matched_controls = {}

matching_pair_rows = []

matching_set_rows = []

## Sample matched-control gene sets

In [ ]:
for disease in DISEASES:

    feature_matrix = (
        matching_features[
            disease
        ]
    )

    for program in PROGRAM_ORDER:

        target_genes = (
            program_genes[
                program
            ]
        )

        target_indices = np.array(
            [
                gene_to_idx[g]
                for g in target_genes
            ],
            dtype=int,
        )

        print(
            f"\nMatched controls: "
            f"{disease} / {program} "
            f"(N={len(target_genes)})"
        )

        # Calculate candidate ordering only ONCE per target
        # gene rather than once per random control replicate

        nearest_candidates = {}


        for target_idx in (
            target_indices
        ):

            difference = (
                feature_matrix[
                    control_candidate_indices
                ]
                -
                feature_matrix[
                    target_idx
                ]
            )


            distance = np.sqrt(
                np.sum(
                    (
                        difference
                        * MATCH_WEIGHTS
                    )
                    ** 2,
                    axis=1,
                )
            )


            order = np.argsort(
                distance
            )


            nearest_candidates[
                target_idx
            ] = (
                control_candidate_indices[
                    order
                ],
                distance[
                    order
                ],
            )


        program_controls = []


        for control_id in range(
            N_MATCHED_CONTROLS
        ):

            used = set()

            chosen_indices = []

            selected_distances = []

            # Randomize assignment order so the first gene
            # does not always receive preferential access to
            # its closest candidate
            target_order = (
                target_indices.copy()
            )

            rng.shuffle(
                target_order
            )

            for target_idx in (
                target_order
            ):

                candidate_order, candidate_distances = (
                    nearest_candidates[
                        target_idx
                    ]
                )

                # Prefer the nearest MATCH_POOL_K candidates
                # that have not already been used
                available_local = [
                    (
                        int(candidate_idx),
                        float(candidate_distance),
                    )

                    for candidate_idx, candidate_distance
                    in zip(
                        candidate_order[
                            :MATCH_POOL_K
                        ],
                        candidate_distances[
                            :MATCH_POOL_K
                        ],
                    )

                    if int(
                        candidate_idx
                    )
                    not in used
                ]

                # Fallback:
                # if all local candidates were consumed,
                # search the full ranked candidate list
                if len(
                    available_local
                ) == 0:

                    available_local = [
                        (
                            int(candidate_idx),
                            float(candidate_distance),
                        )

                        for candidate_idx, candidate_distance
                        in zip(
                            candidate_order,
                            candidate_distances,
                        )

                        if int(
                            candidate_idx
                        )
                        not in used
                    ]

                choice_pos = int(
                    rng.integers(
                        len(
                            available_local
                        )
                    )
                )

                chosen_idx, match_distance = (
                    available_local[
                        choice_pos
                    ]
                )

                used.add(
                    chosen_idx
                )

                chosen_indices.append(
                    chosen_idx
                )

                selected_distances.append(
                    match_distance
                )

                matching_pair_rows.append({
                    "disease":
                        disease,

                    "program":
                        program,

                    "control_id":
                        control_id,

                    "target_gene":
                        genes[
                            target_idx
                        ],

                    "control_gene":
                        genes[
                            chosen_idx
                        ],

                    "matching_distance":
                        match_distance,
                })

            control_genes = [
                genes[
                    idx
                ]
                for idx
                in chosen_indices
            ]

            if len(
                control_genes
            ) != len(
                target_genes
            ):

                raise RuntimeError(
                    "Matched control size mismatch."
                )

            if len(
                set(
                    control_genes
                )
            ) != len(
                control_genes
            ):

                raise RuntimeError(
                    "Duplicate gene inside "
                    "matched control set."
                )


            program_controls.append(
                control_genes
            )

            matching_set_rows.append({
                "disease":
                    disease,

                "program":
                    program,

                "control_id":
                    control_id,

                "n_genes":
                    len(
                        control_genes
                    ),

                "mean_matching_distance":
                    float(
                        np.mean(
                            selected_distances
                        )
                    ),

                "max_matching_distance":
                    float(
                        np.max(
                            selected_distances
                        )
                    ),
            })

        matched_controls[
            (
                disease,
                program,
            )
        ] = program_controls

## Write matched-control gene-set files

In [ ]:
matching_pairs_df = pd.DataFrame(
    matching_pair_rows
)

matching_sets_df = pd.DataFrame(
    matching_set_rows
)

matching_pairs_df.to_csv(
    OUT_DIR
    / "Fig6_Top100_matched_control_gene_pairs.csv",
    index=False,
)

matching_sets_df.to_csv(
    OUT_DIR
    / "Fig6_Top100_matched_control_QC.csv",
    index=False,
)

with open(
    OUT_DIR
    / "Fig6_Top100_matched_control_gene_sets.json",
    "w",
) as f:

    json.dump(
        {
            f"{disease}__{program}":
                matched_controls[
                    (
                        disease,
                        program,
                    )
                ]

            for disease in DISEASES
            for program in PROGRAM_ORDER
        },
        f,
        indent=2,
    )


print(
    "\nMatched controls constructed and saved."
)

## Helper: patient-balanced latent mean

In [ ]:
def patient_balanced_latent_mean(
    Z,
    mask,
):

    pts = np.unique(
        patient_ids[
            mask
        ]
    )


    patient_means = []


    for pt in pts:

        idx = np.where(
            mask
            &
            (
                patient_ids
                == pt
            )
        )[0]


        patient_means.append(
            np.asarray(
                Z[
                    idx
                ],
                dtype=np.float32,
            ).mean(
                axis=0
            )
        )

    return np.mean(
        np.stack(
            patient_means,
            axis=0,
        ),
        axis=0,
    )


## Helper: patient-balanced scalar mean

In [ ]:
def patient_balanced_scalar_mean(
    values,
    mask,
):

    pts = np.unique(
        patient_ids[
            mask
        ]
    )


    patient_means = []


    for pt in pts:

        idx = np.where(
            mask
            &
            (
                patient_ids
                == pt
            )
        )[0]


        patient_means.append(
            np.mean(
                values[
                    idx
                ]
            )
        )


    return float(
        np.mean(
            patient_means
        )
    )


## Helper: normalized axis

In [ ]:
# Reconstruct frozen disease-specific risk coordinate

def make_normalized_axis(
    Z,
    risk_mask,
    reference_mask,
):

    risk_centroid = (
        patient_balanced_latent_mean(
            Z,
            risk_mask,
        )
    )


    reference_centroid = (
        patient_balanced_latent_mean(
            Z,
            reference_mask,
        )
    )


    direction = (
        risk_centroid
        - reference_centroid
    )


    norm = np.linalg.norm(
        direction
    )


    if norm <= 1e-12:

        raise RuntimeError(
            "Zero-length clinical direction."
        )


    direction = (
        direction
        / norm
    )


    scores = (
        np.asarray(
            Z
        )
        @ direction
    )


    clinical_gap = (
        patient_balanced_scalar_mean(
            scores,
            risk_mask,
        )
        -
        patient_balanced_scalar_mean(
            scores,
            reference_mask,
        )
    )


    if clinical_gap <= 0:

        raise RuntimeError(
            "Non-positive clinical gap."
        )


    # Normalize so this component contributes on comparable clinical scale across diseases and programs
    return (
        direction
        / clinical_gap
    )


## Helper: primary risk weight

In [ ]:
def build_primary_risk_weight(
    Z,
    disease,
):

    masks = clinical_masks[
        disease
    ]


    if disease == "CAR-T_CRS":

        weight = (
            make_normalized_axis(
                Z,
                masks["risk"],
                masks[
                    "temporal_reference"
                ],
            )
        )


    elif disease == "COVID19":

        stage_weight = (
            make_normalized_axis(
                Z,
                masks["risk"],
                masks[
                    "stage_reference"
                ],
            )
        )


        temporal_weight = (
            make_normalized_axis(
                Z,
                masks["risk"],
                masks[
                    "temporal_reference"
                ],
            )
        )


        # Frozen equal-weight COVID primary coordinate
        weight = (
            0.5
            * stage_weight
            +
            0.5
            * temporal_weight
        )


    elif disease == "SLE":

        weight = (
            make_normalized_axis(
                Z,
                masks["risk"],
                masks[
                    "severity_reference"
                ],
            )
        )


    else:

        raise ValueError(
            disease
        )


    primary_scores = (
        np.asarray(
            Z
        )
        @ weight
    )


    final_gap = (
        patient_balanced_scalar_mean(
            primary_scores,
            masks["risk"],
        )
        -
        patient_balanced_scalar_mean(
            primary_scores,
            masks["reference"],
        )
    )


    if final_gap <= 0:

        raise RuntimeError(
            f"{disease}: invalid final "
            f"clinical gap."
        )


    return (
        weight.astype(
            np.float32
        ),
        float(
            final_gap
        ),
    )


## Helper: dense row extraction

In [ ]:

# Dense batch helper

def get_dense_rows(
    X,
    idx,
):

    result = X[
        idx
    ]


    if sparse.issparse(
        result
    ):

        result = (
            result.toarray()
        )


    return np.asarray(
        result,
        dtype=np.float32,
    )


## Define gene-set reversion evaluator

In [ ]:
def evaluate_gene_set(
    encoder,
    disease,
    gene_set,
    risk_idx,
    clean_projection,
    risk_weight,
    clinical_gap,
):

    gene_idx = np.array(
        [
            gene_to_idx[
                gene
            ]
            for gene
            in gene_set
        ],
        dtype=int,
    )


    reference_vector = (
        reference_expression[
            disease
        ]
    )


    patient_accumulator = {}


    for start in range(
        0,
        len(
            risk_idx
        ),
        ENCODE_BATCH_SIZE,
    ):

        stop = min(
            start
            + ENCODE_BATCH_SIZE,
            len(
                risk_idx
            ),
        )


        batch_idx = (
            risk_idx[
                start:
                stop
            ]
        )


        X_reverted = (
            get_dense_rows(
                adata.X,
                batch_idx,
            )
        )


        # Formal counterfactual:
        #
        # program genes -> patient-balanced clinical-reference expression
        #
        # Everything else remains observed
        X_reverted[
            :,
            gene_idx
        ] = (
            reference_vector[
                gene_idx
            ]
        )

        Z_reverted = (
            encoder.encode(
                X_reverted
            )
        )

        reverted_projection = (
            Z_reverted
            @ risk_weight
        )

        efficacy = (
            (
                clean_projection[
                    start:
                    stop
                ]
                -
                reverted_projection
            )
            / clinical_gap
        )

        batch_patients = (
            patient_ids[
                batch_idx
            ]
        )

        for pt in np.unique(
            batch_patients
        ):

            values = efficacy[
                batch_patients
                == pt
            ]

            if pt not in (
                patient_accumulator
            ):

                patient_accumulator[
                    pt
                ] = [
                    0.0,
                    0,
                ]

            patient_accumulator[
                pt
            ][0] += (
                float(
                    values.sum()
                )
            )

            patient_accumulator[
                pt
            ][1] += (
                len(
                    values
                )
            )

    return {
        pt:
            total
            / count

        for pt, (
            total,
            count,
        )
        in patient_accumulator.items()
    }

## Evaluate observed and matched program reversion

In [ ]:
result_rows = []

for seed in SEEDS:

    print("\n")
    print("=" * 80)
    print(
        f"SEED {seed}"
    )
    print("=" * 80)

    encoder = (
        load_frozen_encoder(
            seed
        )
    )

    latent_path = (
        LATENT_DIR
        / f"Fig6_Z_seed_{seed}.npy"
    )

    Z = np.load(
        latent_path,
        mmap_mode="r",
    )

    if Z.shape != (
        adata.n_obs,
        64,
    ):

        raise ValueError(
            f"Seed {seed}: unexpected "
            f"latent shape {Z.shape}"
        )

    for disease in DISEASES:

        print(
            f"\n--- {disease} ---"
        )

        masks = (
            clinical_masks[
                disease
            ]
        )

        risk_idx = np.where(
            masks[
                "risk"
            ]
        )[0]

        risk_weight, clinical_gap = (
            build_primary_risk_weight(
                Z,
                disease,
            )
        )

        # All biological/control gene-set counterfactuals are
        # compared against this same clean state

        Z_risk_clean = np.asarray(
            Z[
                risk_idx
            ],
            dtype=np.float32,
        )


        clean_projection = (
            Z_risk_clean
            @ risk_weight
        )


        print(
            f"Risk cells={len(risk_idx):,}, "
            f"patients="
            f"{len(np.unique(patient_ids[risk_idx]))}, "
            f"clinical gap="
            f"{clinical_gap:.6f}"
        )


        for program in (
            PROGRAM_ORDER
        ):

            observed_genes = (
                program_genes[
                    program
                ]
            )


            print(
                f"\n  {program}"
                f" | N={len(observed_genes)}"
            )

            # Biological atlas program
            observed_patient_values = (
                evaluate_gene_set(
                    encoder=
                        encoder,

                    disease=
                        disease,

                    gene_set=
                        observed_genes,

                    risk_idx=
                        risk_idx,

                    clean_projection=
                        clean_projection,

                    risk_weight=
                        risk_weight,

                    clinical_gap=
                        clinical_gap,
                )
            )

            for pt, efficacy in (
                observed_patient_values.items()
            ):

                result_rows.append({

                    "seed":
                        seed,

                    "disease":
                        disease,

                    "program":
                        program,

                    "set_type":
                        "atlas_program",

                    "control_id":
                        -1,

                    "patient_id":
                        pt,

                    "n_genes":
                        len(
                            observed_genes
                        ),

                    "normalized_efficacy":
                        efficacy,
                })

            # Matched controls
            controls = (
                matched_controls[
                    (
                        disease,
                        program,
                    )
                ]
            )

            for control_id, control_genes in enumerate(
                controls
            ):

                control_patient_values = (
                    evaluate_gene_set(
                        encoder=
                            encoder,

                        disease=
                            disease,

                        gene_set=
                            control_genes,

                        risk_idx=
                            risk_idx,

                        clean_projection=
                            clean_projection,

                        risk_weight=
                            risk_weight,

                        clinical_gap=
                            clinical_gap,
                    )
                )

                for pt, efficacy in (
                    control_patient_values.items()
                ):

                    result_rows.append({

                        "seed":
                            seed,

                        "disease":
                            disease,

                        "program":
                            program,

                        "set_type":
                            "matched_control",

                        "control_id":
                            control_id,

                        "patient_id":
                            pt,

                        "n_genes":
                            len(
                                control_genes
                            ),

                        "normalized_efficacy":
                            efficacy,
                    })

            print(
                f"    finished "
                f"{N_MATCHED_CONTROLS} "
                f"matched controls"
            )

    del Z
    del encoder

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


## Write reversion tables

In [ ]:
raw_df = pd.DataFrame(
    result_rows
)

raw_df.to_csv(
    OUT_DIR
    / "Fig6_Top100_program_reversion_RAW.csv",
    index=False,
)

# Seed consensus at patient level
# Five seeds are model-robustness replicates
# Median across seeds prevents them from becoming artificial outliers in the patient-level summary
patient_consensus = (
    raw_df
    .groupby(
        [
            "disease",
            "program",
            "set_type",
            "control_id",
            "patient_id",
        ],
        as_index=False,
    )
    .agg(

        normalized_efficacy=(
            "normalized_efficacy",
            "median",
        ),

        seed_mean=(
            "normalized_efficacy",
            "mean",
        ),

        seed_sd=(
            "normalized_efficacy",
            "std",
        ),

        n_seeds=(
            "seed",
            "nunique",
        ),

        n_genes=(
            "n_genes",
            "first",
        ),
    )
)


patient_consensus.to_csv(
    OUT_DIR
    / "Fig6_Top100_program_reversion_patient_consensus.csv",
    index=False,
)
# Equal-patient summary for every gene set
set_level = (
    patient_consensus
    .groupby(
        [
            "disease",
            "program",
            "set_type",
            "control_id",
        ],
        as_index=False,
    )
    .agg(

        mean_efficacy=(
            "normalized_efficacy",
            "mean",
        ),

        median_efficacy=(
            "normalized_efficacy",
            "median",
        ),

        n_patients=(
            "patient_id",
            "nunique",
        ),

        n_genes=(
            "n_genes",
            "first",
        ),
    )
)

set_level.to_csv(
    OUT_DIR
    / "Fig6_Top100_program_reversion_set_level.csv",
    index=False,
)


## Summarize matched-control effects

In [ ]:
summary_rows = []

for disease in DISEASES:

    for program in PROGRAM_ORDER:

        observed_row = (
            set_level.loc[
                (
                    set_level[
                        "disease"
                    ]
                    == disease
                )
                &
                (
                    set_level[
                        "program"
                    ]
                    == program
                )
                &
                (
                    set_level[
                        "set_type"
                    ]
                    == "atlas_program"
                )
            ]
            .iloc[0]
        )


        controls = (
            set_level.loc[
                (
                    set_level[
                        "disease"
                    ]
                    == disease
                )
                &
                (
                    set_level[
                        "program"
                    ]
                    == program
                )
                &
                (
                    set_level[
                        "set_type"
                    ]
                    == "matched_control"
                ),
                "mean_efficacy",
            ]
            .to_numpy()
        )


        observed = float(
            observed_row[
                "mean_efficacy"
            ]
        )


        control_mean = float(
            np.mean(
                controls
            )
        )


        control_median = float(
            np.median(
                controls
            )
        )


        control_sd = float(
            np.std(
                controls,
                ddof=1,
            )
        )


        control_q025 = float(
            np.quantile(
                controls,
                0.025,
            )
        )


        control_q975 = float(
            np.quantile(
                controls,
                0.975,
            )
        )


        delta = (
            observed
            - control_median
        )

        empirical_p_greater = (
            1
            +
            np.sum(
                controls
                >= observed
            )
        ) / (
            N_MATCHED_CONTROLS
            + 1
        )

        # Also store two-sided extremeness for audit
        center = (
            control_median
        )


        observed_distance = abs(
            observed
            - center
        )


        control_distances = np.abs(
            controls
            - center
        )

        empirical_p_two_sided = (
            1
            +
            np.sum(
                control_distances
                >= observed_distance
            )
        ) / (
            N_MATCHED_CONTROLS
            + 1
        )

        percentile = float(
            100
            * np.mean(
                controls
                < observed
            )
        )

        z_score = (
            (
                observed
                - control_mean
            )
            / control_sd

            if control_sd > 0
            else np.nan
        )

        summary_rows.append({

            "disease":
                disease,

            "program":
                program,

            "n_genes":
                int(
                    observed_row[
                        "n_genes"
                    ]
                ),

            "n_patients":
                int(
                    observed_row[
                        "n_patients"
                    ]
                ),

            "observed_mean_efficacy":
                observed,

            "matched_control_mean":
                control_mean,

            "matched_control_median":
                control_median,

            "matched_control_sd":
                control_sd,

            "matched_control_2.5pct":
                control_q025,

            "matched_control_97.5pct":
                control_q975,

            "delta_vs_control_median":
                delta,

            "z_vs_matched_controls":
                z_score,

            "control_percentile":
                percentile,

            "empirical_p_greater":
                empirical_p_greater,

            "empirical_p_two_sided":
                empirical_p_two_sided,
        })


summary = pd.DataFrame(
    summary_rows
)

## Adjust reversion matrices

In [ ]:
def bh_adjust(
    pvalues,
):

    p = np.asarray(
        pvalues,
        dtype=float,
    )


    n = len(
        p
    )


    order = np.argsort(
        p
    )


    p_sorted = (
        p[
            order
        ]
    )


    q_sorted = (
        p_sorted
        * n
        / (
            np.arange(
                1,
                n + 1,
            )
        )
    )


    q_sorted = (
        np.minimum.accumulate(
            q_sorted[::-1]
        )[::-1]
    )


    q_sorted = np.minimum(
        q_sorted,
        1.0,
    )


    q = np.empty_like(
        q_sorted
    )


    q[
        order
    ] = q_sorted


    return q


summary[
    "empirical_q_greater"
] = bh_adjust(
    summary[
        "empirical_p_greater"
    ].to_numpy()
)


summary.to_csv(
    OUT_DIR
    / "Fig6_Top100_program_reversion_MATCHED_summary.csv",
    index=False,
)

# Print primary results
print("\n")
print("=" * 110)
print(
    "TOP100-AVAILABLE PROGRAM REVERSION — MATCHED CONTROL RESULTS"
)
print("=" * 110)


display_columns = [

    "disease",
    "program",
    "n_genes",
    "n_patients",

    "observed_mean_efficacy",

    "matched_control_median",

    "delta_vs_control_median",

    "control_percentile",

    "empirical_p_greater",

    "empirical_q_greater",
]


print(
    summary[
        display_columns
    ]
    .to_string(
        index=False
    )
)
# Raw and control-adjusted matrices
raw_matrix = (
    summary
    .pivot(
        index="program",
        columns="disease",
        values=
            "observed_mean_efficacy",
    )
    .reindex(
        index=
            PROGRAM_ORDER,

        columns=
            DISEASES,
    )
)

adjusted_matrix = (
    summary
    .pivot(
        index="program",
        columns="disease",
        values=
            "delta_vs_control_median",
    )
    .reindex(
        index=
            PROGRAM_ORDER,

        columns=
            DISEASES,
    )
)

raw_matrix.to_csv(
    OUT_DIR
    / "Fig6_Top100_program_reversion_RAW_matrix.csv"
)

adjusted_matrix.to_csv(
    OUT_DIR
    / "Fig6_Top100_program_reversion_CONTROL_ADJUSTED_matrix.csv"
)

print(
    "\nRaw mean efficacy:"
)

print(
    raw_matrix
)

## Prepare bubble-plot data

In [ ]:
row_labels = [

    "Inflammatory CD14\n(IL1B)",

    "Emergency CD14\n(S100A8)",

    "CD16 sensing\n(LST1)",
]

column_labels = [

    "CAR-T CRS",

    "COVID-19",

    "SLE",
]

plot_df = (
    summary
    .set_index(
        [
            "program",
            "disease",
        ]
    )
)

max_abs = max(
    float(
        np.max(
            np.abs(
                summary[
                    "delta_vs_control_median"
                ]
            )
        )
    ),
    1e-6,
)


## Draw control-adjusted bubble plot

In [ ]:
fig, ax = plt.subplots(
    figsize=(
        7.2,
        5.5,
    )
)

for row_i, program in enumerate(
    PROGRAM_ORDER
):

    for col_i, disease in enumerate(
        DISEASES
    ):

        row = plot_df.loc[
            (
                program,
                disease,
            )
        ]


        effect = float(
            row[
                "delta_vs_control_median"
            ]
        )


        q = float(
            row[
                "empirical_q_greater"
            ]
        )


        size = (
            250
            +
            1800
            * abs(
                effect
            )
            / max_abs
        )


        ax.scatter(

            col_i,
            row_i,

            s=size,

            c=[
                effect
            ],

            cmap="RdBu_r",

            vmin=
                -max_abs,

            vmax=
                max_abs,

            edgecolors=
                "black",

            linewidths=
                1.1,
        )


        if q < 0.001:

            stars = "***"

        elif q < 0.01:

            stars = "**"

        elif q < 0.05:

            stars = "*"

        else:

            stars = ""


        ax.text(

            col_i,
            row_i,

            stars,

            ha="center",
            va="center",

            fontsize=13,
            fontweight="bold",
        )


In [ ]:
ax.set_xticks(
    range(
        len(
            DISEASES
        )
    )
)

ax.set_xticklabels(
    column_labels
)


ax.set_yticks(
    range(
        len(
            PROGRAM_ORDER
        )
    )
)


ax.set_yticklabels(
    row_labels
)


ax.set_xlim(
    -0.5,
    len(
        DISEASES
    )
    - 0.5,
)

ax.set_ylim(
    len(
        PROGRAM_ORDER
    )
    - 0.5,
    -0.5,
)

ax.set_xlabel(
    "Learned disease-specific risk representation"
)

ax.set_ylabel(
    "Atlas-defined monocyte program reverted"
)

ax.set_title(
    "Top-100 atlas-program reference-state reversion"
)

sm = plt.cm.ScalarMappable(

    cmap="RdBu_r",

    norm=plt.Normalize(
        vmin=
            -max_abs,

        vmax=
            max_abs,
    ),
)

sm.set_array([])

cbar = plt.colorbar(

    sm,

    ax=ax,

    fraction=0.046,

    pad=0.04,
)

cbar.set_label(
    "Program efficacy − matched-control efficacy"
)

plt.tight_layout()

plt.savefig(
    OUT_DIR
    / "Fig6_Top100_program_reversion_CONTROL_ADJUSTED_bubble.pdf",
    bbox_inches="tight",
)

plt.savefig(
    OUT_DIR
    / "Fig6_Top100_program_reversion_CONTROL_ADJUSTED_bubble.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(
    f"\nDone.\nOutputs: {OUT_DIR}"
)


## Fig. 6F lollipop plot

In [ ]:
# ============================================================
# Fig. 6F:
# disease-wise lollipop panels for Top100 matched-control
# reversion results
#
# Input:
#   Fig6_Top100_program_reversion_MATCHED_summary.csv
#
# Plot:
#   3 horizontal panels (CAR-T CRS / COVID-19 / SLE)
#   x = control-adjusted efficacy (% clinical gap)
#   y = atlas-defined monocyte programs
# ============================================================

from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path


# Paths
ROOT = resolve_analysis_path(os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai")))

IN_FILE = (
    ROOT
    / "Fig6_atlas_program_reversion_Top100_matched"
    / "Fig6_Top100_program_reversion_MATCHED_summary.csv"
)

OUT_DIR = (
    ROOT
    / "Fig6_atlas_program_reversion_Top100_matched"
    / "clean_lollipop_plot"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load summary
df = pd.read_csv(IN_FILE)

required_cols = [
    "disease",
    "program",
    "delta_vs_control_median",
]

missing = [
    c for c in required_cols
    if c not in df.columns
]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

## Prepare lollipop-plot table

In [ ]:
# Frozen display order and labels

disease_order = [
    "CAR-T_CRS",
    "COVID19",
    "SLE",
]

disease_labels = {
    "CAR-T_CRS": "CAR-T CRS",
    "COVID19": "COVID-19",
    "SLE": "SLE",
}

program_order = [
    "Inflammatory_CD14_IL1B",
    "Emergency_CD14_S100A8",
    "CD16_sensing_LST1",
]

program_labels = {
    "Inflammatory_CD14_IL1B": "Inflammatory CD14\n(IL1B)",
    "Emergency_CD14_S100A8": "Emergency CD14\n(S100A8)",
    "CD16_sensing_LST1": "CD16 sensing\n(LST1)",
}

program_colors = {
    "Inflammatory_CD14_IL1B": "#447296",  
    "Emergency_CD14_S100A8": "#DB9B6D",   
    "CD16_sensing_LST1": "#4F3D80",       
}

# Convert to plotting table
plot_df = (
    df.loc[
        df["disease"].isin(disease_order)
        & df["program"].isin(program_order)
    ].copy()
)

# Express as percentage of disease-specific clinical gap
plot_df["effect_pct"] = (
    100.0
    * plot_df["delta_vs_control_median"]
)

# Keep plotting order frozen
plot_df["disease"] = pd.Categorical(
    plot_df["disease"],
    categories=disease_order,
    ordered=True,
)

plot_df["program"] = pd.Categorical(
    plot_df["program"],
    categories=program_order,
    ordered=True,
)

plot_df = plot_df.sort_values(
    ["disease", "program"]
)


In [ ]:
# Determine shared x-axis range
max_abs = np.max(
    np.abs(plot_df["effect_pct"])
)

if max_abs == 0:
    max_abs = 1.0

x_limit = 1.15 * max_abs

raw_lim = x_limit

if raw_lim <= 2:
    step = 0.5
elif raw_lim <= 5:
    step = 1.0
elif raw_lim <= 10:
    step = 2.0
else:
    step = 5.0

x_limit = np.ceil(raw_lim / step) * step


In [ ]:
# Create plot
fig, axes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(9.2, 3.8),
    sharey=True,
)

y_positions = np.array([2, 1, 0])

for ax, disease in zip(axes, disease_order):

    sub = (
        plot_df.loc[
            plot_df["disease"] == disease
        ]
        .set_index("program")
        .loc[program_order]
        .reset_index()
    )

    ax.axvline(
        0,
        color="0.75",
        linewidth=1.0,
        zorder=0,
    )

    for y, (_, row) in zip(y_positions, sub.iterrows()):

        program = row["program"]
        effect = row["effect_pct"]

        ax.hlines(
            y=y,
            xmin=0,
            xmax=effect,
            color=program_colors[program],
            linewidth=2.2,
            zorder=2,
        )

        ax.scatter(
            effect,
            y,
            s=100,
            color=program_colors[program],

            linewidth=0.9,
            zorder=3,
        )

    ax.set_title(
        disease_labels[disease],
        fontsize=12,
        pad=10,
    )

    ax.set_xlim(-x_limit, x_limit)
    ax.set_ylim(-0.6, 2.6)

    ax.set_xticks(
        [-x_limit, 0, x_limit]
    )

    ax.set_xlabel(
        "% clinical gap",
        fontsize=10,
    )

    ax.grid(
        axis="x",
        color="0.92",
        linewidth=0.8,
    )

    # Make panel visually clean.
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0].set_yticks(y_positions)
axes[0].set_yticklabels(
    [program_labels[p] for p in program_order],
    fontsize=10,
)

axes[0].set_ylabel(
    "Atlas-defined monocyte program reverted",
    fontsize=11,
)

for ax in axes[1:]:
    ax.set_yticks(y_positions)
    ax.tick_params(axis="y", left=False, labelleft=False)


fig.suptitle(
    "Top-100 atlas-program reference-state reversion",
    fontsize=14,
    y=1.03,
)

fig.text(
    0.5,
    -0.02,
    "Control-adjusted risk-state reduction beyond matched controls",
    ha="center",
    fontsize=11,
)

plt.tight_layout()

In [ ]:
png_file = OUT_DIR / "Fig6_Top100_lollipop_clean.png"
pdf_file = OUT_DIR / "Fig6_Top100_lollipop_clean.pdf"

plt.savefig(
    png_file,
    dpi=300,
    bbox_inches="tight",
)

plt.savefig(
    pdf_file,
    bbox_inches="tight",
)

plt.show()

print("Saved:")
print(png_file)
print(pdf_file)

# 14. Fig. 6I predicted regulators

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ChEA3 top-10 TF enrichment results
df = pd.DataFrame({
    "TF": [
        "BATF2",
        "IRF7",
        "IRF9",
        "SP100",
        "BATF3",
        "IRF1",
        "SP140",
        "IRF8",
        "NFE4",
        "BATF"
    ],
    "mean_rank": [
        2.0,
        6.333,
        8.25,
        9.0,
        10.67,
        10.67,
        14.67,
        17.5,
        21.0,
        23.8
    ],
    "overlap_n": [
        23,
        21,
        21,
        20,
        18,
        23,
        20,
        19,
        5,
        20
    ]
})

# Total number of conserved genes
N_CONSERVED = 34

# Convert overlap count to fraction
df["overlap_frac"] = df["overlap_n"] / N_CONSERVED

df = df.sort_values(
    by=["overlap_n", "mean_rank"],
    ascending=[False, True]
).reset_index(drop=True)


# Square panel for compact multi-panel figure assembly
fig, ax = plt.subplots(
    figsize=(4.2, 4.2)
)

bars = ax.barh(
    y=df["TF"],
    width=df["overlap_frac"],

    height=0.68,
    linewidth=0,

    color="#4682B4"
)

# Highest-overlap TF appears at the top
ax.invert_yaxis()

for bar, n in zip(
    bars,
    df["overlap_n"]
):

    x = bar.get_width()
    y = (
        bar.get_y()
        + bar.get_height() / 2
    )

    ax.text(
        x + 0.015,
        y,

        f"{n}/{N_CONSERVED}",

        va="center",
        ha="left",
        fontsize=9
    )


# Axes and labels

ax.set_xlim(
    0,
    0.90
)

ax.set_xticks([
    0,
    0.2,
    0.4,
    0.6,
    0.8
])

ax.set_xlabel(
    "Overlap with conserved risk genes",
    fontsize=10
)

ax.set_ylabel("")

ax.set_title(
    "TF enrichment of conserved risk genes",
    fontsize=12,
    pad=10
)

# Style
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_linewidth(1.2)

ax.tick_params(
    axis="both",
    labelsize=9,
    width=1.0,
    length=4
)

plt.tight_layout()

# Export

fig.savefig(
    "TF_enrichment_conserved_risk_genes.pdf",

    # Crop excess whitespace around the figure
    bbox_inches="tight"
)

plt.show()